In [ ]:
# ============================================================
# QC + PAIRWISE COMPARISON PLOTS
# old vs soft, feng vs old, feng vs soft
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OLD_DIR = Path(r"C:\Users\_s2218026\Documents\lab\mcs_output")
NEW_DIR = Path(r"C:\Users\_s2218026\Documents\lab\mcs_output_softtest")

# if you already exported these from robustness:
FENG_STD_FILE = Path(r"C:\Users\_s2218026\Documents\lab\robustness_exact_zhang_case\standardized_feng_overlap.csv")
OURS_STD_FILE = Path(r"C:\Users\_s2218026\Documents\lab\robustness_exact_zhang_case\standardized_ours_overlap.csv")


# ------------------------------------------------------------
# BASIC SUMMARY
# ------------------------------------------------------------
def summarize_catalog(obj_df, trk_df, name="dataset"):
    return pd.DataFrame([{
        "dataset": name,
        "n_objects": len(obj_df),
        "n_track_rows": len(trk_df),
        "n_tracks": trk_df["track_id"].nunique() if "track_id" in trk_df.columns else np.nan,
        "mean_area_km2": obj_df["area_km2"].mean() if "area_km2" in obj_df.columns else np.nan,
        "median_area_km2": obj_df["area_km2"].median() if "area_km2" in obj_df.columns else np.nan,
        "mean_tb_mean": obj_df["tb_mean"].mean() if "tb_mean" in obj_df.columns else np.nan,
        "mean_tb_min": obj_df["tb_min"].mean() if "tb_min" in obj_df.columns else np.nan,
        "mean_speed_kmh": trk_df["speed_kmh"].dropna().mean() if "speed_kmh" in trk_df.columns else np.nan,
        "median_speed_kmh": trk_df["speed_kmh"].dropna().median() if "speed_kmh" in trk_df.columns else np.nan,
    }])


# ------------------------------------------------------------
# LOAD DAY / MONTH OUTPUTS
# ------------------------------------------------------------
def load_day_outputs(date_str, old_dir=OLD_DIR, new_dir=NEW_DIR):
    ymd = pd.Timestamp(date_str).strftime("%Y%m%d")

    old_obj = pd.read_csv(old_dir / f"mcs_objects_3h_{ymd}.csv")
    old_trk = pd.read_csv(old_dir / f"mcs_tracks_3h_{ymd}.csv")
    new_obj = pd.read_csv(new_dir / f"mcs_objects_soft_3h_{ymd}.csv")
    new_trk = pd.read_csv(new_dir / f"mcs_tracks_soft_3h_{ymd}.csv")

    for df in [old_obj, old_trk, new_obj, new_trk]:
        if "time" in df.columns:
            df["time"] = pd.to_datetime(df["time"], errors="coerce")

    return old_obj, old_trk, new_obj, new_trk


def load_month_outputs(year, month, old_dir=OLD_DIR, new_dir=NEW_DIR):
    ym = f"{year}{month:02d}"

    old_obj = pd.read_csv(old_dir / f"mcs_objects_3h_{ym}.csv")
    old_trk = pd.read_csv(old_dir / f"mcs_tracks_3h_{ym}.csv")
    new_obj = pd.read_csv(new_dir / f"mcs_objects_soft_3h_{ym}.csv")
    new_trk = pd.read_csv(new_dir / f"mcs_tracks_soft_3h_{ym}.csv")

    for df in [old_obj, old_trk, new_obj, new_trk]:
        if "time" in df.columns:
            df["time"] = pd.to_datetime(df["time"], errors="coerce")

    return old_obj, old_trk, new_obj, new_trk


# ------------------------------------------------------------
# STANDARDIZE old/new object+track outputs to same columns as Feng
# ------------------------------------------------------------
def standardize_old_new(obj_df, trk_df, dataset_name):
    obj = obj_df.copy()
    trk = trk_df.copy()

    if "time" in obj.columns:
        obj["time"] = pd.to_datetime(obj["time"], errors="coerce")
    if "time" in trk.columns:
        trk["time"] = pd.to_datetime(trk["time"], errors="coerce")

    keep_obj = pd.DataFrame({
        "time": obj["time"],
        "lat": pd.to_numeric(obj["centroid_lat"], errors="coerce"),
        "lon": pd.to_numeric(obj["centroid_lon"], errors="coerce"),
        "area_km2": pd.to_numeric(obj["area_km2"], errors="coerce"),
        "tb_mean": pd.to_numeric(obj["tb_mean"], errors="coerce"),
        "tb_min": pd.to_numeric(obj["tb_min"], errors="coerce"),
    })

    # bring speed/direction from tracks by time + centroid match if possible
    trk_keep = trk[["time", "centroid_lat", "centroid_lon", "track_id", "speed_kmh", "direction_deg"]].copy()
    trk_keep["centroid_lat"] = pd.to_numeric(trk_keep["centroid_lat"], errors="coerce")
    trk_keep["centroid_lon"] = pd.to_numeric(trk_keep["centroid_lon"], errors="coerce")

    keep_obj = keep_obj.merge(
        trk_keep,
        left_on=["time", "lat", "lon"],
        right_on=["time", "centroid_lat", "centroid_lon"],
        how="left"
    ).drop(columns=["centroid_lat", "centroid_lon"])

    keep_obj["dataset"] = dataset_name
    return keep_obj


def load_feng_standardized(feng_std_file=FENG_STD_FILE):
    feng = pd.read_csv(feng_std_file)
    if "time" in feng.columns:
        feng["time"] = pd.to_datetime(feng["time"], errors="coerce")
    return feng


# ------------------------------------------------------------
# YOUR MISSING QC HISTOGRAM FUNCTION
# ------------------------------------------------------------
def plot_qc_histograms(old_obj, old_trk, new_obj, new_trk, label):
    plt.figure(figsize=(7,4))
    plt.hist(old_obj["area_km2"].dropna(), bins=50, alpha=0.5, label="old")
    plt.hist(new_obj["area_km2"].dropna(), bins=50, alpha=0.5, label="soft")
    plt.xlabel("Area (km²)")
    plt.ylabel("Count")
    plt.title(f"Object area comparison: {label}")
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(7,4))
    plt.hist(old_obj["tb_mean"].dropna(), bins=40, alpha=0.5, label="old")
    plt.hist(new_obj["tb_mean"].dropna(), bins=40, alpha=0.5, label="soft")
    plt.xlabel("tb_mean (K)")
    plt.ylabel("Count")
    plt.title(f"tb_mean comparison: {label}")
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(7,4))
    plt.hist(old_trk["speed_kmh"].dropna(), bins=40, alpha=0.5, label="old")
    plt.hist(new_trk["speed_kmh"].dropna(), bins=40, alpha=0.5, label="soft")
    plt.xlabel("Speed (km/h)")
    plt.ylabel("Count")
    plt.title(f"Track speed comparison: {label}")
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_track_length_comparison(old_trk, new_trk, label):
    old_len = old_trk.groupby("track_id").size()
    new_len = new_trk.groupby("track_id").size()

    bins = np.arange(1, max(old_len.max(), new_len.max()) + 2) - 0.5
    plt.figure(figsize=(7,4))
    plt.hist(old_len, bins=bins, alpha=0.5, label="old")
    plt.hist(new_len, bins=bins, alpha=0.5, label="soft")
    plt.xlabel("Track length (number of 3-hour snapshots)")
    plt.ylabel("Count")
    plt.title(f"Track length comparison: {label}")
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_centroid_map(df_a, df_b, label_a, label_b, title,
                      lon_col_a="centroid_lon", lat_col_a="centroid_lat",
                      lon_col_b="centroid_lon", lat_col_b="centroid_lat",
                      lon_range=(100,180), lat_range=(0,30)):
    plt.figure(figsize=(8,5))
    plt.scatter(df_a[lon_col_a], df_a[lat_col_a], s=8, alpha=0.5, label=label_a)
    plt.scatter(df_b[lon_col_b], df_b[lat_col_b], s=8, alpha=0.5, label=label_b)
    plt.xlim(lon_range)
    plt.ylim(lat_range)
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()


# ------------------------------------------------------------
# GENERIC PAIRWISE HIST FUNCTION
# works for Feng vs old/new or old vs new
# ------------------------------------------------------------
def plot_pair_hist(df1, df2, col, label1, label2, bins=40, title=None):
    plt.figure(figsize=(7,4))
    plt.hist(pd.to_numeric(df1[col], errors="coerce").dropna(), bins=bins, alpha=0.5, label=label1)
    plt.hist(pd.to_numeric(df2[col], errors="coerce").dropna(), bins=bins, alpha=0.5, label=label2)
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.title(title if title else f"{col}: {label1} vs {label2}")
    plt.legend()
    plt.tight_layout()
    plt.show()


# ------------------------------------------------------------
# ONE-CLICK PAIRWISE COMPARISON
# standardized dataframes only
# ------------------------------------------------------------
def plot_pairwise_dataset_comparison(df1, df2, label1, label2, lon_range=(100,180), lat_range=(0,30)):
    for col, bins in [("area_km2", 50), ("tb_mean", 40), ("tb_min", 40)]:
        plot_pair_hist(df1, df2, col, label1, label2, bins=bins)

    # only if both have usable speed
    if ("speed_kmh" in df1.columns) and ("speed_kmh" in df2.columns):
        if df1["speed_kmh"].notna().sum() > 0 and df2["speed_kmh"].notna().sum() > 0:
            plot_pair_hist(df1, df2, "speed_kmh", label1, label2, bins=40)

    plt.figure(figsize=(8,5))
    plt.scatter(df1["lon"], df1["lat"], s=8, alpha=0.4, label=label1)
    plt.scatter(df2["lon"], df2["lat"], s=8, alpha=0.4, label=label2)
    plt.xlim(lon_range)
    plt.ylim(lat_range)
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.title(f"Centroid map: {label1} vs {label2}")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
old_obj_m, old_trk_m, new_obj_m, new_trk_m = load_month_outputs(2015, 8)

summary_month = pd.concat([
    summarize_catalog(old_obj_m, old_trk_m, "old_month"),
    summarize_catalog(new_obj_m, new_trk_m, "soft_month"),
], ignore_index=True)

print(summary_month.to_string(index=False))

plot_qc_histograms(old_obj_m, old_trk_m, new_obj_m, new_trk_m, "2015-08")
plot_track_length_comparison(old_trk_m, new_trk_m, "2015-08")
plot_centroid_map(old_obj_m, new_obj_m, "old", "soft", "Centroid map: old vs soft")

In [ ]:
old_std = standardize_old_new(old_obj_m, old_trk_m, "old")
new_std = standardize_old_new(new_obj_m, new_trk_m, "soft")
feng_std = load_feng_standardized()

In [ ]:
feng_aug2015 = feng_std[
    (feng_std["time"].dt.year == 2015) &
    (feng_std["time"].dt.month == 8)
].copy()

In [ ]:
plot_pairwise_dataset_comparison(feng_aug2015, old_std, "feng", "old")
plot_pairwise_dataset_comparison(feng_aug2015, new_std, "feng", "soft")
plot_pairwise_dataset_comparison(new_std, old_std, "soft", "old")

In [ ]:
import pandas as pd
from pathlib import Path

old_dir = Path(r"C:\Users\_s2218026\Documents\lab\mcs_output\zhang_case_tables_2009_2017")
soft_dir = Path(r"C:\Users\_s2218026\Documents\lab\mcs_output_softtest\zhang_case_tables_soft_2009_2017")

rows = []
for L in [6, 12, 24, 48]:
    old_fp = old_dir / f"zhang_case_table_{L}h.csv"
    soft_fp = soft_dir / f"zhang_case_table_{L}h.csv"   # adjust if your filename has "soft" in it

    old = pd.read_csv(old_fp)
    soft = pd.read_csv(soft_fp)

    rows.append({
        "lead_h": L,
        "old_total": len(old),
        "old_pos": int(old["y"].sum()),
        "old_neg": int((old["y"] == 0).sum()),
        "soft_total": len(soft),
        "soft_pos": int(soft["y"].sum()),
        "soft_neg": int((soft["y"] == 0).sum()),
    })

cmp = pd.DataFrame(rows)
print(cmp)

In [ ]:
from pathlib import Path
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================================================
# CONFIG
# =========================================================
LAB = Path(r"C:\Users\_s2218026\Documents\lab")

# main / old master table
OLD_MASTER_CSV = LAB / "mcs_output" / "mcs_master_zhang_ready_2009_2017_JJASO.csv"

# old lead-specific case tables
OLD_CASE_DIR = LAB / "mcs_output" / "zhang_case_tables_2009_2017"

# genesis table
GEN_CSV = LAB / "ibtracs_genesis_sorted.csv"

# optional: if you want comparison later
SOFT_CASE_DIR = LAB / "mcs_output_softtest" / "zhang_case_tables_soft_2009_2017"
FENG_STD_CSV = LAB / "robustness_exact_zhang_case" / "standardized_feng_overlap.csv"

LEADS = [6, 12, 24, 48]

# map extent for WNP
LON_MIN, LON_MAX = 100, 180
LAT_MIN, LAT_MAX = 0, 30

In [ ]:
# =========================================================
# LOAD DATA
# =========================================================
master = pd.read_csv(OLD_MASTER_CSV)
master["time"] = pd.to_datetime(master["time"], errors="coerce")

gen = pd.read_csv(GEN_CSV)
gen["genesis_time"] = pd.to_datetime(gen["genesis_time"], errors="coerce", utc=True).dt.tz_convert(None)

# keep thesis period only
master = master[
    master["time"].dt.year.between(2009, 2017) &
    master["time"].dt.month.isin([6, 7, 8, 9, 10])
].copy()

gen = gen[
    gen["genesis_time"].dt.year.between(2009, 2017) &
    gen["genesis_time"].dt.month.isin([6, 7, 8, 9, 10])
].copy()

master = master.dropna(subset=["time", "centroid_lat", "centroid_lon"]).reset_index(drop=True)

print("Master rows:", len(master))
print("Unique tracks:", master["track_id"].nunique())
print("Genesis rows:", len(gen))
print(master.columns.tolist())

In [ ]:
# =========================================================
# HELPER: LOAD OLD CASE TABLE
# =========================================================
def load_case_table(case_dir, lead):
    candidates = [
        case_dir / f"zhang_case_table_{lead}h.csv",
        case_dir / f"zhang_case_table_old_{lead}h.csv",
        case_dir / f"case_table_{lead}h.csv",
    ]
    for fp in candidates:
        if fp.exists():
            df = pd.read_csv(fp)
            if "time" in df.columns:
                df["time"] = pd.to_datetime(df["time"], errors="coerce")
            return df, fp
    raise FileNotFoundError(f"No case table found for lead {lead}h in {case_dir}")

In [ ]:
# =========================================================
# QC PLOT A: SUMMARY TABLE
# =========================================================
summary = pd.DataFrame([{
    "n_rows": len(master),
    "n_tracks": master["track_id"].nunique(),
    "mean_area_km2": master["area_km2"].mean(),
    "median_area_km2": master["area_km2"].median(),
    "mean_tb_mean": master["tb_mean"].mean(),
    "mean_tb_min": master["tb_min"].mean(),
    "mean_speed_kmh": master["speed_kmh"].mean(),
    "median_speed_kmh": master["speed_kmh"].median(),
}])

summary

In [ ]:
# =========================================================
# QC PLOT G: POSITIVE MAPS BY LEAD
# =========================================================
def plot_positive_map_for_lead(master, case_dir, gen, lead):
    df, fp = load_case_table(case_dir, lead)
    pos = df[df["y"] == 1].copy()

    plt.figure(figsize=(8,5))

    # background = all MCS density
    plt.hexbin(
        master["centroid_lon"], master["centroid_lat"],
        gridsize=55,
        extent=(LON_MIN, LON_MAX, LAT_MIN, LAT_MAX),
        mincnt=1
    )

    # positive MCS
    plt.scatter(
        pos["centroid_lon"], pos["centroid_lat"],
        s=25, alpha=0.8, label=f"positive MCS ({lead}h)"
    )

    # genesis points
    plt.scatter(
        gen["genesis_lon"], gen["genesis_lat"],
        s=35, facecolors="none", edgecolors="black", label="IBTrACS genesis"
    )

    plt.xlim(LON_MIN, LON_MAX)
    plt.ylim(LAT_MIN, LAT_MAX)
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.title(f"Main dataset: positive cases at {lead}h lead")
    plt.legend()
    plt.tight_layout()
    plt.show()

for L in LEADS:
    plot_positive_map_for_lead(master, OLD_CASE_DIR, gen, L)

In [ ]:
# =========================================================
# QC PLOT H: POSITIVE-ONLY MAPS
# =========================================================
def plot_positive_only_map(case_dir, gen, lead):
    df, fp = load_case_table(case_dir, lead)
    pos = df[df["y"] == 1].copy()

    plt.figure(figsize=(8,5))
    plt.scatter(pos["centroid_lon"], pos["centroid_lat"], s=30, alpha=0.8, label=f"positive MCS ({lead}h)")
    plt.scatter(gen["genesis_lon"], gen["genesis_lat"], s=35, facecolors="none", edgecolors="black", label="IBTrACS genesis")

    plt.xlim(LON_MIN, LON_MAX)
    plt.ylim(LAT_MIN, LAT_MAX)
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.title(f"Main dataset: positive MCS vs genesis ({lead}h)")
    plt.legend()
    plt.tight_layout()
    plt.show()

for L in LEADS:
    plot_positive_only_map(OLD_CASE_DIR, gen, L)

In [ ]:
# ============================================================
# PAIRWISE ROBUSTNESS PLOTS
# old vs feng, soft vs feng, old vs soft
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# 1) CONFIG
# -----------------------------
LAB = Path(r"C:\Users\_s2218026\Documents\lab")

# --- standardized positive-snapshot tables ---
# IMPORTANT:
# "old" = your original/main dataset
# if your previous file was "standardized_ours_overlap.csv",
# then use that as OLD.
POS_FILES = {
    "old":  LAB / "robustness_exact_zhang_case" / "standardized_ours_overlap.csv",
    "feng": LAB / "robustness_exact_zhang_case" / "standardized_feng_overlap.csv",
    "soft": LAB / "robustness_exact_zhang_case" / "standardized_soft_overlap.csv",  # <-- change if needed
}

# --- total number of rows in each FULL master dataset ---
# used for positive fraction plot: fraction = positive_snapshots / total_master_rows
# fill these with your real totals
TOTAL_ROWS = {
    "old":  None,   # e.g. len(old master)
    "feng": None,   # e.g. len(feng master)
    "soft": None,   # e.g. len(soft master)
}

# OPTIONAL: if you want, you can compute totals automatically from full master files
MASTER_FILES = {
    "old":  None,   # e.g. LAB / "mcs_output" / "mcs_master_zhang_ready_2009_2017_JJASO.csv"
    "feng": None,   # e.g. LAB / "..." / "mcs_master_feng_2009_2017_JJASO.csv"
    "soft": None,   # e.g. LAB / "mcs_output_softtest" / "mcs_master_soft_2009_2017_JJASO.csv"
}

OUTDIR = LAB / "robustness_exact_zhang_case" / "pairwise_plots"
OUTDIR.mkdir(parents=True, exist_ok=True)

LEADS = [6, 12, 24, 48]


# -----------------------------
# 2) HELPERS
# -----------------------------
def standardize_positive_df(df, dataset_name="dataset"):
    """
    Standardize columns so plotting works even if original names differ.
    Expected final columns:
      - lead_h
      - centroid_lon
      - centroid_lat
    Optional:
      - track_id
      - time
      - genesis_id
    """

    # rename likely variants
    rename_map = {}
    for c in df.columns:
        lc = c.lower().strip()

        # lead
        if lc in ["lead_h", "lead", "leadtime", "lead_time", "lead_time_h", "leadhr", "lead_hours"]:
            rename_map[c] = "lead_h"

        # lon
        elif lc in ["centroid_lon", "lon", "longitude", "center_lon", "cent_lon"]:
            rename_map[c] = "centroid_lon"

        # lat
        elif lc in ["centroid_lat", "lat", "latitude", "center_lat", "cent_lat"]:
            rename_map[c] = "centroid_lat"

        # time
        elif lc == "time":
            rename_map[c] = "time"

        # track id
        elif lc in ["track_id", "trackid"]:
            rename_map[c] = "track_id"

        # genesis id
        elif lc in ["genesis_id", "storm_id", "sid"]:
            rename_map[c] = "genesis_id"

        # label
        elif lc in ["y", "label", "is_positive"]:
            rename_map[c] = "y"

    df = df.rename(columns=rename_map).copy()

    # if label column exists, keep only positives
    if "y" in df.columns:
        df = df[df["y"] == 1].copy()

    # required columns
    needed = ["lead_h", "centroid_lon", "centroid_lat"]
    for col in needed:
        if col not in df.columns:
            raise ValueError(f"[{dataset_name}] Missing required column: {col}. Found columns: {df.columns.tolist()}")

    # types
    df["lead_h"] = pd.to_numeric(df["lead_h"], errors="coerce")
    df["centroid_lon"] = pd.to_numeric(df["centroid_lon"], errors="coerce")
    df["centroid_lat"] = pd.to_numeric(df["centroid_lat"], errors="coerce")

    if "time" in df.columns:
        df["time"] = pd.to_datetime(df["time"], errors="coerce")

    df = df.dropna(subset=["lead_h", "centroid_lon", "centroid_lat"]).copy()
    df["lead_h"] = df["lead_h"].astype(int)
    df = df[df["lead_h"].isin(LEADS)].copy()

    # drop duplicates if possible
    dedup_cols = [c for c in ["lead_h", "track_id", "time", "centroid_lon", "centroid_lat"] if c in df.columns]
    if len(dedup_cols) > 0:
        df = df.drop_duplicates(subset=dedup_cols).copy()
    else:
        df = df.drop_duplicates().copy()

    df["dataset"] = dataset_name
    return df.reset_index(drop=True)


def load_positive_dataset(path, dataset_name):
    df = pd.read_csv(path)
    df = standardize_positive_df(df, dataset_name=dataset_name)
    print(f"[LOADED] {dataset_name}: {len(df)} positive rows")
    print(df.head())
    return df


def get_total_rows(dataset_name):
    """
    Returns total rows of full master dataset for fraction calculation.
    Priority:
      1) TOTAL_ROWS[dataset]
      2) MASTER_FILES[dataset] if provided
    """
    if TOTAL_ROWS.get(dataset_name) is not None:
        return TOTAL_ROWS[dataset_name]

    master_fp = MASTER_FILES.get(dataset_name)
    if master_fp is not None:
        if Path(master_fp).exists():
            n = len(pd.read_csv(master_fp))
            TOTAL_ROWS[dataset_name] = n
            return n

    raise ValueError(
        f"Total rows for '{dataset_name}' not available.\n"
        f"Please set TOTAL_ROWS['{dataset_name}'] manually or give MASTER_FILES['{dataset_name}']."
    )


def build_pair_summary(df_a, df_b, name_a, name_b):
    rows = []
    total_a = get_total_rows(name_a)
    total_b = get_total_rows(name_b)

    for L in LEADS:
        n_a = int((df_a["lead_h"] == L).sum())
        n_b = int((df_b["lead_h"] == L).sum())

        rows.append({
            "lead_h": L,
            f"{name_a}_count": n_a,
            f"{name_b}_count": n_b,
            f"{name_a}_fraction": n_a / total_a,
            f"{name_b}_fraction": n_b / total_b,
        })

    return pd.DataFrame(rows)


# -----------------------------
# 3) PLOTTING FUNCTIONS
# -----------------------------
def plot_pair_scatter_by_lead(df_a, df_b, name_a, name_b, save_dir=None):
    """
    Scatter comparison for each lead.
    """
    for L in LEADS:
        da = df_a[df_a["lead_h"] == L]
        db = df_b[df_b["lead_h"] == L]

        plt.figure(figsize=(10, 6))
        plt.scatter(
            da["centroid_lon"], da["centroid_lat"],
            s=25, alpha=0.6, label=name_a
        )
        plt.scatter(
            db["centroid_lon"], db["centroid_lat"],
            s=25, alpha=0.6, label=name_b
        )

        plt.xlim(100, 180)
        plt.ylim(0, 30)
        plt.xlabel("Longitude")
        plt.ylabel("Latitude")
        plt.title(f"Positive snapshots comparison ({L} h)")
        plt.legend()
        plt.tight_layout()

        if save_dir is not None:
            out = Path(save_dir) / f"scatter_{name_a}_vs_{name_b}_{L}h.png"
            plt.savefig(out, dpi=200, bbox_inches="tight")

        plt.show()


def plot_pair_counts(summary_df, name_a, name_b, save_dir=None):
    x = np.arange(len(summary_df))
    width = 0.23

    plt.figure(figsize=(9, 5))
    plt.bar(x - width/2, summary_df[f"{name_a}_count"], width=width, label=name_a)
    plt.bar(x + width/2, summary_df[f"{name_b}_count"], width=width, label=name_b)

    plt.xticks(x, summary_df["lead_h"])
    plt.xlabel("Lead time (h)")
    plt.ylabel("Positive snapshots")
    plt.title("Positive counts by lead")
    plt.legend(title="dataset")
    plt.tight_layout()

    if save_dir is not None:
        out = Path(save_dir) / f"counts_{name_a}_vs_{name_b}.png"
        plt.savefig(out, dpi=200, bbox_inches="tight")

    plt.show()


def plot_pair_fraction(summary_df, name_a, name_b, save_dir=None):
    plt.figure(figsize=(9, 5))
    plt.plot(summary_df["lead_h"], summary_df[f"{name_a}_fraction"], marker="o", label=name_a)
    plt.plot(summary_df["lead_h"], summary_df[f"{name_b}_fraction"], marker="o", label=name_b)

    plt.xlabel("Lead time (h)")
    plt.ylabel("Positive fraction")
    plt.title("Positive fraction by lead")
    plt.legend(title="dataset")
    plt.tight_layout()

    if save_dir is not None:
        out = Path(save_dir) / f"fraction_{name_a}_vs_{name_b}.png"
        plt.savefig(out, dpi=200, bbox_inches="tight")

    plt.show()


def make_all_pair_plots(df_a, df_b, name_a, name_b, outdir):
    pair_dir = Path(outdir) / f"{name_a}_vs_{name_b}"
    pair_dir.mkdir(parents=True, exist_ok=True)

    # summary table
    summary_df = build_pair_summary(df_a, df_b, name_a, name_b)
    print(f"\n===== SUMMARY: {name_a} vs {name_b} =====")
    print(summary_df)

    summary_df.to_csv(pair_dir / f"summary_{name_a}_vs_{name_b}.csv", index=False)

    # counts/fraction
    plot_pair_counts(summary_df, name_a, name_b, save_dir=pair_dir)
    plot_pair_fraction(summary_df, name_a, name_b, save_dir=pair_dir)

    # lead-specific scatter
    plot_pair_scatter_by_lead(df_a, df_b, name_a, name_b, save_dir=pair_dir)


# -----------------------------
# 4) LOAD DATA
# -----------------------------
datasets = {}
for name, fp in POS_FILES.items():
    if fp is None:
        continue
    if Path(fp).exists():
        datasets[name] = load_positive_dataset(fp, name)
    else:
        print(f"[WARNING] File not found for {name}: {fp}")

print("\nLoaded datasets:", list(datasets.keys()))


# -----------------------------
# 5) OPTIONAL: SET TOTALS MANUALLY HERE
# -----------------------------
# Example:
# TOTAL_ROWS["old"]  = 124511
# TOTAL_ROWS["soft"] = 147857
# TOTAL_ROWS["feng"] = 526xxx   # replace with your actual Feng total

print("\nCurrent TOTAL_ROWS =", TOTAL_ROWS)


# -----------------------------
# 6) RUN THE THREE PAIRS
# -----------------------------
pairs = [
    ("old", "feng"),
    ("soft", "feng"),
    ("old", "soft"),
]

for a, b in pairs:
    if a in datasets and b in datasets:
        make_all_pair_plots(datasets[a], datasets[b], a, b, OUTDIR)
    else:
        print(f"[SKIP] Missing dataset for pair: {a} vs {b}")

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# CONFIG
# ============================================================
LAB = Path(r"C:\Users\_s2218026\Documents\lab")

OLD_CASE_DIR  = LAB / "mcs_output" / "zhang_case_tables_2009_2017"
SOFT_CASE_DIR = LAB / "mcs_output_softtest" / "zhang_case_tables_soft_2009_2017"
FENG_CASE_DIR = LAB / "robustness_exact_zhang_case" / "feng_case_tables"

# full catalog/master files for fraction denominator
OLD_MASTER_FP  = LAB / "mcs_output" / "mcs_master_zhang_ready_2009_2017_JJASO.csv"
SOFT_MASTER_FP = LAB / "mcs_output_softtest" / "mcs_master_soft_features_PI_GPI_2009_2017_JJASO.csv"
FENG_STD_FP    = LAB / "robustness_exact_zhang_case" / "standardized_feng_overlap.csv"

OUTDIR = LAB / "robustness_exact_zhang_case" / "pairwise_plots_case_based"
OUTDIR.mkdir(parents=True, exist_ok=True)

LEADS = [6, 12, 24, 48]

# ============================================================
# HELPERS
# ============================================================
def detect_lat_lon_cols(df):
    lat_candidates = ["centroid_lat", "lat", "latitude"]
    lon_candidates = ["centroid_lon", "lon", "longitude"]

    lat_col = next((c for c in lat_candidates if c in df.columns), None)
    lon_col = next((c for c in lon_candidates if c in df.columns), None)

    if lat_col is None or lon_col is None:
        raise ValueError(f"Could not find lat/lon columns. Found: {df.columns.tolist()}")

    return lat_col, lon_col

def find_case_file(case_dir, dataset_name, lead):
    candidates = [
        case_dir / f"zhang_case_table_{lead}h.csv",
        case_dir / f"zhang_case_table_{dataset_name}_{lead}h.csv",
        case_dir / f"case_table_{lead}h.csv",
        case_dir / f"zhang_case_table_feng_{lead}h.csv",
    ]
    for fp in candidates:
        if fp.exists():
            return fp
    raise FileNotFoundError(f"No case-table file found for {dataset_name} lead {lead}h in {case_dir}")

def load_positive_snapshots_from_case_dir(case_dir, dataset_name):
    rows = []
    for L in LEADS:
        fp = find_case_file(case_dir, dataset_name, L)
        df = pd.read_csv(fp)

        if "y" not in df.columns:
            raise ValueError(f"{fp} has no 'y' column.")

        df = df[df["y"] == 1].copy()
        if len(df) == 0:
            continue

        lat_col, lon_col = detect_lat_lon_cols(df)

        out = pd.DataFrame({
            "lead_h": L,
            "centroid_lat": pd.to_numeric(df[lat_col], errors="coerce"),
            "centroid_lon": pd.to_numeric(df[lon_col], errors="coerce"),
        })

        if "time" in df.columns:
            out["time"] = pd.to_datetime(df["time"], errors="coerce")
        if "track_id" in df.columns:
            out["track_id"] = df["track_id"].astype(str)

        out["dataset"] = dataset_name
        out = out.dropna(subset=["centroid_lat", "centroid_lon"]).copy()
        rows.append(out)

        print(f"[LOADED] {dataset_name} {L}h: {len(out)} positives from {fp.name}")

    if len(rows) == 0:
        raise ValueError(f"No positive snapshots loaded for {dataset_name}")

    return pd.concat(rows, ignore_index=True)

def get_total_rows():
    totals = {}
    totals["old"] = len(pd.read_csv(OLD_MASTER_FP))
    totals["soft"] = len(pd.read_csv(SOFT_MASTER_FP))
    totals["feng"] = len(pd.read_csv(FENG_STD_FP))
    return totals

def build_pair_summary(df_a, df_b, name_a, name_b, total_rows):
    rows = []
    total_a = total_rows.get(name_a)
    total_b = total_rows.get(name_b)

    for L in LEADS:
        n_a = int((df_a["lead_h"] == L).sum())
        n_b = int((df_b["lead_h"] == L).sum())

        row = {
            "lead_h": L,
            f"{name_a}_count": n_a,
            f"{name_b}_count": n_b,
            f"{name_a}_fraction": n_a / total_a if total_a else np.nan,
            f"{name_b}_fraction": n_b / total_b if total_b else np.nan,
        }
        rows.append(row)

    return pd.DataFrame(rows)

def plot_pair_counts(summary_df, name_a, name_b, save_dir=None):
    x = np.arange(len(summary_df))
    width = 0.35

    plt.figure(figsize=(9, 5))
    plt.bar(x - width/2, summary_df[f"{name_a}_count"], width=width, label=name_a)
    plt.bar(x + width/2, summary_df[f"{name_b}_count"], width=width, label=name_b)
    plt.xticks(x, summary_df["lead_h"])
    plt.xlabel("Lead time (h)")
    plt.ylabel("Positive snapshots")
    plt.title("Positive counts by lead")
    plt.legend(title="dataset")
    plt.tight_layout()
    if save_dir is not None:
        plt.savefig(Path(save_dir) / f"counts_{name_a}_vs_{name_b}.png", dpi=200, bbox_inches="tight")
    plt.show()

def plot_pair_fraction(summary_df, name_a, name_b, save_dir=None):
    plt.figure(figsize=(9, 5))
    plt.plot(summary_df["lead_h"], summary_df[f"{name_a}_fraction"], marker="o", label=name_a)
    plt.plot(summary_df["lead_h"], summary_df[f"{name_b}_fraction"], marker="o", label=name_b)
    plt.xlabel("Lead time (h)")
    plt.ylabel("Positive fraction")
    plt.title("Positive fraction by lead")
    plt.legend(title="dataset")
    plt.tight_layout()
    if save_dir is not None:
        plt.savefig(Path(save_dir) / f"fraction_{name_a}_vs_{name_b}.png", dpi=200, bbox_inches="tight")
    plt.show()

def plot_pair_scatter_by_lead(df_a, df_b, name_a, name_b, save_dir=None):
    for L in LEADS:
        da = df_a[df_a["lead_h"] == L]
        db = df_b[df_b["lead_h"] == L]

        plt.figure(figsize=(10, 6))
        plt.scatter(da["centroid_lon"], da["centroid_lat"], s=25, alpha=0.6, label=name_a)
        plt.scatter(db["centroid_lon"], db["centroid_lat"], s=25, alpha=0.6, label=name_b)
        plt.xlim(100, 180)
        plt.ylim(0, 30)
        plt.xlabel("Longitude")
        plt.ylabel("Latitude")
        plt.title(f"Positive snapshots comparison ({L} h)")
        plt.legend()
        plt.tight_layout()
        if save_dir is not None:
            plt.savefig(Path(save_dir) / f"scatter_{name_a}_vs_{name_b}_{L}h.png", dpi=200, bbox_inches="tight")
        plt.show()

def make_all_pair_plots(df_a, df_b, name_a, name_b, outdir, total_rows):
    pair_dir = Path(outdir) / f"{name_a}_vs_{name_b}"
    pair_dir.mkdir(parents=True, exist_ok=True)

    summary_df = build_pair_summary(df_a, df_b, name_a, name_b, total_rows)
    print(f"\n===== SUMMARY: {name_a} vs {name_b} =====")
    print(summary_df)

    summary_df.to_csv(pair_dir / f"summary_{name_a}_vs_{name_b}.csv", index=False)

    plot_pair_counts(summary_df, name_a, name_b, save_dir=pair_dir)
    plot_pair_fraction(summary_df, name_a, name_b, save_dir=pair_dir)
    plot_pair_scatter_by_lead(df_a, df_b, name_a, name_b, save_dir=pair_dir)

# ============================================================
# LOAD POSITIVES + TOTALS + RUN
# ============================================================
old_pos  = load_positive_snapshots_from_case_dir(OLD_CASE_DIR, "old")
soft_pos = load_positive_snapshots_from_case_dir(SOFT_CASE_DIR, "soft")
feng_pos = load_positive_snapshots_from_case_dir(FENG_CASE_DIR, "feng")

TOTAL_ROWS = get_total_rows()
print("\nTOTAL_ROWS =", TOTAL_ROWS)

make_all_pair_plots(old_pos,  feng_pos, "old",  "feng", OUTDIR, TOTAL_ROWS)
make_all_pair_plots(soft_pos, feng_pos, "soft", "feng", OUTDIR, TOTAL_ROWS)
make_all_pair_plots(old_pos,  soft_pos, "old",  "soft", OUTDIR, TOTAL_ROWS)

In [ ]:
TOTAL_ROWS = {
    "old": 124511,
    "soft": 124511,
    "feng": 124511,   # only if your Feng master used for the strict robustness had the same overlap catalog size
}

In [ ]:
def check_case_period(case_dir, dataset_name):
    rows = []
    for L in [6, 12, 24, 48]:
        fp = find_case_file(case_dir, dataset_name, L)
        df = pd.read_csv(fp)
        df["time"] = pd.to_datetime(df["time"], errors="coerce")
        rows.append({
            "lead_h": L,
            "n_rows": len(df),
            "time_min": df["time"].min(),
            "time_max": df["time"].max(),
            "years": sorted(df["time"].dropna().dt.year.unique().tolist()),
            "months": sorted(df["time"].dropna().dt.month.unique().tolist()),
        })
    return pd.DataFrame(rows)

print("OLD")
print(check_case_period(OLD_CASE_DIR, "old")[["lead_h","n_rows","time_min","time_max","years","months"]])

print("\nSOFT")
print(check_case_period(SOFT_CASE_DIR, "soft")[["lead_h","n_rows","time_min","time_max","years","months"]])

print("\nFENG")
print(check_case_period(FENG_CASE_DIR, "feng")[["lead_h","n_rows","time_min","time_max","years","months"]])

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# CONFIG
# ============================================================
LAB = Path(r"C:\Users\_s2218026\Documents\lab")

OLD_CASE_DIR  = LAB / "mcs_output" / "zhang_case_tables_2009_2017"
SOFT_CASE_DIR = LAB / "mcs_output_softtest" / "zhang_case_tables_soft_2009_2017"
FENG_CASE_DIR = LAB / "robustness_exact_zhang_case" / "feng_case_tables"

OLD_MASTER_FP  = LAB / "mcs_output" / "mcs_master_zhang_ready_2009_2017_JJASO.csv"
SOFT_MASTER_FP = LAB / "mcs_output_softtest" / "mcs_master_soft_features_PI_GPI_2009_2017_JJASO.csv"
FENG_MASTER_FP = LAB / "robustness_exact_zhang_case" / "standardized_feng_overlap.csv"

OUTDIR = LAB / "robustness_exact_zhang_case" / "pairwise_plots_2015"
OUTDIR.mkdir(parents=True, exist_ok=True)

LEADS = [6, 12, 24, 48]

YEAR_START = pd.Timestamp("2011-01-01 00:00:00")
YEAR_END   = pd.Timestamp("2011-12-31 23:59:59")


# ============================================================
# HELPERS
# ============================================================
def detect_lat_lon_cols(df):
    lat_candidates = ["centroid_lat", "lat", "latitude"]
    lon_candidates = ["centroid_lon", "lon", "longitude"]

    lat_col = next((c for c in lat_candidates if c in df.columns), None)
    lon_col = next((c for c in lon_candidates if c in df.columns), None)

    if lat_col is None or lon_col is None:
        raise ValueError(f"Could not find lat/lon columns. Found: {df.columns.tolist()}")

    return lat_col, lon_col


def find_case_file(case_dir, dataset_name, lead):
    candidates = [
        case_dir / f"zhang_case_table_{lead}h.csv",
        case_dir / f"zhang_case_table_{dataset_name}_{lead}h.csv",
        case_dir / f"case_table_{lead}h.csv",
        case_dir / f"zhang_case_table_feng_{lead}h.csv",
    ]
    for fp in candidates:
        if fp.exists():
            return fp
    raise FileNotFoundError(f"No case-table file found for {dataset_name} lead {lead}h in {case_dir}")


def load_positive_snapshots_from_case_dir(case_dir, dataset_name, start_time, end_time):
    rows = []

    for L in LEADS:
        fp = find_case_file(case_dir, dataset_name, L)
        df = pd.read_csv(fp)

        if "y" not in df.columns:
            raise ValueError(f"{fp} has no 'y' column.")
        if "time" not in df.columns:
            raise ValueError(f"{fp} has no 'time' column.")

        df["time"] = pd.to_datetime(df["time"], errors="coerce")
        df = df[(df["time"] >= start_time) & (df["time"] <= end_time)].copy()
        df = df[df["y"] == 1].copy()

        if len(df) == 0:
            print(f"[LOADED] {dataset_name} {L}h: 0 positives in 2015")
            continue

        lat_col, lon_col = detect_lat_lon_cols(df)

        out = pd.DataFrame({
            "lead_h": L,
            "time": df["time"],
            "centroid_lat": pd.to_numeric(df[lat_col], errors="coerce"),
            "centroid_lon": pd.to_numeric(df[lon_col], errors="coerce"),
        })

        if "track_id" in df.columns:
            out["track_id"] = df["track_id"].astype(str)

        out["dataset"] = dataset_name
        out = out.dropna(subset=["time", "centroid_lat", "centroid_lon"]).copy()

        rows.append(out)

        print(f"[LOADED] {dataset_name} {L}h: {len(out)} positives from {fp.name}")

    if len(rows) == 0:
        raise ValueError(f"No positive snapshots loaded for {dataset_name} in 2015")

    return pd.concat(rows, ignore_index=True)


def load_catalog_in_window(fp, start_time, end_time):
    df = pd.read_csv(fp)
    if "time" not in df.columns:
        raise ValueError(f"{fp} has no 'time' column.")
    df["time"] = pd.to_datetime(df["time"], errors="coerce")
    df = df[(df["time"] >= start_time) & (df["time"] <= end_time)].copy()
    return df


old_pos_2011 = load_positive_snapshots_from_case_dir(
    OLD_CASE_DIR, "old", YEAR_START, YEAR_END
)

soft_pos_2011 = load_positive_snapshots_from_case_dir(
    SOFT_CASE_DIR, "soft", YEAR_START, YEAR_END
)

feng_pos_2011 = load_positive_snapshots_from_case_dir(
    FENG_CASE_DIR, "feng", YEAR_START, YEAR_END
)

print("\nLoaded positive snapshots in 2011:")
print("old :", len(old_pos_2011))
print("soft:", len(soft_pos_2011))
print("feng:", len(feng_pos_2011))

In [ ]:
old_master_2011  = load_catalog_in_window(OLD_MASTER_FP, YEAR_START, YEAR_END)
soft_master_2011 = load_catalog_in_window(SOFT_MASTER_FP, YEAR_START, YEAR_END)
feng_master_2011 = load_catalog_in_window(FENG_MASTER_FP, YEAR_START, YEAR_END)

TOTAL_ROWS_2011 = {
    "old": len(old_master_2011),
    "soft": len(soft_master_2011),
    "feng": len(feng_master_2011),
}

print("\nTOTAL_ROWS_2011 =", TOTAL_ROWS_2011)

In [ ]:
make_all_pair_plots(old_pos_2011,  feng_pos_2011, "old",  "feng", OUTDIR, TOTAL_ROWS_2011)
make_all_pair_plots(soft_pos_2011, feng_pos_2011, "soft", "feng", OUTDIR, TOTAL_ROWS_2011)
make_all_pair_plots(old_pos_2011,  soft_pos_2011, "old",  "soft", OUTDIR, TOTAL_ROWS_2011)

In [ ]:
print(build_pair_summary(old_pos_2011,  feng_pos_2011, "old",  "feng", TOTAL_ROWS_2011))
print(build_pair_summary(soft_pos_2011, feng_pos_2011, "soft", "feng", TOTAL_ROWS_2011))
print(build_pair_summary(old_pos_2011,  soft_pos_2011, "old",  "soft", TOTAL_ROWS_2011))

In [ ]:
c

In [ ]:
make_all_pair_plots(old_pos,  feng_pos, "old",  "feng", OUTDIR, TOTAL_ROWS)
make_all_pair_plots(soft_pos, feng_pos, "soft", "feng", OUTDIR, TOTAL_ROWS)
make_all_pair_plots(old_pos,  soft_pos, "old",  "soft", OUTDIR, TOTAL_ROWS)

In [ ]:
summary_old_feng  = build_pair_summary(old_pos,  feng_pos, "old",  "feng", TOTAL_ROWS)
summary_soft_feng = build_pair_summary(soft_pos, feng_pos, "soft", "feng", TOTAL_ROWS)
summary_old_soft  = build_pair_summary(old_pos,  soft_pos, "old",  "soft", TOTAL_ROWS)

print(summary_old_feng)
print(summary_soft_feng)
print(summary_old_soft)

In [ ]:
# ============================================================
# OLD vs NEW POSITIVE MAPS — 2009–2025
# With map background using Cartopy
#
# Outputs:
#   1. Positive counts by lead
#   2. Positive fraction by lead
#   3. Positive centroid maps with coastlines / borders
#   4. Summary CSV
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# OPTIONAL CARTOPY IMPORT
# ============================================================

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAS_CARTOPY = True
except Exception as e:
    HAS_CARTOPY = False
    print("[WARNING] Cartopy not available. Using plain matplotlib scatter map.")
    print("Reason:", repr(e))

# ============================================================
# CONFIG — 2009–2025 UPDATED DATA
# ============================================================

LAB = Path(r"C:\Users\_s2218026\Documents\lab")

OLD_CASE_DIR = (
    LAB
    / "mcs_output"
    / "exactlead_box2deg_case_tables_old_2009_2025_UPDATED"
)

NEW_CASE_DIR = (
    LAB
    / "mcs_output_softtest"
    / "exactlead_box2deg_case_tables_new_2009_2025_UPDATED"
)

OLD_MASTER_FP = (
    LAB
    / "mcs_output"
    / "mcs_master_old_exactlead_ready_2009_2025_UPDATED.csv"
)

NEW_MASTER_FP = (
    LAB
    / "mcs_output_softtest"
    / "mcs_master_new_exactlead_ready_2009_2025_UPDATED.csv"
)

OUTDIR = (
    LAB
    / "old_vs_new_positive_maps_2009_2025_with_map_background"
)

OUTDIR.mkdir(parents=True, exist_ok=True)

LEADS = [6, 12, 24, 48]

# WNP extent
LON_MIN, LON_MAX = 100, 180
LAT_MIN, LAT_MAX = 0, 30

START_YEAR = 2009
END_YEAR = 2025

LABEL_NAME = "box2deg"

# ============================================================
# HELPERS
# ============================================================

def detect_lat_lon_cols(df):
    lat_candidates = [
        "centroid_lat",
        "lat",
        "latitude",
        "mcs_lat",
        "genesis_lat",
    ]

    lon_candidates = [
        "centroid_lon",
        "lon",
        "longitude",
        "mcs_lon",
        "genesis_lon",
    ]

    lat_col = next((c for c in lat_candidates if c in df.columns), None)
    lon_col = next((c for c in lon_candidates if c in df.columns), None)

    if lat_col is None or lon_col is None:
        raise ValueError(
            "Could not find lat/lon columns.\n"
            f"Available columns:\n{df.columns.tolist()}"
        )

    return lat_col, lon_col


def find_case_file(case_dir, dataset_name, lead):
    """
    Supports current 2009–2025 exact-lead naming and older naming.
    """

    candidates = [
        case_dir / f"{LABEL_NAME}_case_table_{dataset_name}_{lead}h.csv",
        case_dir / f"zhang_case_table_{dataset_name}_{lead}h.csv",
        case_dir / f"zhang_case_table_{lead}h.csv",
        case_dir / f"case_table_{lead}h.csv",
    ]

    for fp in candidates:
        if fp.exists():
            return fp

    raise FileNotFoundError(
        f"No case-table file found for {dataset_name}, lead={lead}h.\n"
        "Checked:\n"
        + "\n".join(str(fp) for fp in candidates)
    )


def parse_time_column(df, fp):
    if "time" not in df.columns:
        raise ValueError(f"{fp} has no 'time' column.")

    df["time"] = pd.to_datetime(
        df["time"],
        errors="coerce",
        utc=True
    ).dt.tz_convert(None)

    return df


def filter_period(df, start_year=2009, end_year=2025):
    df = df.dropna(subset=["time"]).copy()

    df = df[
        (df["time"].dt.year >= start_year)
        & (df["time"].dt.year <= end_year)
    ].copy()

    return df


def load_positive_snapshots_from_case_dir(case_dir, dataset_name):
    rows = []

    for L in LEADS:
        fp = find_case_file(case_dir, dataset_name, L)

        df = pd.read_csv(fp, low_memory=False)
        df = parse_time_column(df, fp)
        df = filter_period(df, START_YEAR, END_YEAR)

        if "y" not in df.columns:
            raise ValueError(f"{fp} has no 'y' column.")

        df = df[df["y"].astype(int) == 1].copy()

        if len(df) == 0:
            print(f"[LOADED] {dataset_name.upper()} {L}h: 0 positives from {fp.name}")
            continue

        lat_col, lon_col = detect_lat_lon_cols(df)

        out = pd.DataFrame({
            "dataset": dataset_name,
            "lead_h": L,
            "time": df["time"],
            "centroid_lat": pd.to_numeric(df[lat_col], errors="coerce"),
            "centroid_lon": pd.to_numeric(df[lon_col], errors="coerce"),
        })

        if "track_id" in df.columns:
            out["track_id"] = df["track_id"].astype(str)

        if "genesis_time" in df.columns:
            out["genesis_time"] = pd.to_datetime(
                df["genesis_time"],
                errors="coerce",
                utc=True
            ).dt.tz_convert(None)

        if "genesis_lat" in df.columns:
            out["genesis_lat"] = pd.to_numeric(df["genesis_lat"], errors="coerce")

        if "genesis_lon" in df.columns:
            out["genesis_lon"] = pd.to_numeric(df["genesis_lon"], errors="coerce")

        out = out.dropna(subset=["time", "centroid_lat", "centroid_lon"]).copy()

        rows.append(out)

        print(
            f"[LOADED] {dataset_name.upper()} {L}h: "
            f"{len(out)} positives from {fp.name} | "
            f"{out['time'].min()} to {out['time'].max()}"
        )

    if len(rows) == 0:
        raise ValueError(f"No positive snapshots loaded for {dataset_name}.")

    return pd.concat(rows, ignore_index=True)


def load_catalog(fp):
    if not fp.exists():
        raise FileNotFoundError(fp)

    df = pd.read_csv(fp, low_memory=False)
    df = parse_time_column(df, fp)
    df = filter_period(df, START_YEAR, END_YEAR)

    return df


def check_case_period(case_dir, dataset_name):
    rows = []

    for L in LEADS:
        fp = find_case_file(case_dir, dataset_name, L)

        df = pd.read_csv(fp, low_memory=False)
        df = parse_time_column(df, fp)

        years = sorted(df["time"].dropna().dt.year.unique().tolist())
        months = sorted(df["time"].dropna().dt.month.unique().tolist())

        rows.append({
            "dataset": dataset_name,
            "lead_h": L,
            "file": str(fp),
            "n_rows": len(df),
            "time_min": df["time"].min(),
            "time_max": df["time"].max(),
            "years_min": min(years) if years else np.nan,
            "years_max": max(years) if years else np.nan,
            "years": years,
            "months": months,
            "positive": int(df["y"].sum()) if "y" in df.columns else np.nan,
        })

    return pd.DataFrame(rows)


def build_pair_summary(df_a, df_b, name_a, name_b, total_rows):
    rows = []

    total_a = total_rows.get(name_a)
    total_b = total_rows.get(name_b)

    for L in LEADS:
        n_a = int((df_a["lead_h"] == L).sum())
        n_b = int((df_b["lead_h"] == L).sum())

        rows.append({
            "lead_h": L,
            f"{name_a}_count": n_a,
            f"{name_b}_count": n_b,
            f"{name_a}_fraction": n_a / total_a if total_a else np.nan,
            f"{name_b}_fraction": n_b / total_b if total_b else np.nan,
        })

    return pd.DataFrame(rows)


def setup_map_ax(ax):
    if HAS_CARTOPY:
        ax.set_extent(
            [LON_MIN, LON_MAX, LAT_MIN, LAT_MAX],
            crs=ccrs.PlateCarree()
        )

        ax.add_feature(cfeature.LAND, facecolor="lightgray", alpha=0.35)
        ax.add_feature(cfeature.OCEAN, facecolor="white", alpha=1.0)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
        ax.add_feature(cfeature.BORDERS, linewidth=0.4, alpha=0.6)

        gl = ax.gridlines(
            draw_labels=True,
            linewidth=0.4,
            color="gray",
            alpha=0.5,
            linestyle="--"
        )

        gl.top_labels = False
        gl.right_labels = False

    else:
        ax.set_xlim(LON_MIN, LON_MAX)
        ax.set_ylim(LAT_MIN, LAT_MAX)
        ax.grid(True, alpha=0.3)

    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")


# ============================================================
# LOAD DATA
# ============================================================

print("\n" + "=" * 80)
print("LOADING POSITIVE SNAPSHOTS — 2009–2025")
print("=" * 80)

old_pos = load_positive_snapshots_from_case_dir(OLD_CASE_DIR, "old")
new_pos = load_positive_snapshots_from_case_dir(NEW_CASE_DIR, "new")

old_master = load_catalog(OLD_MASTER_FP)
new_master = load_catalog(NEW_MASTER_FP)

TOTAL_ROWS = {
    "old": len(old_master),
    "new": len(new_master),
}

print("\nLoaded positives:")
print("OLD:", len(old_pos))
print("NEW:", len(new_pos))

print("\nTOTAL_ROWS:")
print(TOTAL_ROWS)

print("\nOLD case period check:")
old_period = check_case_period(OLD_CASE_DIR, "old")
display(old_period[[
    "lead_h", "n_rows", "positive",
    "time_min", "time_max", "years_min", "years_max", "months"
]])

print("\nNEW case period check:")
new_period = check_case_period(NEW_CASE_DIR, "new")
display(new_period[[
    "lead_h", "n_rows", "positive",
    "time_min", "time_max", "years_min", "years_max", "months"
]])

old_period.to_csv(OUTDIR / "period_check_old_2009_2025.csv", index=False)
new_period.to_csv(OUTDIR / "period_check_new_2009_2025.csv", index=False)

# ============================================================
# SUMMARY TABLE
# ============================================================

summary_df = build_pair_summary(
    old_pos,
    new_pos,
    "old",
    "new",
    TOTAL_ROWS
)

summary_fp = OUTDIR / "summary_old_vs_new_positive_snapshots_2009_2025.csv"
summary_df.to_csv(summary_fp, index=False)

print("\nSummary:")
display(summary_df)

print("Saved summary:")
print(summary_fp)

# ============================================================
# PLOT 1 — POSITIVE COUNTS
# ============================================================

x = np.arange(len(summary_df))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(
    x - width / 2,
    summary_df["old_count"],
    width=width,
    label="OLD"
)

ax.bar(
    x + width / 2,
    summary_df["new_count"],
    width=width,
    label="NEW"
)

ax.set_xticks(x)
ax.set_xticklabels([f"{L} h" for L in summary_df["lead_h"]])
ax.set_xlabel("Lead time")
ax.set_ylabel("Positive snapshots")
ax.set_title("OLD vs NEW positive counts by lead time, 2009–2025")
ax.legend(title="Dataset")
ax.grid(True, axis="y", alpha=0.3)

plt.tight_layout()

out_png = OUTDIR / "positive_counts_old_vs_new_2009_2025.png"
plt.savefig(out_png, dpi=250, bbox_inches="tight")
plt.show()

print("Saved:", out_png)

# ============================================================
# PLOT 2 — POSITIVE FRACTION
# ============================================================

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(
    summary_df["lead_h"],
    summary_df["old_fraction"],
    marker="o",
    linewidth=2,
    label="OLD"
)

ax.plot(
    summary_df["lead_h"],
    summary_df["new_fraction"],
    marker="s",
    linewidth=2,
    label="NEW"
)

ax.set_xticks(LEADS)
ax.set_xlabel("Lead time (h)")
ax.set_ylabel("Positive fraction")
ax.set_title("OLD vs NEW positive fraction by lead time, 2009–2025")
ax.grid(True, alpha=0.3)
ax.legend(title="Dataset")

plt.tight_layout()

out_png = OUTDIR / "positive_fraction_old_vs_new_2009_2025.png"
plt.savefig(out_png, dpi=250, bbox_inches="tight")
plt.show()

print("Saved:", out_png)

# ============================================================
# PLOT 3 — POSITIVE MAPS WITH MAP BACKGROUND
# One figure, 2x2 panels for each lead
# ============================================================

if HAS_CARTOPY:
    fig, axes = plt.subplots(
        2,
        2,
        figsize=(14, 9),
        subplot_kw={"projection": ccrs.PlateCarree()},
        constrained_layout=True
    )
else:
    fig, axes = plt.subplots(
        2,
        2,
        figsize=(14, 9),
        constrained_layout=True
    )

axes = axes.flatten()

for ax, L in zip(axes, LEADS):
    da = old_pos[old_pos["lead_h"] == L].copy()
    db = new_pos[new_pos["lead_h"] == L].copy()

    setup_map_ax(ax)

    if HAS_CARTOPY:
        transform = ccrs.PlateCarree()
    else:
        transform = None

    if HAS_CARTOPY:
        ax.scatter(
            da["centroid_lon"],
            da["centroid_lat"],
            s=28,
            alpha=0.65,
            marker="o",
            label=f"OLD (n={len(da)})",
            transform=transform,
            zorder=3
        )

        ax.scatter(
            db["centroid_lon"],
            db["centroid_lat"],
            s=34,
            alpha=0.75,
            marker="x",
            label=f"NEW (n={len(db)})",
            transform=transform,
            zorder=4
        )
    else:
        ax.scatter(
            da["centroid_lon"],
            da["centroid_lat"],
            s=28,
            alpha=0.65,
            marker="o",
            label=f"OLD (n={len(da)})"
        )

        ax.scatter(
            db["centroid_lon"],
            db["centroid_lat"],
            s=34,
            alpha=0.75,
            marker="x",
            label=f"NEW (n={len(db)})"
        )

    ax.set_title(f"Positive MCS snapshots — {L} h lead")
    ax.legend(loc="lower left", fontsize=8)

fig.suptitle(
    "OLD vs NEW positive MCS centroid locations, 2009–2025\n"
    "Box2deg exact-lead labels",
    fontsize=15
)

out_png = OUTDIR / "positive_centroid_maps_old_vs_new_2009_2025_2x2_map_background.png"
plt.savefig(out_png, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", out_png)

# ============================================================
# PLOT 4 — POSITIVE MAPS, ONE FILE PER LEAD
# ============================================================

for L in LEADS:
    da = old_pos[old_pos["lead_h"] == L].copy()
    db = new_pos[new_pos["lead_h"] == L].copy()

    if HAS_CARTOPY:
        fig, ax = plt.subplots(
            figsize=(10, 6.5),
            subplot_kw={"projection": ccrs.PlateCarree()}
        )
    else:
        fig, ax = plt.subplots(figsize=(10, 6.5))

    setup_map_ax(ax)

    if HAS_CARTOPY:
        transform = ccrs.PlateCarree()

        ax.scatter(
            da["centroid_lon"],
            da["centroid_lat"],
            s=35,
            alpha=0.70,
            marker="o",
            label=f"OLD (n={len(da)})",
            transform=transform,
            zorder=3
        )

        ax.scatter(
            db["centroid_lon"],
            db["centroid_lat"],
            s=45,
            alpha=0.80,
            marker="x",
            label=f"NEW (n={len(db)})",
            transform=transform,
            zorder=4
        )

    else:
        ax.scatter(
            da["centroid_lon"],
            da["centroid_lat"],
            s=35,
            alpha=0.70,
            marker="o",
            label=f"OLD (n={len(da)})"
        )

        ax.scatter(
            db["centroid_lon"],
            db["centroid_lat"],
            s=45,
            alpha=0.80,
            marker="x",
            label=f"NEW (n={len(db)})"
        )

    ax.set_title(
        f"OLD vs NEW positive MCS centroid locations — {L} h lead\n"
        "2009–2025, box2deg exact-lead labels"
    )

    ax.legend(loc="lower left")

    plt.tight_layout()

    out_png = OUTDIR / f"positive_centroid_map_old_vs_new_{L}h_2009_2025_map_background.png"
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved:", out_png)

In [ ]:
# ============================================================
# OLD vs NEW POSITIVE MAPS — 2009–2025
# Teal / Salmon color version
#
# Outputs:
#   1. Positive counts bar plot
#   2. Positive fraction line plot
#   3. 2x2 positive centroid maps with map background
#   4. One positive centroid map per lead
#   5. Summary CSV + period-check CSV
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    display = print

# ============================================================
# OPTIONAL CARTOPY IMPORT
# ============================================================

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAS_CARTOPY = True
except Exception as e:
    HAS_CARTOPY = False
    print("[WARNING] Cartopy not available. Using plain matplotlib scatter map.")
    print("Reason:", repr(e))

# ============================================================
# CONFIG — 2009–2025 UPDATED DATA
# ============================================================

LAB = Path(r"C:\Users\_s2218026\Documents\lab")

OLD_CASE_DIR = (
    LAB
    / "mcs_output"
    / "exactlead_box2deg_case_tables_old_2009_2025_UPDATED"
)

NEW_CASE_DIR = (
    LAB
    / "mcs_output_softtest"
    / "exactlead_box2deg_case_tables_new_2009_2025_UPDATED"
)

OLD_MASTER_FP = (
    LAB
    / "mcs_output"
    / "mcs_master_old_exactlead_ready_2009_2025_UPDATED.csv"
)

NEW_MASTER_FP = (
    LAB
    / "mcs_output_softtest"
    / "mcs_master_new_exactlead_ready_2009_2025_UPDATED.csv"
)

OUTDIR = (
    LAB
    / "old_vs_new_positive_maps_2009_2025_teal_salmon"
)

OUTDIR.mkdir(parents=True, exist_ok=True)

LEADS = [6, 12, 24, 48]

# Western North Pacific extent
LON_MIN, LON_MAX = 100, 180
LAT_MIN, LAT_MAX = 0, 30

START_YEAR = 2009
END_YEAR = 2025

LABEL_NAME = "box2deg"

# ============================================================
# COLORS
# ============================================================

COLOR_OLD = "#2A9D8F"   # teal
COLOR_NEW = "#E76F51"   # salmon

COLOR_OLD_LIGHT = "#BFE3DD"
COLOR_NEW_LIGHT = "#F6C1AE"

MAP_LAND = "#F2F2F2"
MAP_OCEAN = "white"
MAP_GRID = "#9E9E9E"

# ============================================================
# HELPERS
# ============================================================

def detect_lat_lon_cols(df):
    lat_candidates = [
        "centroid_lat",
        "lat",
        "latitude",
        "mcs_lat",
        "genesis_lat",
    ]

    lon_candidates = [
        "centroid_lon",
        "lon",
        "longitude",
        "mcs_lon",
        "genesis_lon",
    ]

    lat_col = next((c for c in lat_candidates if c in df.columns), None)
    lon_col = next((c for c in lon_candidates if c in df.columns), None)

    if lat_col is None or lon_col is None:
        raise ValueError(
            "Could not find lat/lon columns.\n"
            f"Available columns:\n{df.columns.tolist()}"
        )

    return lat_col, lon_col


def find_case_file(case_dir, dataset_name, lead):
    candidates = [
        case_dir / f"{LABEL_NAME}_case_table_{dataset_name}_{lead}h.csv",
        case_dir / f"zhang_case_table_{dataset_name}_{lead}h.csv",
        case_dir / f"zhang_case_table_{lead}h.csv",
        case_dir / f"case_table_{lead}h.csv",
    ]

    for fp in candidates:
        if fp.exists():
            return fp

    raise FileNotFoundError(
        f"No case-table file found for {dataset_name}, lead={lead}h.\n"
        "Checked:\n"
        + "\n".join(str(fp) for fp in candidates)
    )


def parse_time_column(df, fp):
    if "time" not in df.columns:
        raise ValueError(f"{fp} has no 'time' column.")

    df["time"] = pd.to_datetime(
        df["time"],
        errors="coerce",
        utc=True
    ).dt.tz_convert(None)

    return df


def filter_period(df, start_year=2009, end_year=2025):
    df = df.dropna(subset=["time"]).copy()

    df = df[
        (df["time"].dt.year >= start_year)
        & (df["time"].dt.year <= end_year)
    ].copy()

    return df


def load_positive_snapshots_from_case_dir(case_dir, dataset_name):
    rows = []

    for L in LEADS:
        fp = find_case_file(case_dir, dataset_name, L)

        df = pd.read_csv(fp, low_memory=False)
        df = parse_time_column(df, fp)
        df = filter_period(df, START_YEAR, END_YEAR)

        if "y" not in df.columns:
            raise ValueError(f"{fp} has no 'y' column.")

        df = df[df["y"].astype(int) == 1].copy()

        if len(df) == 0:
            print(f"[LOADED] {dataset_name.upper()} {L}h: 0 positives from {fp.name}")
            continue

        lat_col, lon_col = detect_lat_lon_cols(df)

        out = pd.DataFrame({
            "dataset": dataset_name,
            "lead_h": L,
            "time": df["time"],
            "centroid_lat": pd.to_numeric(df[lat_col], errors="coerce"),
            "centroid_lon": pd.to_numeric(df[lon_col], errors="coerce"),
        })

        if "track_id" in df.columns:
            out["track_id"] = df["track_id"].astype(str)

        if "genesis_time" in df.columns:
            out["genesis_time"] = pd.to_datetime(
                df["genesis_time"],
                errors="coerce",
                utc=True
            ).dt.tz_convert(None)

        if "genesis_lat" in df.columns:
            out["genesis_lat"] = pd.to_numeric(df["genesis_lat"], errors="coerce")

        if "genesis_lon" in df.columns:
            out["genesis_lon"] = pd.to_numeric(df["genesis_lon"], errors="coerce")

        out = out.dropna(subset=["time", "centroid_lat", "centroid_lon"]).copy()

        rows.append(out)

        print(
            f"[LOADED] {dataset_name.upper()} {L}h: "
            f"{len(out)} positives from {fp.name} | "
            f"{out['time'].min()} to {out['time'].max()}"
        )

    if len(rows) == 0:
        raise ValueError(f"No positive snapshots loaded for {dataset_name}.")

    return pd.concat(rows, ignore_index=True)


def load_catalog(fp):
    if not fp.exists():
        raise FileNotFoundError(fp)

    df = pd.read_csv(fp, low_memory=False)
    df = parse_time_column(df, fp)
    df = filter_period(df, START_YEAR, END_YEAR)

    return df


def check_case_period(case_dir, dataset_name):
    rows = []

    for L in LEADS:
        fp = find_case_file(case_dir, dataset_name, L)

        df = pd.read_csv(fp, low_memory=False)
        df = parse_time_column(df, fp)

        years = sorted(df["time"].dropna().dt.year.unique().tolist())
        months = sorted(df["time"].dropna().dt.month.unique().tolist())

        rows.append({
            "dataset": dataset_name,
            "lead_h": L,
            "file": str(fp),
            "n_rows": len(df),
            "time_min": df["time"].min(),
            "time_max": df["time"].max(),
            "years_min": min(years) if years else np.nan,
            "years_max": max(years) if years else np.nan,
            "years": years,
            "months": months,
            "positive": int(df["y"].sum()) if "y" in df.columns else np.nan,
        })

    return pd.DataFrame(rows)


def build_pair_summary(df_a, df_b, name_a, name_b, total_rows):
    rows = []

    total_a = total_rows.get(name_a)
    total_b = total_rows.get(name_b)

    for L in LEADS:
        n_a = int((df_a["lead_h"] == L).sum())
        n_b = int((df_b["lead_h"] == L).sum())

        rows.append({
            "lead_h": L,
            f"{name_a}_count": n_a,
            f"{name_b}_count": n_b,
            f"{name_a}_fraction": n_a / total_a if total_a else np.nan,
            f"{name_b}_fraction": n_b / total_b if total_b else np.nan,
        })

    return pd.DataFrame(rows)


def setup_map_ax(ax):
    if HAS_CARTOPY:
        ax.set_extent(
            [LON_MIN, LON_MAX, LAT_MIN, LAT_MAX],
            crs=ccrs.PlateCarree()
        )

        ax.add_feature(cfeature.OCEAN, facecolor=MAP_OCEAN, alpha=1.0, zorder=0)
        ax.add_feature(cfeature.LAND, facecolor=MAP_LAND, alpha=0.85, zorder=0)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.75, zorder=1)
        ax.add_feature(cfeature.BORDERS, linewidth=0.35, alpha=0.65, zorder=1)

        gl = ax.gridlines(
            draw_labels=True,
            linewidth=0.35,
            color=MAP_GRID,
            alpha=0.45,
            linestyle="--"
        )

        gl.top_labels = False
        gl.right_labels = False

    else:
        ax.set_xlim(LON_MIN, LON_MAX)
        ax.set_ylim(LAT_MIN, LAT_MAX)
        ax.grid(True, alpha=0.3)

    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")


# ============================================================
# LOAD DATA
# ============================================================

print("\n" + "=" * 80)
print("LOADING POSITIVE SNAPSHOTS — 2009–2025")
print("=" * 80)

old_pos = load_positive_snapshots_from_case_dir(OLD_CASE_DIR, "old")
new_pos = load_positive_snapshots_from_case_dir(NEW_CASE_DIR, "new")

old_master = load_catalog(OLD_MASTER_FP)
new_master = load_catalog(NEW_MASTER_FP)

TOTAL_ROWS = {
    "old": len(old_master),
    "new": len(new_master),
}

print("\nLoaded positives:")
print("OLD:", len(old_pos))
print("NEW:", len(new_pos))

print("\nTOTAL_ROWS:")
print(TOTAL_ROWS)

print("\nOLD case period check:")
old_period = check_case_period(OLD_CASE_DIR, "old")
display(old_period[[
    "lead_h", "n_rows", "positive",
    "time_min", "time_max", "years_min", "years_max", "months"
]])

print("\nNEW case period check:")
new_period = check_case_period(NEW_CASE_DIR, "new")
display(new_period[[
    "lead_h", "n_rows", "positive",
    "time_min", "time_max", "years_min", "years_max", "months"
]])

old_period.to_csv(OUTDIR / "period_check_old_2009_2025.csv", index=False)
new_period.to_csv(OUTDIR / "period_check_new_2009_2025.csv", index=False)

old_pos.to_csv(OUTDIR / "old_positive_snapshots_2009_2025.csv", index=False)
new_pos.to_csv(OUTDIR / "new_positive_snapshots_2009_2025.csv", index=False)

# ============================================================
# SUMMARY TABLE
# ============================================================

summary_df = build_pair_summary(
    old_pos,
    new_pos,
    "old",
    "new",
    TOTAL_ROWS
)

summary_fp = OUTDIR / "summary_old_vs_new_positive_snapshots_2009_2025.csv"
summary_df.to_csv(summary_fp, index=False)

print("\nSummary:")
display(summary_df)

print("Saved summary:")
print(summary_fp)

# ============================================================
# PLOT 1 — POSITIVE COUNTS BAR PLOT
# ============================================================

x = np.arange(len(summary_df))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(
    x - width / 2,
    summary_df["old_count"],
    width=width,
    label="OLD",
    color=COLOR_OLD,
    edgecolor="black",
    linewidth=0.6
)

ax.bar(
    x + width / 2,
    summary_df["new_count"],
    width=width,
    label="NEW",
    color=COLOR_NEW,
    edgecolor="black",
    linewidth=0.6
)

ax.set_xticks(x)
ax.set_xticklabels([f"{L} h" for L in summary_df["lead_h"]])
ax.set_xlabel("Lead time")
ax.set_ylabel("Positive snapshots")
ax.set_title("OLD vs NEW positive counts by lead time, 2009–2025")
ax.legend(title="Dataset")
ax.grid(True, axis="y", alpha=0.3)

plt.tight_layout()

out_png = OUTDIR / "positive_counts_old_vs_new_2009_2025_teal_salmon.png"
plt.savefig(out_png, dpi=250, bbox_inches="tight")
plt.show()

print("Saved:", out_png)

# ============================================================
# PLOT 2 — POSITIVE FRACTION LINE PLOT
# ============================================================

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(
    summary_df["lead_h"],
    summary_df["old_fraction"],
    marker="o",
    markersize=7,
    linewidth=2.4,
    label="OLD",
    color=COLOR_OLD
)

ax.plot(
    summary_df["lead_h"],
    summary_df["new_fraction"],
    marker="s",
    markersize=7,
    linewidth=2.4,
    label="NEW",
    color=COLOR_NEW
)

ax.set_xticks(LEADS)
ax.set_xlabel("Lead time (h)")
ax.set_ylabel("Positive fraction")
ax.set_title("OLD vs NEW positive fraction by lead time, 2009–2025")
ax.grid(True, alpha=0.3)
ax.legend(title="Dataset")

plt.tight_layout()

out_png = OUTDIR / "positive_fraction_old_vs_new_2009_2025_teal_salmon.png"
plt.savefig(out_png, dpi=250, bbox_inches="tight")
plt.show()

print("Saved:", out_png)

# ============================================================
# PLOT 3 — POSITIVE MAPS WITH MAP BACKGROUND
# One figure, 2x2 panels for each lead
# ============================================================

if HAS_CARTOPY:
    fig, axes = plt.subplots(
        2,
        2,
        figsize=(14, 8.4),
        subplot_kw={"projection": ccrs.PlateCarree()},
        constrained_layout=True
    )
else:
    fig, axes = plt.subplots(
        2,
        2,
        figsize=(14, 8.4),
        constrained_layout=True
    )

axes = axes.flatten()

for ax, L in zip(axes, LEADS):
    da = old_pos[old_pos["lead_h"] == L].copy()
    db = new_pos[new_pos["lead_h"] == L].copy()

    setup_map_ax(ax)

    if HAS_CARTOPY:
        transform = ccrs.PlateCarree()

        ax.scatter(
            da["centroid_lon"],
            da["centroid_lat"],
            s=28,
            alpha=0.72,
            marker="o",
            color=COLOR_OLD,
            edgecolor="white",
            linewidth=0.35,
            label=f"OLD (n={len(da)})",
            transform=transform,
            zorder=3
        )

        ax.scatter(
            db["centroid_lon"],
            db["centroid_lat"],
            s=36,
            alpha=0.88,
            marker="x",
            color=COLOR_NEW,
            linewidth=1.25,
            label=f"NEW (n={len(db)})",
            transform=transform,
            zorder=4
        )

    else:
        ax.scatter(
            da["centroid_lon"],
            da["centroid_lat"],
            s=28,
            alpha=0.72,
            marker="o",
            color=COLOR_OLD,
            edgecolor="white",
            linewidth=0.35,
            label=f"OLD (n={len(da)})",
            zorder=3
        )

        ax.scatter(
            db["centroid_lon"],
            db["centroid_lat"],
            s=36,
            alpha=0.88,
            marker="x",
            color=COLOR_NEW,
            linewidth=1.25,
            label=f"NEW (n={len(db)})",
            zorder=4
        )

    ax.set_title(f"{L} h lead", fontsize=12)
    ax.legend(loc="lower left", fontsize=8, frameon=True)

fig.suptitle(
    "OLD vs NEW positive MCS centroid locations, 2009–2025\n"
    "box2deg exact-lead labels",
    fontsize=15
)

out_png = OUTDIR / "positive_centroid_maps_old_vs_new_2009_2025_2x2_teal_salmon.png"
plt.savefig(out_png, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", out_png)

# ============================================================
# PLOT 4 — POSITIVE MAPS, ONE FILE PER LEAD
# ============================================================

for L in LEADS:
    da = old_pos[old_pos["lead_h"] == L].copy()
    db = new_pos[new_pos["lead_h"] == L].copy()

    if HAS_CARTOPY:
        fig, ax = plt.subplots(
            figsize=(10, 6.5),
            subplot_kw={"projection": ccrs.PlateCarree()}
        )
    else:
        fig, ax = plt.subplots(figsize=(10, 6.5))

    setup_map_ax(ax)

    if HAS_CARTOPY:
        transform = ccrs.PlateCarree()

        ax.scatter(
            da["centroid_lon"],
            da["centroid_lat"],
            s=36,
            alpha=0.74,
            marker="o",
            color=COLOR_OLD,
            edgecolor="white",
            linewidth=0.35,
            label=f"OLD (n={len(da)})",
            transform=transform,
            zorder=3
        )

        ax.scatter(
            db["centroid_lon"],
            db["centroid_lat"],
            s=46,
            alpha=0.90,
            marker="x",
            color=COLOR_NEW,
            linewidth=1.35,
            label=f"NEW (n={len(db)})",
            transform=transform,
            zorder=4
        )

    else:
        ax.scatter(
            da["centroid_lon"],
            da["centroid_lat"],
            s=36,
            alpha=0.74,
            marker="o",
            color=COLOR_OLD,
            edgecolor="white",
            linewidth=0.35,
            label=f"OLD (n={len(da)})",
            zorder=3
        )

        ax.scatter(
            db["centroid_lon"],
            db["centroid_lat"],
            s=46,
            alpha=0.90,
            marker="x",
            color=COLOR_NEW,
            linewidth=1.35,
            label=f"NEW (n={len(db)})",
            zorder=4
        )

    ax.set_title(
        f"OLD vs NEW positive MCS centroid locations — {L} h lead\n"
        "2009–2025, box2deg exact-lead labels"
    )

    ax.legend(loc="lower left", frameon=True)

    plt.tight_layout()

    out_png = OUTDIR / f"positive_centroid_map_old_vs_new_{L}h_2009_2025_teal_salmon.png"
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved:", out_png)

print("\nDONE.")
print("Output directory:")
print(OUTDIR)

In [ ]:
# ============================================================
# MASTER PLOTTING CELL — OLD vs NEW, 2009–2025
# Consistent colors:
#   OLD = teal
#   NEW = salmon
#
# Produces:
#   1. Positive count / fraction plots
#   2. Positive centroid maps
#   3. ML performance comparison plots
#   4. ROC-AUC / PR-AUC comparison
#   5. ROC curves
#   6. PR curves
#   7. AdaBoost feature ablation heatmaps
#   8. AdaBoost SHAP heatmaps
# ============================================================

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
    auc,
)

from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import AdaBoostClassifier

try:
    from IPython.display import display
except Exception:
    display = print

# ============================================================
# OPTIONAL CARTOPY
# ============================================================

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAS_CARTOPY = True
except Exception as e:
    HAS_CARTOPY = False
    print("[WARNING] Cartopy not available. Positive maps will use plain axes.")
    print("Reason:", repr(e))

# ============================================================
# CONFIG
# ============================================================

LAB = Path(r"C:\Users\_s2218026\Documents\lab")

LABEL_NAME = "box2deg"
LEADS = [6, 12, 24, 48]
RANDOM_STATE = 42
N_SPLITS = 5

# Western North Pacific domain
LON_MIN, LON_MAX = 100, 180
LAT_MIN, LAT_MAX = 0, 30
START_YEAR = 2009
END_YEAR = 2025

# ------------------------------------------------------------
# Dataset paths
# ------------------------------------------------------------

PATHS = {
    "old": {
        "case_dir": LAB / "mcs_output" / "exactlead_box2deg_case_tables_old_2009_2025_UPDATED",
        "master_fp": LAB / "mcs_output" / "mcs_master_old_exactlead_ready_2009_2025_UPDATED.csv",
        "ml_dir": LAB / "mcs_output" / "ml_old_box2deg_2009_2025_final",
        "oof_candidates": [
            LAB / "mcs_output" / "ml_old_box2deg_2009_2025_final" / "oof_predictions_all_models.csv",
            LAB / "mcs_output" / "ml_old_box2deg_2009_2025_final" / "oof_predictions_allmodels_old.csv",
            LAB / "mcs_output" / "ml_old_box2deg_2009_2025_UPDATED" / "oof_predictions_all_models.csv",
            LAB / "mcs_output" / "ml_old_box2deg_2009_2025_UPDATED" / "oof_predictions_allmodels_old.csv",
        ],
        "ranked_candidates": [
            LAB / "mcs_output" / "ml_old_box2deg_2009_2025_final" / "model_performance_ranked_by_lead_old.csv",
            LAB / "mcs_output" / "ml_old_box2deg_2009_2025_UPDATED" / "model_performance_ranked_by_lead_old.csv",
        ],
    },

    "new": {
        "case_dir": LAB / "mcs_output_softtest" / "exactlead_box2deg_case_tables_new_2009_2025_UPDATED",
        "master_fp": LAB / "mcs_output_softtest" / "mcs_master_new_exactlead_ready_2009_2025_UPDATED.csv",
        "ml_dir": LAB / "mcs_output_softtest" / "ml_new_box2deg_2009_2025_UPDATED",
        "oof_candidates": [
            LAB / "mcs_output_softtest" / "ml_new_box2deg_2009_2025_UPDATED" / "oof_predictions_allmodels_new.csv",
            LAB / "mcs_output_softtest" / "ml_new_box2deg_2009_2025_UPDATED" / "oof_predictions_all_models.csv",
            LAB / "mcs_output_softtest" / "ml_new_box2deg_2009_2025_UPDATED" / "oof_predictions_allmodels_soft.csv",
        ],
        "ranked_candidates": [
            LAB / "mcs_output_softtest" / "ml_new_box2deg_2009_2025_UPDATED" / "model_performance_ranked_by_lead_new.csv",
            LAB / "mcs_output_softtest" / "ml_new_box2deg_2009_2025_UPDATED" / "model_performance_ranked_by_lead_soft.csv",
        ],
    },
}

OUTDIR = LAB / "OLD_NEW_MASTER_PLOTS_2009_2025_TEAL_SALMON"
OUTDIR.mkdir(parents=True, exist_ok=True)

PLOT_DIR = OUTDIR / "plots"
TABLE_DIR = OUTDIR / "tables"
PLOT_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# COLORS
# ============================================================

COLOR_OLD = "#2A9D8F"   # teal
COLOR_NEW = "#E76F51"   # salmon

DATASET_COLORS = {
    "old": COLOR_OLD,
    "new": COLOR_NEW,
}

DATASET_LABELS = {
    "old": "OLD",
    "new": "NEW",
}

LEAD_STYLES = {
    6: "-",
    12: "--",
    24: "-.",
    48: ":",
}

MAP_LAND = "#F2F2F2"
MAP_OCEAN = "white"
MAP_GRID = "#9E9E9E"

# ============================================================
# FEATURE LISTS
# ============================================================

# For interpretation plots, exclude lon / lat / month.
# The trained ML outputs may still include them; this only affects SHAP/ablation recomputation.
INTERPRET_FEATURES = [
    "eta850_abs",
    "rh600",
    "wshear_200_850",
    "pi_vmax_ms",
    "gpi",
    "area_km2",
    "tb_mean",
    "tb_min",
    "speed_kmh",
    "direction_deg",
]

MODEL_ORDER = [
    "Logistic Regression",
    "Naive Bayes",
    "KNN",
    "SVM",
    "Decision Tree",
    "MLP",
    "QDA",
    "Random Forest",
    "AdaBoost",
]

# ============================================================
# SWITCHES
# ============================================================

RUN_POSITIVE_MAPS = True
RUN_ML_PERFORMANCE = True
RUN_ROC_PR_CURVES = True

# These two can be slow.
RUN_FEATURE_ABLATION = True
RUN_SHAP = True

# For best-model selection in comparison plots:
#   "default" uses threshold 0.5
#   "tuned" uses best F1 threshold from OOF scores
MAIN_THRESHOLD_TYPE = "default"

# SHAP sample settings
SHAP_BACKGROUND_N = 80
SHAP_EXPLAIN_N = 300

# ============================================================
# GENERAL HELPERS
# ============================================================

def first_existing(candidates, label):
    for fp in candidates:
        if fp.exists():
            return fp
    raise FileNotFoundError(
        f"No file found for {label}. Checked:\n"
        + "\n".join(str(fp) for fp in candidates)
    )


def clean_df(df):
    df = df.loc[:, ~df.columns.astype(str).str.startswith("Unnamed")].copy()
    return df


def safe_metrics(y_true, y_score, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)

    y_pred = (y_score >= threshold).astype(int)

    out = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }

    try:
        out["roc_auc"] = roc_auc_score(y_true, y_score)
    except Exception:
        out["roc_auc"] = np.nan

    try:
        out["pr_auc"] = average_precision_score(y_true, y_score)
    except Exception:
        out["pr_auc"] = np.nan

    return out


def threshold_scan(y_true, y_score):
    rows = []
    thresholds = np.round(np.arange(0.05, 0.951, 0.01), 2)

    for thr in thresholds:
        m = safe_metrics(y_true, y_score, threshold=thr)
        rows.append({
            "threshold": float(thr),
            **m,
        })

    scan = pd.DataFrame(rows)

    scan_ranked = (
        scan
        .sort_values(["f1", "recall", "precision"], ascending=[False, False, False])
        .reset_index(drop=True)
    )

    best_thr = float(scan_ranked.loc[0, "threshold"])

    return scan, best_thr


def find_case_file(case_dir, dataset_name, lead):
    candidates = [
        case_dir / f"{LABEL_NAME}_case_table_{dataset_name}_{lead}h.csv",
        case_dir / f"zhang_case_table_{dataset_name}_{lead}h.csv",
        case_dir / f"zhang_case_table_{lead}h.csv",
        case_dir / f"case_table_{lead}h.csv",
    ]

    for fp in candidates:
        if fp.exists():
            return fp

    raise FileNotFoundError(
        f"No case-table file found for {dataset_name}, lead={lead}h.\n"
        + "\n".join(str(fp) for fp in candidates)
    )


def parse_time_column(df, fp):
    if "time" not in df.columns:
        raise ValueError(f"{fp} has no 'time' column.")

    df["time"] = pd.to_datetime(
        df["time"],
        errors="coerce",
        utc=True
    ).dt.tz_convert(None)

    return df


def filter_period(df, start_year=2009, end_year=2025):
    df = df.dropna(subset=["time"]).copy()
    df = df[
        (df["time"].dt.year >= start_year)
        & (df["time"].dt.year <= end_year)
    ].copy()
    return df


def detect_lat_lon_cols(df):
    lat_candidates = [
        "centroid_lat",
        "lat",
        "latitude",
        "mcs_lat",
        "genesis_lat",
    ]

    lon_candidates = [
        "centroid_lon",
        "lon",
        "longitude",
        "mcs_lon",
        "genesis_lon",
    ]

    lat_col = next((c for c in lat_candidates if c in df.columns), None)
    lon_col = next((c for c in lon_candidates if c in df.columns), None)

    if lat_col is None or lon_col is None:
        raise ValueError(
            "Could not find lat/lon columns.\n"
            f"Available columns:\n{df.columns.tolist()}"
        )

    return lat_col, lon_col


# ============================================================
# LOAD OOF PREDICTIONS AND BUILD PERFORMANCE SUMMARY
# ============================================================

def load_oof_predictions(dataset_name):
    fp = first_existing(PATHS[dataset_name]["oof_candidates"], f"{dataset_name} OOF predictions")
    df = pd.read_csv(fp, low_memory=False)
    df = clean_df(df)

    required = ["lead_h", "model", "y_true", "y_score"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in {fp}: {missing}")

    print(f"[OOF] {dataset_name.upper()} loaded from:")
    print(fp)
    print("Rows:", len(df))
    print("Models:", sorted(df["model"].dropna().unique().tolist()))

    return df, fp


def build_performance_summary_from_oof(oof, dataset_name):
    rows = []
    scan_rows = []

    for (lead, model), sub in oof.groupby(["lead_h", "model"]):
        sub = sub.dropna(subset=["y_true", "y_score"]).copy()

        if len(sub) == 0:
            continue

        y_true = sub["y_true"].astype(int).values
        y_score = sub["y_score"].astype(float).values

        scan, best_thr = threshold_scan(y_true, y_score)
        scan["dataset"] = dataset_name
        scan["lead_h"] = int(lead)
        scan["model"] = model
        scan_rows.append(scan)

        default_metrics = safe_metrics(y_true, y_score, threshold=0.5)
        tuned_metrics = safe_metrics(y_true, y_score, threshold=best_thr)

        rows.append({
            "dataset": dataset_name,
            "lead_h": int(lead),
            "model": model,
            "threshold_type": "default",
            "threshold": 0.5,
            "n": len(sub),
            "positive": int(y_true.sum()),
            "negative": int((y_true == 0).sum()),
            **default_metrics,
        })

        rows.append({
            "dataset": dataset_name,
            "lead_h": int(lead),
            "model": model,
            "threshold_type": "tuned",
            "threshold": best_thr,
            "n": len(sub),
            "positive": int(y_true.sum()),
            "negative": int((y_true == 0).sum()),
            **tuned_metrics,
        })

    summary = pd.DataFrame(rows)
    scans = pd.concat(scan_rows, ignore_index=True) if scan_rows else pd.DataFrame()

    return summary, scans


def get_best_by_lead(perf_df, threshold_type="default"):
    sub = perf_df[perf_df["threshold_type"] == threshold_type].copy()

    best = (
        sub
        .sort_values(["dataset", "lead_h", "f1"], ascending=[True, True, False])
        .groupby(["dataset", "lead_h"], as_index=False)
        .first()
    )

    return best


# ============================================================
# POSITIVE MAP DATA
# ============================================================

def load_positive_snapshots_from_case_dir(case_dir, dataset_name):
    rows = []

    for L in LEADS:
        fp = find_case_file(case_dir, dataset_name, L)

        df = pd.read_csv(fp, low_memory=False)
        df = clean_df(df)
        df = parse_time_column(df, fp)
        df = filter_period(df, START_YEAR, END_YEAR)

        if "y" not in df.columns:
            raise ValueError(f"{fp} has no 'y' column.")

        df = df[df["y"].astype(int) == 1].copy()

        if len(df) == 0:
            print(f"[LOADED] {dataset_name.upper()} {L}h: 0 positives from {fp.name}")
            continue

        lat_col, lon_col = detect_lat_lon_cols(df)

        out = pd.DataFrame({
            "dataset": dataset_name,
            "lead_h": L,
            "time": df["time"],
            "centroid_lat": pd.to_numeric(df[lat_col], errors="coerce"),
            "centroid_lon": pd.to_numeric(df[lon_col], errors="coerce"),
        })

        if "track_id" in df.columns:
            out["track_id"] = df["track_id"].astype(str)

        out = out.dropna(subset=["time", "centroid_lat", "centroid_lon"]).copy()
        rows.append(out)

        print(
            f"[LOADED] {dataset_name.upper()} {L}h: "
            f"{len(out)} positives from {fp.name} | "
            f"{out['time'].min()} to {out['time'].max()}"
        )

    if len(rows) == 0:
        raise ValueError(f"No positive snapshots loaded for {dataset_name}.")

    return pd.concat(rows, ignore_index=True)


def load_master_count(dataset_name):
    fp = PATHS[dataset_name]["master_fp"]

    if not fp.exists():
        print(f"[WARNING] Master file not found for {dataset_name}: {fp}")
        return np.nan

    df = pd.read_csv(fp, low_memory=False)
    df = clean_df(df)
    df = parse_time_column(df, fp)
    df = filter_period(df, START_YEAR, END_YEAR)

    return len(df)


def setup_map_ax(ax):
    if HAS_CARTOPY:
        ax.set_extent(
            [LON_MIN, LON_MAX, LAT_MIN, LAT_MAX],
            crs=ccrs.PlateCarree()
        )

        ax.add_feature(cfeature.OCEAN, facecolor=MAP_OCEAN, alpha=1.0, zorder=0)
        ax.add_feature(cfeature.LAND, facecolor=MAP_LAND, alpha=0.85, zorder=0)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.75, zorder=1)
        ax.add_feature(cfeature.BORDERS, linewidth=0.35, alpha=0.65, zorder=1)

        gl = ax.gridlines(
            draw_labels=True,
            linewidth=0.35,
            color=MAP_GRID,
            alpha=0.45,
            linestyle="--"
        )

        gl.top_labels = False
        gl.right_labels = False

    else:
        ax.set_xlim(LON_MIN, LON_MAX)
        ax.set_ylim(LAT_MIN, LAT_MAX)
        ax.grid(True, alpha=0.3)

    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")


# ============================================================
# PLOT: POSITIVE COUNTS / FRACTION / MAPS
# ============================================================

if RUN_POSITIVE_MAPS:
    print("\n" + "=" * 90)
    print("POSITIVE MAPS AND POSITIVE COUNT PLOTS")
    print("=" * 90)

    old_pos = load_positive_snapshots_from_case_dir(PATHS["old"]["case_dir"], "old")
    new_pos = load_positive_snapshots_from_case_dir(PATHS["new"]["case_dir"], "new")

    old_total = load_master_count("old")
    new_total = load_master_count("new")

    old_pos.to_csv(TABLE_DIR / "old_positive_snapshots_2009_2025.csv", index=False)
    new_pos.to_csv(TABLE_DIR / "new_positive_snapshots_2009_2025.csv", index=False)

    summary_rows = []

    for L in LEADS:
        old_n = int((old_pos["lead_h"] == L).sum())
        new_n = int((new_pos["lead_h"] == L).sum())

        summary_rows.append({
            "lead_h": L,
            "old_count": old_n,
            "new_count": new_n,
            "old_fraction": old_n / old_total if pd.notna(old_total) else np.nan,
            "new_fraction": new_n / new_total if pd.notna(new_total) else np.nan,
        })

    positive_summary = pd.DataFrame(summary_rows)
    positive_summary.to_csv(TABLE_DIR / "positive_summary_old_vs_new_2009_2025.csv", index=False)

    print("\nPositive summary:")
    display(positive_summary)

    # Positive count bar plot
    x = np.arange(len(positive_summary))
    width = 0.35

    fig, ax = plt.subplots(figsize=(9, 5))

    ax.bar(
        x - width / 2,
        positive_summary["old_count"],
        width=width,
        label="OLD",
        color=COLOR_OLD,
        edgecolor="black",
        linewidth=0.6
    )

    ax.bar(
        x + width / 2,
        positive_summary["new_count"],
        width=width,
        label="NEW",
        color=COLOR_NEW,
        edgecolor="black",
        linewidth=0.6
    )

    ax.set_xticks(x)
    ax.set_xticklabels([f"{L} h" for L in positive_summary["lead_h"]])
    ax.set_xlabel("Lead time")
    ax.set_ylabel("Positive snapshots")
    ax.set_title("OLD vs NEW positive counts by lead time, 2009–2025")
    ax.legend(title="Dataset")
    ax.grid(True, axis="y", alpha=0.3)

    plt.tight_layout()
    out_png = PLOT_DIR / "positive_counts_old_vs_new_2009_2025_teal_salmon.png"
    plt.savefig(out_png, dpi=250, bbox_inches="tight")
    plt.show()
    print("Saved:", out_png)

    # Positive fraction line plot
    fig, ax = plt.subplots(figsize=(9, 5))

    ax.plot(
        positive_summary["lead_h"],
        positive_summary["old_fraction"],
        marker="o",
        markersize=7,
        linewidth=2.4,
        label="OLD",
        color=COLOR_OLD
    )

    ax.plot(
        positive_summary["lead_h"],
        positive_summary["new_fraction"],
        marker="s",
        markersize=7,
        linewidth=2.4,
        label="NEW",
        color=COLOR_NEW
    )

    ax.set_xticks(LEADS)
    ax.set_xlabel("Lead time (h)")
    ax.set_ylabel("Positive fraction")
    ax.set_title("OLD vs NEW positive fraction by lead time, 2009–2025")
    ax.grid(True, alpha=0.3)
    ax.legend(title="Dataset")

    plt.tight_layout()
    out_png = PLOT_DIR / "positive_fraction_old_vs_new_2009_2025_teal_salmon.png"
    plt.savefig(out_png, dpi=250, bbox_inches="tight")
    plt.show()
    print("Saved:", out_png)

    # Positive centroid maps
    if HAS_CARTOPY:
        fig, axes = plt.subplots(
            2, 2,
            figsize=(14, 8.4),
            subplot_kw={"projection": ccrs.PlateCarree()},
            constrained_layout=True
        )
    else:
        fig, axes = plt.subplots(2, 2, figsize=(14, 8.4), constrained_layout=True)

    axes = axes.flatten()

    for ax, L in zip(axes, LEADS):
        da = old_pos[old_pos["lead_h"] == L].copy()
        db = new_pos[new_pos["lead_h"] == L].copy()

        setup_map_ax(ax)

        transform = ccrs.PlateCarree() if HAS_CARTOPY else None

        scatter_kwargs_old = dict(
            s=28,
            alpha=0.72,
            marker="o",
            color=COLOR_OLD,
            edgecolor="white",
            linewidth=0.35,
            label=f"OLD (n={len(da)})",
            zorder=3
        )

        scatter_kwargs_new = dict(
            s=36,
            alpha=0.88,
            marker="x",
            color=COLOR_NEW,
            linewidth=1.25,
            label=f"NEW (n={len(db)})",
            zorder=4
        )

        if HAS_CARTOPY:
            scatter_kwargs_old["transform"] = transform
            scatter_kwargs_new["transform"] = transform

        ax.scatter(da["centroid_lon"], da["centroid_lat"], **scatter_kwargs_old)
        ax.scatter(db["centroid_lon"], db["centroid_lat"], **scatter_kwargs_new)

        ax.set_title(f"{L} h lead", fontsize=12)
        ax.legend(loc="lower left", fontsize=8, frameon=True)

    fig.suptitle(
        "OLD vs NEW positive MCS centroid locations, 2009–2025\n"
        "box2deg exact-lead labels",
        fontsize=15
    )

    out_png = PLOT_DIR / "positive_centroid_maps_old_vs_new_2009_2025_2x2_teal_salmon.png"
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", out_png)

# ============================================================
# LOAD ML PERFORMANCE DATA
# ============================================================

if RUN_ML_PERFORMANCE or RUN_ROC_PR_CURVES:
    print("\n" + "=" * 90)
    print("LOADING ML OUTPUTS")
    print("=" * 90)

    oof_all_dict = {}
    perf_list = []
    threshold_scan_list = []

    for ds in ["old", "new"]:
        oof, oof_fp = load_oof_predictions(ds)
        oof_all_dict[ds] = oof

        perf, scans = build_performance_summary_from_oof(oof, ds)
        perf_list.append(perf)
        threshold_scan_list.append(scans)

    perf_all = pd.concat(perf_list, ignore_index=True)
    threshold_scans_all = pd.concat(threshold_scan_list, ignore_index=True)

    perf_all.to_csv(TABLE_DIR / "performance_summary_default_and_tuned_old_vs_new.csv", index=False)
    threshold_scans_all.to_csv(TABLE_DIR / "threshold_scans_old_vs_new.csv", index=False)

    best_default = get_best_by_lead(perf_all, "default")
    best_tuned = get_best_by_lead(perf_all, "tuned")

    best_default.to_csv(TABLE_DIR / "best_model_by_lead_default_old_vs_new.csv", index=False)
    best_tuned.to_csv(TABLE_DIR / "best_model_by_lead_tuned_old_vs_new.csv", index=False)

    print("\nBest model by lead — default threshold:")
    display(best_default)

    print("\nBest model by lead — tuned threshold:")
    display(best_tuned)

# ============================================================
# PLOT: ML PERFORMANCE COMPARISON
# ============================================================

if RUN_ML_PERFORMANCE:
    print("\n" + "=" * 90)
    print("ML PERFORMANCE PLOTS")
    print("=" * 90)

    def plot_best_metric_panel(best_df, threshold_type):
        metrics = [
            ("f1", "F1"),
            ("precision", "Precision"),
            ("recall", "Recall"),
        ]

        fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), constrained_layout=True)

        for ax, (metric, label) in zip(axes, metrics):
            for ds in ["old", "new"]:
                sub = best_df[best_df["dataset"] == ds].sort_values("lead_h")

                ax.plot(
                    sub["lead_h"],
                    sub[metric],
                    marker="o" if ds == "old" else "s",
                    linewidth=2.4,
                    markersize=7,
                    color=DATASET_COLORS[ds],
                    label=DATASET_LABELS[ds],
                )

                # annotate model names
                for _, r in sub.iterrows():
                    ax.text(
                        r["lead_h"],
                        r[metric] + 0.008,
                        str(r["model"]),
                        fontsize=7,
                        ha="center",
                        color=DATASET_COLORS[ds]
                    )

            ax.set_xticks(LEADS)
            ax.set_xlabel("Lead time (h)")
            ax.set_ylabel(label)
            ax.set_title(label)
            ax.grid(True, alpha=0.3)

        axes[0].legend(title="Dataset")

        fig.suptitle(
            f"OLD vs NEW best-model performance ({threshold_type} threshold)",
            fontsize=15
        )

        out_png = PLOT_DIR / f"best_model_f1_precision_recall_{threshold_type}_old_vs_new_teal_salmon.png"
        plt.savefig(out_png, dpi=250, bbox_inches="tight")
        plt.show()
        print("Saved:", out_png)

    plot_best_metric_panel(best_default, "default")
    plot_best_metric_panel(best_tuned, "tuned")

    # ROC-AUC and PR-AUC comparison
    def plot_auc_panel(best_df, threshold_type):
        fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), constrained_layout=True)

        for ax, metric, label in [
            (axes[0], "roc_auc", "ROC-AUC"),
            (axes[1], "pr_auc", "PR-AUC"),
        ]:
            for ds in ["old", "new"]:
                sub = best_df[best_df["dataset"] == ds].sort_values("lead_h")

                ax.plot(
                    sub["lead_h"],
                    sub[metric],
                    marker="o" if ds == "old" else "s",
                    linewidth=2.4,
                    markersize=7,
                    color=DATASET_COLORS[ds],
                    label=DATASET_LABELS[ds],
                )

                for _, r in sub.iterrows():
                    ax.text(
                        r["lead_h"],
                        r[metric] + 0.006,
                        str(r["model"]),
                        fontsize=7,
                        ha="center",
                        color=DATASET_COLORS[ds]
                    )

            ax.set_xticks(LEADS)
            ax.set_xlabel("Lead time (h)")
            ax.set_ylabel(label)
            ax.set_title(label)
            ax.grid(True, alpha=0.3)

        axes[0].legend(title="Dataset")

        fig.suptitle(
            f"OLD vs NEW best-model ROC-AUC and PR-AUC ({threshold_type} selected model)",
            fontsize=15
        )

        out_png = PLOT_DIR / f"best_model_roc_auc_pr_auc_{threshold_type}_old_vs_new_teal_salmon.png"
        plt.savefig(out_png, dpi=250, bbox_inches="tight")
        plt.show()
        print("Saved:", out_png)

    plot_auc_panel(best_default, "default")
    plot_auc_panel(best_tuned, "tuned")

    # F1 bar plots for all models, each lead
    perf_default = perf_all[perf_all["threshold_type"] == "default"].copy()

    fig, axes = plt.subplots(2, 2, figsize=(15, 9), constrained_layout=True)
    axes = axes.flatten()

    for ax, L in zip(axes, LEADS):
        sub = perf_default[perf_default["lead_h"] == L].copy()

        pivot = sub.pivot_table(
            index="model",
            columns="dataset",
            values="f1",
            aggfunc="first"
        )

        ordered_models = [m for m in MODEL_ORDER if m in pivot.index]
        remaining = [m for m in pivot.index if m not in ordered_models]
        pivot = pivot.reindex(ordered_models + remaining)

        x = np.arange(len(pivot.index))
        width = 0.36

        ax.bar(
            x - width / 2,
            pivot.get("old", pd.Series(index=pivot.index, dtype=float)),
            width=width,
            color=COLOR_OLD,
            label="OLD",
            edgecolor="black",
            linewidth=0.4
        )

        ax.bar(
            x + width / 2,
            pivot.get("new", pd.Series(index=pivot.index, dtype=float)),
            width=width,
            color=COLOR_NEW,
            label="NEW",
            edgecolor="black",
            linewidth=0.4
        )

        ax.set_title(f"{L} h lead")
        ax.set_ylabel("F1")
        ax.set_xticks(x)
        ax.set_xticklabels(pivot.index, rotation=35, ha="right", fontsize=8)
        ax.grid(True, axis="y", alpha=0.3)

    axes[0].legend(title="Dataset")

    fig.suptitle(
        "OLD vs NEW F1 score by model and lead time\nDefault threshold",
        fontsize=15
    )

    out_png = PLOT_DIR / "all_models_f1_by_lead_default_old_vs_new_teal_salmon.png"
    plt.savefig(out_png, dpi=250, bbox_inches="tight")
    plt.show()
    print("Saved:", out_png)

# ============================================================
# PLOT: ROC AND PR CURVES
# ============================================================

if RUN_ROC_PR_CURVES:
    print("\n" + "=" * 90)
    print("ROC AND PR CURVES")
    print("=" * 90)

    best_for_curves = best_default if MAIN_THRESHOLD_TYPE == "default" else best_tuned

    # ROC curves: one 2x2 figure
    fig, axes = plt.subplots(2, 2, figsize=(12, 9), constrained_layout=True)
    axes = axes.flatten()

    for ax, L in zip(axes, LEADS):
        for ds in ["old", "new"]:
            model_name = best_for_curves[
                (best_for_curves["dataset"] == ds)
                & (best_for_curves["lead_h"] == L)
            ]["model"].iloc[0]

            oof = oof_all_dict[ds]
            sub = oof[(oof["lead_h"] == L) & (oof["model"] == model_name)].copy()

            y_true = sub["y_true"].astype(int).values
            y_score = sub["y_score"].astype(float).values

            fpr, tpr, _ = roc_curve(y_true, y_score)
            roc_auc = auc(fpr, tpr)

            ax.plot(
                fpr,
                tpr,
                color=DATASET_COLORS[ds],
                linewidth=2.2,
                label=f"{DATASET_LABELS[ds]}: {model_name} (AUC={roc_auc:.3f})"
            )

        ax.plot([0, 1], [0, 1], color="gray", linestyle="--", linewidth=1)
        ax.set_title(f"ROC curve — {L} h lead")
        ax.set_xlabel("False Positive Rate")
        ax.set_ylabel("True Positive Rate")
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)

    fig.suptitle(
        f"ROC curves of best-F1 models ({MAIN_THRESHOLD_TYPE} selected)",
        fontsize=15
    )

    out_png = PLOT_DIR / f"roc_curves_best_models_{MAIN_THRESHOLD_TYPE}_old_vs_new_teal_salmon.png"
    plt.savefig(out_png, dpi=250, bbox_inches="tight")
    plt.show()
    print("Saved:", out_png)

    # PR curves: one 2x2 figure
    fig, axes = plt.subplots(2, 2, figsize=(12, 9), constrained_layout=True)
    axes = axes.flatten()

    for ax, L in zip(axes, LEADS):
        for ds in ["old", "new"]:
            model_name = best_for_curves[
                (best_for_curves["dataset"] == ds)
                & (best_for_curves["lead_h"] == L)
            ]["model"].iloc[0]

            oof = oof_all_dict[ds]
            sub = oof[(oof["lead_h"] == L) & (oof["model"] == model_name)].copy()

            y_true = sub["y_true"].astype(int).values
            y_score = sub["y_score"].astype(float).values

            precision, recall, _ = precision_recall_curve(y_true, y_score)
            pr_auc = average_precision_score(y_true, y_score)

            ax.plot(
                recall,
                precision,
                color=DATASET_COLORS[ds],
                linewidth=2.2,
                label=f"{DATASET_LABELS[ds]}: {model_name} (AP={pr_auc:.3f})"
            )

        ax.set_title(f"Precision-recall curve — {L} h lead")
        ax.set_xlabel("Recall")
        ax.set_ylabel("Precision")
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)

    fig.suptitle(
        f"PR curves of best-F1 models ({MAIN_THRESHOLD_TYPE} selected)",
        fontsize=15
    )

    out_png = PLOT_DIR / f"pr_curves_best_models_{MAIN_THRESHOLD_TYPE}_old_vs_new_teal_salmon.png"
    plt.savefig(out_png, dpi=250, bbox_inches="tight")
    plt.show()
    print("Saved:", out_png)

# ============================================================
# FEATURE ABLATION — AdaBoost, no lon/lat/month
# ============================================================

def make_adaboost(random_state=42):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", AdaBoostClassifier(
            n_estimators=300,
            random_state=random_state
        ))
    ])


def make_cv(y, groups=None, n_splits=5, random_state=42):
    y = np.asarray(y).astype(int)
    min_class_count = np.bincount(y).min()
    n_splits_eff = min(n_splits, int(min_class_count))

    if n_splits_eff < 2:
        raise ValueError("Not enough class samples for cross-validation.")

    if groups is not None:
        try:
            cv = StratifiedGroupKFold(
                n_splits=n_splits_eff,
                shuffle=True,
                random_state=random_state
            )
            return cv, "StratifiedGroupKFold", n_splits_eff
        except Exception:
            pass

    cv = StratifiedKFold(
        n_splits=n_splits_eff,
        shuffle=True,
        random_state=random_state
    )
    return cv, "StratifiedKFold", n_splits_eff


def load_case_df(dataset_name, lead):
    fp = find_case_file(PATHS[dataset_name]["case_dir"], dataset_name, lead)
    df = pd.read_csv(fp, low_memory=False)
    df = clean_df(df)

    if "time" in df.columns:
        df = parse_time_column(df, fp)
        df = filter_period(df, START_YEAR, END_YEAR)

    missing = [c for c in INTERPRET_FEATURES + ["y"] if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in {fp.name}: {missing}")

    df = df.dropna(subset=["y"]).copy()
    return df


def evaluate_adaboost_cv(df, feature_cols, random_state=42, n_splits=5):
    X = df[feature_cols].copy()
    y = df["y"].astype(int).values
    groups = df["track_id"].astype(str).values if "track_id" in df.columns else None

    cv, cv_name, n_splits_eff = make_cv(
        y=y,
        groups=groups,
        n_splits=n_splits,
        random_state=random_state
    )

    y_score_oof = np.full(len(df), np.nan)

    splitter = cv.split(X, y, groups) if groups is not None else cv.split(X, y)

    for fold_id, (train_idx, test_idx) in enumerate(splitter, start=1):
        pipe = make_adaboost(random_state=random_state + fold_id)
        pipe.fit(X.iloc[train_idx], y[train_idx])
        y_score_oof[test_idx] = pipe.predict_proba(X.iloc[test_idx])[:, 1]

    scan, best_thr = threshold_scan(y, y_score_oof)

    out = {
        "default_threshold": 0.5,
        "tuned_threshold": best_thr,
        "default_f1": safe_metrics(y, y_score_oof, 0.5)["f1"],
        "tuned_f1": safe_metrics(y, y_score_oof, best_thr)["f1"],
        "roc_auc": safe_metrics(y, y_score_oof, 0.5)["roc_auc"],
        "pr_auc": safe_metrics(y, y_score_oof, 0.5)["pr_auc"],
    }

    return out


if RUN_FEATURE_ABLATION:
    print("\n" + "=" * 90)
    print("FEATURE ABLATION — AdaBoost, no lon/lat/month")
    print("=" * 90)

    ablation_rows = []

    for ds in ["old", "new"]:
        for L in LEADS:
            df = load_case_df(ds, L)

            print(f"\n{ds.upper()} | {L}h | rows={len(df)} | pos={int(df['y'].sum())}")

            baseline = evaluate_adaboost_cv(
                df=df,
                feature_cols=INTERPRET_FEATURES,
                random_state=RANDOM_STATE + L,
                n_splits=N_SPLITS
            )

            for removed_feature in INTERPRET_FEATURES:
                reduced_features = [c for c in INTERPRET_FEATURES if c != removed_feature]

                ablated = evaluate_adaboost_cv(
                    df=df,
                    feature_cols=reduced_features,
                    random_state=RANDOM_STATE + L,
                    n_splits=N_SPLITS
                )

                row = {
                    "dataset": ds,
                    "lead_h": L,
                    "model": "AdaBoost",
                    "removed_feature": removed_feature,
                    "baseline_default_f1": baseline["default_f1"],
                    "ablated_default_f1": ablated["default_f1"],
                    "delta_default_f1": baseline["default_f1"] - ablated["default_f1"],
                    "baseline_tuned_f1": baseline["tuned_f1"],
                    "ablated_tuned_f1": ablated["tuned_f1"],
                    "delta_tuned_f1": baseline["tuned_f1"] - ablated["tuned_f1"],
                    "baseline_tuned_threshold": baseline["tuned_threshold"],
                    "ablated_tuned_threshold": ablated["tuned_threshold"],
                }

                ablation_rows.append(row)

                print(
                    f"remove {removed_feature:18s} | "
                    f"default ΔF1={row['delta_default_f1']:+.4f} | "
                    f"tuned ΔF1={row['delta_tuned_f1']:+.4f}"
                )

    ablation_df = pd.DataFrame(ablation_rows)
    ablation_df.to_csv(TABLE_DIR / "adaboost_feature_ablation_old_vs_new_no_geo.csv", index=False)

    display(ablation_df)

    def plot_ablation_heatmap(value_col, title_suffix, outname):
        vmax = np.nanmax(np.abs(ablation_df[value_col].values))
        if not np.isfinite(vmax) or vmax == 0:
            vmax = 0.01

        fig, axes = plt.subplots(1, 2, figsize=(13.5, 7.2), constrained_layout=True)

        for ax, ds in zip(axes, ["old", "new"]):
            sub = ablation_df[ablation_df["dataset"] == ds].copy()

            pivot = sub.pivot_table(
                index="removed_feature",
                columns="lead_h",
                values=value_col,
                aggfunc="mean"
            )

            pivot = pivot.reindex(INTERPRET_FEATURES)
            pivot = pivot[LEADS]

            im = ax.imshow(
                pivot.values,
                aspect="auto",
                cmap="RdBu_r",
                vmin=-vmax,
                vmax=vmax
            )

            ax.set_title(
                f"{DATASET_LABELS[ds]} AdaBoost\n{title_suffix}",
                color=DATASET_COLORS[ds],
                fontsize=13
            )

            ax.set_xlabel("Lead time")
            ax.set_ylabel("Removed predictor")

            ax.set_xticks(np.arange(len(LEADS)))
            ax.set_xticklabels([f"{L} h" for L in LEADS])

            ax.set_yticks(np.arange(len(INTERPRET_FEATURES)))
            ax.set_yticklabels(INTERPRET_FEATURES)

            for i in range(pivot.shape[0]):
                for j in range(pivot.shape[1]):
                    val = pivot.values[i, j]
                    if np.isfinite(val):
                        ax.text(
                            j,
                            i,
                            f"{val:.3f}",
                            ha="center",
                            va="center",
                            fontsize=8,
                            color="white" if abs(val) > vmax * 0.55 else "black"
                        )

        cbar = fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.86)
        cbar.set_label("F1 drop")

        fig.suptitle(
            "OLD vs NEW AdaBoost feature ablation heatmap\n"
            "No lon/lat/month | value = baseline F1 − ablated F1",
            fontsize=15
        )

        out_png = PLOT_DIR / outname
        plt.savefig(out_png, dpi=250, bbox_inches="tight")
        plt.show()
        print("Saved:", out_png)

    plot_ablation_heatmap(
        "delta_default_f1",
        "Feature ablation — default threshold",
        "adaboost_feature_ablation_default_old_vs_new_no_geo.png"
    )

    plot_ablation_heatmap(
        "delta_tuned_f1",
        "Feature ablation — tuned threshold",
        "adaboost_feature_ablation_tuned_old_vs_new_no_geo.png"
    )

# ============================================================
# SHAP HEATMAP — AdaBoost, no lon/lat/month
# ============================================================

def balanced_sample(df, n_total=300, random_state=42):
    pos = df[df["y"].astype(int) == 1].copy()
    neg = df[df["y"].astype(int) == 0].copy()

    n_pos = min(len(pos), n_total // 2)
    n_neg = min(len(neg), n_total - n_pos)

    parts = []

    if n_pos > 0:
        parts.append(pos.sample(n=n_pos, random_state=random_state))

    if n_neg > 0:
        parts.append(neg.sample(n=n_neg, random_state=random_state))

    if not parts:
        raise ValueError("No rows available for SHAP sampling.")

    out = pd.concat(parts, ignore_index=True)
    out = out.sample(frac=1, random_state=random_state).reset_index(drop=True)

    return out


def compute_adaboost_shap_importance(df, feature_cols, random_state=42):
    try:
        import shap
    except ImportError:
        raise ImportError("SHAP is not installed. Install it with: pip install shap")

    df = df.dropna(subset=["y"]).copy()

    X = df[feature_cols].copy()
    y = df["y"].astype(int).values

    pipe = make_adaboost(random_state=random_state)
    pipe.fit(X, y)

    bg = X.sample(
        n=min(SHAP_BACKGROUND_N, len(X)),
        random_state=random_state
    ).copy()

    explain_df = balanced_sample(
        df,
        n_total=min(SHAP_EXPLAIN_N, len(df)),
        random_state=random_state
    )

    X_explain = explain_df[feature_cols].copy()

    def predict_positive_class(X_array):
        X_df = pd.DataFrame(X_array, columns=feature_cols)
        return pipe.predict_proba(X_df)[:, 1]

    explainer = shap.Explainer(
        predict_positive_class,
        bg.values,
        algorithm="permutation"
    )

    shap_values = explainer(
        X_explain.values,
        max_evals=2 * len(feature_cols) + 1
    )

    mean_abs_shap = np.abs(shap_values.values).mean(axis=0)

    imp = pd.DataFrame({
        "feature": feature_cols,
        "mean_abs_shap": mean_abs_shap
    }).sort_values("mean_abs_shap", ascending=False)

    return imp


if RUN_SHAP:
    print("\n" + "=" * 90)
    print("SHAP HEATMAP — AdaBoost, no lon/lat/month")
    print("=" * 90)

    try:
        import shap
        HAS_SHAP = True
    except ImportError:
        HAS_SHAP = False
        print("[WARNING] SHAP is not installed. Skipping SHAP plots.")
        print("Install with: pip install shap")

    if HAS_SHAP:
        shap_rows = []

        for ds in ["old", "new"]:
            for L in LEADS:
                df = load_case_df(ds, L)

                print(f"\n{ds.upper()} | {L}h | rows={len(df)} | pos={int(df['y'].sum())}")

                imp = compute_adaboost_shap_importance(
                    df=df,
                    feature_cols=INTERPRET_FEATURES,
                    random_state=RANDOM_STATE + L
                )

                print("Top SHAP features:")
                display(imp.head(5))

                imp.to_csv(
                    TABLE_DIR / f"shap_importance_{ds}_adaboost_{L}h_no_geo.csv",
                    index=False
                )

                for _, r in imp.iterrows():
                    shap_rows.append({
                        "dataset": ds,
                        "lead_h": L,
                        "model": "AdaBoost",
                        "feature": r["feature"],
                        "mean_abs_shap": r["mean_abs_shap"],
                    })

        shap_df = pd.DataFrame(shap_rows)
        shap_df.to_csv(TABLE_DIR / "shap_importance_adaboost_old_vs_new_no_geo.csv", index=False)

        display(shap_df)

        vmax = shap_df["mean_abs_shap"].max()
        if not np.isfinite(vmax) or vmax == 0:
            vmax = 0.01

        fig, axes = plt.subplots(1, 2, figsize=(13.5, 7.2), constrained_layout=True)

        for ax, ds in zip(axes, ["old", "new"]):
            sub = shap_df[shap_df["dataset"] == ds].copy()

            pivot = sub.pivot_table(
                index="feature",
                columns="lead_h",
                values="mean_abs_shap",
                aggfunc="mean"
            )

            pivot = pivot.reindex(INTERPRET_FEATURES)
            pivot = pivot[LEADS]

            im = ax.imshow(
                pivot.values,
                aspect="auto",
                cmap="viridis",
                vmin=0,
                vmax=vmax
            )

            ax.set_title(
                f"{DATASET_LABELS[ds]} AdaBoost\nSHAP heatmap",
                color=DATASET_COLORS[ds],
                fontsize=13
            )

            ax.set_xlabel("Lead time")
            ax.set_ylabel("Predictor")

            ax.set_xticks(np.arange(len(LEADS)))
            ax.set_xticklabels([f"{L} h" for L in LEADS])

            ax.set_yticks(np.arange(len(INTERPRET_FEATURES)))
            ax.set_yticklabels(INTERPRET_FEATURES)

            for i in range(pivot.shape[0]):
                for j in range(pivot.shape[1]):
                    val = pivot.values[i, j]
                    if np.isfinite(val):
                        ax.text(
                            j,
                            i,
                            f"{val:.3f}",
                            ha="center",
                            va="center",
                            fontsize=8,
                            color="white" if val > vmax * 0.45 else "black"
                        )

        cbar = fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.86)
        cbar.set_label("Mean absolute SHAP value")

        fig.suptitle(
            "OLD vs NEW AdaBoost SHAP heatmap\n"
            "No lon/lat/month",
            fontsize=15
        )

        out_png = PLOT_DIR / "adaboost_shap_heatmap_old_vs_new_no_geo.png"
        plt.savefig(out_png, dpi=250, bbox_inches="tight")
        plt.show()

        print("Saved:", out_png)

# ============================================================
# DONE
# ============================================================

print("\n" + "=" * 90)
print("DONE — MASTER OLD vs NEW PLOTS GENERATED")
print("=" * 90)
print("Output directory:")
print(OUTDIR)
print("\nPlots:")
print(PLOT_DIR)
print("\nTables:")
print(TABLE_DIR)

In [ ]:
# ============================================================
# 2x2 OLD vs NEW POSITIVE CENTROID MAP
# 2009–2025 | box2deg exact-lead case tables
# With map background + n counts
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# CARTOPY
# ============================================================

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAS_CARTOPY = True
except Exception as e:
    HAS_CARTOPY = False
    print("[WARNING] Cartopy is not available. Using plain scatter plot.")
    print("Reason:", repr(e))

# ============================================================
# CONFIG
# ============================================================

LAB = Path(r"C:\Users\_s2218026\Documents\lab")

OLD_CASE_DIR = (
    LAB
    / "mcs_output"
    / "exactlead_box2deg_case_tables_old_2009_2025_UPDATED"
)

NEW_CASE_DIR = (
    LAB
    / "mcs_output_softtest"
    / "exactlead_box2deg_case_tables_new_2009_2025_UPDATED"
)

OUTDIR = (
    LAB
    / "old_vs_new_positive_centroid_maps_2009_2025"
)

OUTDIR.mkdir(parents=True, exist_ok=True)

LEADS = [6, 12, 24, 48]
LABEL_NAME = "box2deg"

START_YEAR = 2009
END_YEAR = 2025

# Western North Pacific domain
LON_MIN, LON_MAX = 100, 180
LAT_MIN, LAT_MAX = 0, 30

# ============================================================
# HELPERS
# ============================================================

def find_case_file(case_dir, dataset_name, lead):
    candidates = [
        case_dir / f"{LABEL_NAME}_case_table_{dataset_name}_{lead}h.csv",
        case_dir / f"zhang_case_table_{dataset_name}_{lead}h.csv",
        case_dir / f"zhang_case_table_{lead}h.csv",
        case_dir / f"case_table_{lead}h.csv",
    ]

    for fp in candidates:
        if fp.exists():
            return fp

    raise FileNotFoundError(
        f"No case-table file found for {dataset_name}, lead={lead}h.\n"
        + "\n".join(str(fp) for fp in candidates)
    )


def detect_lat_lon_cols(df):
    lat_candidates = [
        "centroid_lat",
        "lat",
        "latitude",
        "mcs_lat",
    ]

    lon_candidates = [
        "centroid_lon",
        "lon",
        "longitude",
        "mcs_lon",
    ]

    lat_col = next((c for c in lat_candidates if c in df.columns), None)
    lon_col = next((c for c in lon_candidates if c in df.columns), None)

    if lat_col is None or lon_col is None:
        raise ValueError(
            "Could not find centroid latitude/longitude columns.\n"
            f"Available columns:\n{df.columns.tolist()}"
        )

    return lat_col, lon_col


def load_positive_centroids(case_dir, dataset_name):
    rows = []

    for lead in LEADS:
        fp = find_case_file(case_dir, dataset_name, lead)

        df = pd.read_csv(fp, low_memory=False)

        if "time" not in df.columns:
            raise ValueError(f"{fp} has no 'time' column.")

        if "y" not in df.columns:
            raise ValueError(f"{fp} has no 'y' column.")

        df["time"] = pd.to_datetime(
            df["time"],
            errors="coerce",
            utc=True
        ).dt.tz_convert(None)

        df = df.dropna(subset=["time"]).copy()

        # Force 2009–2025 only
        df = df[
            (df["time"].dt.year >= START_YEAR)
            & (df["time"].dt.year <= END_YEAR)
        ].copy()

        # Positive only
        df = df[df["y"].astype(int) == 1].copy()

        if len(df) == 0:
            print(f"[{dataset_name.upper()}] {lead}h: 0 positives")
            continue

        lat_col, lon_col = detect_lat_lon_cols(df)

        out = pd.DataFrame({
            "dataset": dataset_name,
            "lead_h": lead,
            "time": df["time"],
            "centroid_lat": pd.to_numeric(df[lat_col], errors="coerce"),
            "centroid_lon": pd.to_numeric(df[lon_col], errors="coerce"),
        })

        if "track_id" in df.columns:
            out["track_id"] = df["track_id"].astype(str)

        out = out.dropna(
            subset=["time", "centroid_lat", "centroid_lon"]
        ).copy()

        rows.append(out)

        print(
            f"[{dataset_name.upper()}] {lead}h: "
            f"{len(out)} positives | "
            f"{out['time'].min()} to {out['time'].max()} | "
            f"{fp.name}"
        )

    if len(rows) == 0:
        raise ValueError(f"No positive centroids loaded for {dataset_name}.")

    return pd.concat(rows, ignore_index=True)


def setup_map(ax):
    if HAS_CARTOPY:
        ax.set_extent(
            [LON_MIN, LON_MAX, LAT_MIN, LAT_MAX],
            crs=ccrs.PlateCarree()
        )

        ax.add_feature(cfeature.OCEAN, facecolor="white", zorder=0)
        ax.add_feature(cfeature.LAND, facecolor="lightgray", alpha=0.45, zorder=0)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.8, zorder=1)
        ax.add_feature(cfeature.BORDERS, linewidth=0.4, alpha=0.6, zorder=1)

        gl = ax.gridlines(
            draw_labels=True,
            linewidth=0.4,
            color="gray",
            alpha=0.45,
            linestyle="--"
        )

        gl.top_labels = False
        gl.right_labels = False

    else:
        ax.set_xlim(LON_MIN, LON_MAX)
        ax.set_ylim(LAT_MIN, LAT_MAX)
        ax.grid(True, alpha=0.3)

    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")


# ============================================================
# LOAD POSITIVE CENTROIDS
# ============================================================

print("\n" + "=" * 80)
print("LOADING OLD / NEW POSITIVE CENTROIDS, 2009–2025")
print("=" * 80)

old_pos = load_positive_centroids(OLD_CASE_DIR, "old")
new_pos = load_positive_centroids(NEW_CASE_DIR, "new")

print("\nTotal positives loaded:")
print("OLD:", len(old_pos))
print("NEW:", len(new_pos))

# Save loaded positive points
old_pos.to_csv(OUTDIR / "old_positive_centroids_2009_2025.csv", index=False)
new_pos.to_csv(OUTDIR / "new_positive_centroids_2009_2025.csv", index=False)

# ============================================================
# SUMMARY TABLE
# ============================================================

summary_rows = []

for lead in LEADS:
    old_n = int((old_pos["lead_h"] == lead).sum())
    new_n = int((new_pos["lead_h"] == lead).sum())

    summary_rows.append({
        "lead_h": lead,
        "old_positive_n": old_n,
        "new_positive_n": new_n,
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(
    OUTDIR / "summary_positive_counts_old_vs_new_2009_2025.csv",
    index=False
)

print("\nPositive counts by lead:")
display(summary_df)

# ============================================================
# 2x2 MAP FIGURE
# ============================================================

if HAS_CARTOPY:
    fig, axes = plt.subplots(
        2,
        2,
        figsize=(14, 9),
        subplot_kw={"projection": ccrs.PlateCarree()},
        constrained_layout=True
    )
else:
    fig, axes = plt.subplots(
        2,
        2,
        figsize=(14, 9),
        constrained_layout=True
    )

axes = axes.flatten()

for ax, lead in zip(axes, LEADS):
    old_sub = old_pos[old_pos["lead_h"] == lead].copy()
    new_sub = new_pos[new_pos["lead_h"] == lead].copy()

    setup_map(ax)

    if HAS_CARTOPY:
        transform = ccrs.PlateCarree()

        ax.scatter(
            old_sub["centroid_lon"],
            old_sub["centroid_lat"],
            s=28,
            alpha=0.65,
            marker="o",
            label=f"OLD (n={len(old_sub)})",
            transform=transform,
            zorder=3
        )

        ax.scatter(
            new_sub["centroid_lon"],
            new_sub["centroid_lat"],
            s=38,
            alpha=0.80,
            marker="x",
            label=f"NEW (n={len(new_sub)})",
            transform=transform,
            zorder=4
        )

    else:
        ax.scatter(
            old_sub["centroid_lon"],
            old_sub["centroid_lat"],
            s=28,
            alpha=0.65,
            marker="o",
            label=f"OLD (n={len(old_sub)})"
        )

        ax.scatter(
            new_sub["centroid_lon"],
            new_sub["centroid_lat"],
            s=38,
            alpha=0.80,
            marker="x",
            label=f"NEW (n={len(new_sub)})"
        )

    ax.set_title(f"{lead} h lead")
    ax.legend(loc="lower left", fontsize=9, frameon=True)

fig.suptitle(
    "OLD vs NEW positive MCS centroid locations, 2009–2025\n"
    "box2deg exact-lead labels",
    fontsize=15
)

out_png = OUTDIR / "old_vs_new_positive_centroid_map_2x2_2009_2025.png"
plt.savefig(out_png, dpi=300, bbox_inches="tight")
plt.show()

print("\nSaved figure:")
print(out_png)

In [ ]:
# ============================================================
# CONFUSION MATRICES ONLY — OLD vs NEW, 2009–2025
#
# Models:
#   1. AdaBoost
#   2. Naive Bayes
#
# Figures:
#   For each model:
#     - default threshold, raw counts
#     - default threshold, row-normalized
#     - tuned threshold, raw counts
#     - tuned threshold, row-normalized
#
# Colors:
#   Default matplotlib confusion-matrix color style
#
# Axis labels:
#   True outcome
#   Predicted outcome
# ============================================================

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    roc_auc_score,
    average_precision_score,
)

try:
    from IPython.display import display
except Exception:
    display = print

# ============================================================
# CONFIG
# ============================================================

LAB = Path(r"C:\Users\_s2218026\Documents\lab")

LEADS = [6, 12, 24, 48]

TARGET_MODELS = [
    "AdaBoost",
    "Naive Bayes",
]

OUTDIR = LAB / "OLD_NEW_CONFUSION_MATRICES_ADABOOST_NB_DEFAULT_COLORS"
OUTDIR.mkdir(parents=True, exist_ok=True)

PLOT_DIR = OUTDIR / "plots"
TABLE_DIR = OUTDIR / "tables"

PLOT_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

OOF_CANDIDATES = {
    "old": [
        LAB / "mcs_output" / "ml_old_box2deg_2009_2025_final" / "oof_predictions_all_models.csv",
        LAB / "mcs_output" / "ml_old_box2deg_2009_2025_final" / "oof_predictions_allmodels_old.csv",
        LAB / "mcs_output" / "ml_old_box2deg_2009_2025_UPDATED" / "oof_predictions_all_models.csv",
        LAB / "mcs_output" / "ml_old_box2deg_2009_2025_UPDATED" / "oof_predictions_allmodels_old.csv",
    ],
    "new": [
        LAB / "mcs_output_softtest" / "ml_new_box2deg_2009_2025_UPDATED" / "oof_predictions_allmodels_new.csv",
        LAB / "mcs_output_softtest" / "ml_new_box2deg_2009_2025_UPDATED" / "oof_predictions_all_models.csv",
        LAB / "mcs_output_softtest" / "ml_new_box2deg_2009_2025_UPDATED" / "oof_predictions_allmodels_soft.csv",
    ],
}

DATASET_LABELS = {
    "old": "OLD",
    "new": "NEW",
}

CLASS_LABELS = ["Non-dev", "Dev"]

# Default-looking colormap
CMAP = "Blues"

# ============================================================
# HELPERS
# ============================================================

def first_existing(candidates, label):
    for fp in candidates:
        if fp.exists():
            return fp

    raise FileNotFoundError(
        f"No file found for {label}. Checked:\n"
        + "\n".join(str(fp) for fp in candidates)
    )


def clean_df(df):
    return df.loc[:, ~df.columns.astype(str).str.startswith("Unnamed")].copy()


def load_oof_predictions(dataset_name):
    fp = first_existing(
        OOF_CANDIDATES[dataset_name],
        f"{dataset_name.upper()} OOF predictions"
    )

    df = pd.read_csv(fp, low_memory=False)
    df = clean_df(df)

    required = ["lead_h", "model", "y_true", "y_score"]
    missing = [c for c in required if c not in df.columns]

    if missing:
        raise ValueError(f"Missing columns in {fp}: {missing}")

    df["lead_h"] = df["lead_h"].astype(int)
    df["y_true"] = df["y_true"].astype(int)
    df["y_score"] = df["y_score"].astype(float)

    print(f"\nLoaded {dataset_name.upper()} OOF predictions:")
    print(fp)
    print("Rows:", len(df))
    print("Models:", sorted(df["model"].dropna().unique().tolist()))

    return df


def find_model_name(oof, target_name):
    """
    Robust model-name matching.
    Handles 'Naive Bayes', 'NaiveBayes', 'NB', etc.
    """

    models = sorted(oof["model"].dropna().unique().tolist())

    if target_name in models:
        return target_name

    if target_name == "Naive Bayes":
        aliases = [
            "NaiveBayes",
            "Naive_Bayes",
            "GaussianNB",
            "Gaussian Naive Bayes",
            "NB",
        ]

        for a in aliases:
            if a in models:
                return a

        lower_map = {
            m.lower().replace(" ", "").replace("_", ""): m
            for m in models
        }

        for key, original in lower_map.items():
            if "naive" in key and "bayes" in key:
                return original

    if target_name == "AdaBoost":
        aliases = [
            "Ada Boost",
            "Ada_Boost",
            "ADABoost",
            "ADA",
        ]

        for a in aliases:
            if a in models:
                return a

        lower_map = {
            m.lower().replace(" ", "").replace("_", ""): m
            for m in models
        }

        for key, original in lower_map.items():
            if "adaboost" in key or key == "ada":
                return original

    raise ValueError(
        f"Could not find target model '{target_name}'.\n"
        f"Available models:\n{models}"
    )


def safe_metrics(y_true, y_score, threshold):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    y_pred = (y_score >= threshold).astype(int)

    out = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }

    try:
        out["roc_auc"] = roc_auc_score(y_true, y_score)
    except Exception:
        out["roc_auc"] = np.nan

    try:
        out["pr_auc"] = average_precision_score(y_true, y_score)
    except Exception:
        out["pr_auc"] = np.nan

    return out


def get_best_f1_threshold(y_true, y_score):
    rows = []

    for thr in np.round(np.arange(0.05, 0.951, 0.01), 2):
        m = safe_metrics(y_true, y_score, threshold=thr)

        rows.append({
            "threshold": float(thr),
            **m,
        })

    scan = pd.DataFrame(rows)

    scan = (
        scan
        .sort_values(["f1", "recall", "precision"], ascending=[False, False, False])
        .reset_index(drop=True)
    )

    return float(scan.loc[0, "threshold"]), scan


def compute_cm(oof, dataset_name, model_name, lead, threshold_type):
    sub = oof[
        (oof["lead_h"] == lead)
        & (oof["model"] == model_name)
    ].copy()

    if len(sub) == 0:
        raise ValueError(
            f"No OOF rows found for {DATASET_LABELS[dataset_name]}, "
            f"model={model_name}, lead={lead}h"
        )

    y_true = sub["y_true"].astype(int).values
    y_score = sub["y_score"].astype(float).values

    if threshold_type == "default":
        threshold = 0.5
        scan = None

    elif threshold_type == "tuned":
        threshold, scan = get_best_f1_threshold(y_true, y_score)

    else:
        raise ValueError("threshold_type must be 'default' or 'tuned'.")

    y_pred = (y_score >= threshold).astype(int)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    cm_norm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
        normalize="true"
    )

    tn, fp, fn, tp = cm.ravel()

    metrics = safe_metrics(y_true, y_score, threshold)

    row = {
        "dataset": dataset_name,
        "dataset_label": DATASET_LABELS[dataset_name],
        "model": model_name,
        "lead_h": lead,
        "threshold_type": threshold_type,
        "threshold": threshold,
        "n": int(cm.sum()),
        "positive": int(y_true.sum()),
        "negative": int((y_true == 0).sum()),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "tn_norm": float(cm_norm[0, 0]),
        "fp_norm": float(cm_norm[0, 1]),
        "fn_norm": float(cm_norm[1, 0]),
        "tp_norm": float(cm_norm[1, 1]),
        **metrics,
    }

    return cm, cm_norm, row, scan


def draw_cm(ax, cm_values, title, normalized=False):
    if normalized:
        ax.imshow(cm_values, cmap=CMAP, vmin=0, vmax=1)
        values_format = ".2f"
        color_threshold = 0.55
    else:
        ax.imshow(cm_values, cmap=CMAP)
        values_format = "d"
        vmax = cm_values.max() if cm_values.max() > 0 else 1
        color_threshold = vmax * 0.55

    ax.set_title(title, fontsize=10.5, fontweight="bold")

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])

    ax.set_xticklabels(CLASS_LABELS)
    ax.set_yticklabels(CLASS_LABELS)

    ax.set_xlabel("Predicted outcome")
    ax.set_ylabel("True outcome")

    for i in range(2):
        for j in range(2):
            val = cm_values[i, j]

            if normalized:
                text = f"{val:.2f}"
                text_color = "white" if val > color_threshold else "black"
            else:
                text = f"{int(val):d}"
                text_color = "white" if val > color_threshold else "black"

            ax.text(
                j,
                i,
                text,
                ha="center",
                va="center",
                fontsize=13,
                fontweight="bold",
                color=text_color
            )


def safe_filename(text):
    return (
        str(text)
        .replace(" ", "_")
        .replace("/", "_")
        .replace("-", "_")
        .replace("__", "_")
    )


def plot_model_confusion_matrices(oof_dict, model_display_name, threshold_type, normalized=False):
    """
    Creates one separate figure for one model and one threshold type.

    Figure layout:
      rows = OLD, NEW
      columns = 6h, 12h, 24h, 48h
    """

    rows = []
    scan_rows = []

    fig, axes = plt.subplots(
        2,
        4,
        figsize=(17, 7.4),
        constrained_layout=True
    )

    for row_idx, ds in enumerate(["old", "new"]):
        oof = oof_dict[ds]
        model_name = find_model_name(oof, model_display_name)

        for col_idx, lead in enumerate(LEADS):
            cm, cm_norm, row, scan = compute_cm(
                oof=oof,
                dataset_name=ds,
                model_name=model_name,
                lead=lead,
                threshold_type=threshold_type
            )

            rows.append(row)

            if scan is not None:
                scan = scan.copy()
                scan["dataset"] = ds
                scan["model"] = model_name
                scan["lead_h"] = lead
                scan_rows.append(scan)

            title = (
                f"{DATASET_LABELS[ds]} | {lead} h\n"
                f"{model_display_name}"
            )

            draw_cm(
                ax=axes[row_idx, col_idx],
                cm_values=cm_norm if normalized else cm,
                title=title,
                normalized=normalized
            )

    matrix_type = "row-normalized" if normalized else "raw counts"

    fig.suptitle(
        f"{model_display_name} confusion matrices — {matrix_type}\n"
        f"OLD vs NEW | {threshold_type} threshold",
        fontsize=15
    )

    model_safe = safe_filename(model_display_name)
    norm_tag = "normalized" if normalized else "raw"

    out_png = (
        PLOT_DIR
        / f"confusion_matrices_{model_safe}_{threshold_type}_{norm_tag}_old_vs_new_default_colors.png"
    )

    fig.savefig(out_png, dpi=260, bbox_inches="tight")
    plt.show(fig)

    print("Saved figure:")
    print(out_png)

    cm_df = pd.DataFrame(rows)

    out_csv = (
        TABLE_DIR
        / f"confusion_matrix_values_{model_safe}_{threshold_type}_{norm_tag}_old_vs_new.csv"
    )

    cm_df.to_csv(out_csv, index=False)

    print("Saved table:")
    print(out_csv)

    if scan_rows:
        scan_df = pd.concat(scan_rows, ignore_index=True)

        scan_csv = (
            TABLE_DIR
            / f"threshold_scan_{model_safe}_{threshold_type}_old_vs_new.csv"
        )

        scan_df.to_csv(scan_csv, index=False)

        print("Saved threshold scan:")
        print(scan_csv)

    return cm_df


# ============================================================
# LOAD DATA
# ============================================================

print("\n" + "=" * 90)
print("LOADING OOF PREDICTIONS")
print("=" * 90)

oof_old = load_oof_predictions("old")
oof_new = load_oof_predictions("new")

oof_dict = {
    "old": oof_old,
    "new": oof_new,
}

# ============================================================
# RUN FIGURES
# ============================================================

all_cm_tables = []

for model_name in TARGET_MODELS:
    for threshold_type in ["default", "tuned"]:
        for normalized in [False, True]:
            cm_df = plot_model_confusion_matrices(
                oof_dict=oof_dict,
                model_display_name=model_name,
                threshold_type=threshold_type,
                normalized=normalized
            )

            cm_df["figure_model"] = model_name
            cm_df["matrix_type"] = "normalized" if normalized else "raw"
            all_cm_tables.append(cm_df)

all_cm = pd.concat(all_cm_tables, ignore_index=True)

all_cm_csv = TABLE_DIR / "all_confusion_matrix_values_adaboost_naivebayes_old_vs_new.csv"
all_cm.to_csv(all_cm_csv, index=False)

print("\n" + "=" * 90)
print("DONE")
print("=" * 90)

print("Output directory:")
print(OUTDIR)

print("\nCombined CM table:")
print(all_cm_csv)

display(all_cm)

In [ ]:
# ============================================================
# 2x2 OLD vs NEW POSITIVE CENTROID MAP
# DENSITY CONTOURS VERSION (less crowded)
# 2009–2025 | box2deg exact-lead case tables
#
# OLD = blue solid contours
# NEW = red dashed contours
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# OPTIONAL IMPORTS
# ============================================================

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAS_CARTOPY = True
except Exception as e:
    HAS_CARTOPY = False
    print("[WARNING] Cartopy not available. Using plain axes.")
    print("Reason:", repr(e))

try:
    from scipy.ndimage import gaussian_filter
    HAS_SCIPY = True
except Exception as e:
    HAS_SCIPY = False
    print("[WARNING] scipy not available. Density field will not be smoothed.")
    print("Reason:", repr(e))

# ============================================================
# CONFIG
# ============================================================

LAB = Path(r"C:\Users\_s2218026\Documents\lab")

OLD_CASE_DIR = (
    LAB
    / "mcs_output"
    / "exactlead_box2deg_case_tables_old_2009_2025_UPDATED"
)

NEW_CASE_DIR = (
    LAB
    / "mcs_output_softtest"
    / "exactlead_box2deg_case_tables_new_2009_2025_UPDATED"
)

OUTDIR = LAB / "old_vs_new_positive_density_maps_2009_2025"
OUTDIR.mkdir(parents=True, exist_ok=True)

LEADS = [6, 12, 24, 48]
LABEL_NAME = "box2deg"

START_YEAR = 2009
END_YEAR = 2025

# WNP domain
LON_MIN, LON_MAX = 100, 180
LAT_MIN, LAT_MAX = 0, 30

# density grid
LON_BINS = np.linspace(LON_MIN, LON_MAX, 41)   # 2-degree bins
LAT_BINS = np.linspace(LAT_MIN, LAT_MAX, 31)   # 1-degree bins

SMOOTH_SIGMA = 1.0
N_CONTOUR_LEVELS = 6

# ============================================================
# HELPERS
# ============================================================

def find_case_file(case_dir, dataset_name, lead):
    candidates = [
        case_dir / f"{LABEL_NAME}_case_table_{dataset_name}_{lead}h.csv",
        case_dir / f"zhang_case_table_{dataset_name}_{lead}h.csv",
        case_dir / f"zhang_case_table_{lead}h.csv",
        case_dir / f"case_table_{lead}h.csv",
    ]

    for fp in candidates:
        if fp.exists():
            return fp

    raise FileNotFoundError(
        f"No case-table file found for {dataset_name}, lead={lead}h.\n"
        + "\n".join(str(fp) for fp in candidates)
    )


def detect_lat_lon_cols(df):
    lat_candidates = ["centroid_lat", "lat", "latitude", "mcs_lat"]
    lon_candidates = ["centroid_lon", "lon", "longitude", "mcs_lon"]

    lat_col = next((c for c in lat_candidates if c in df.columns), None)
    lon_col = next((c for c in lon_candidates if c in df.columns), None)

    if lat_col is None or lon_col is None:
        raise ValueError(
            "Could not find centroid lat/lon columns.\n"
            f"Available columns:\n{df.columns.tolist()}"
        )

    return lat_col, lon_col


def load_positive_centroids(case_dir, dataset_name):
    rows = []

    for lead in LEADS:
        fp = find_case_file(case_dir, dataset_name, lead)

        df = pd.read_csv(fp, low_memory=False)

        if "time" not in df.columns:
            raise ValueError(f"{fp} has no 'time' column.")
        if "y" not in df.columns:
            raise ValueError(f"{fp} has no 'y' column.")

        df["time"] = pd.to_datetime(df["time"], errors="coerce", utc=True).dt.tz_convert(None)

        df = df.dropna(subset=["time"]).copy()
        df = df[
            (df["time"].dt.year >= START_YEAR)
            & (df["time"].dt.year <= END_YEAR)
        ].copy()

        df = df[df["y"].astype(int) == 1].copy()

        if len(df) == 0:
            print(f"[{dataset_name.upper()}] {lead}h: 0 positives")
            continue

        lat_col, lon_col = detect_lat_lon_cols(df)

        out = pd.DataFrame({
            "dataset": dataset_name,
            "lead_h": lead,
            "time": df["time"],
            "centroid_lat": pd.to_numeric(df[lat_col], errors="coerce"),
            "centroid_lon": pd.to_numeric(df[lon_col], errors="coerce"),
        })

        out = out.dropna(subset=["centroid_lat", "centroid_lon"]).copy()
        rows.append(out)

        print(
            f"[{dataset_name.upper()}] {lead}h: {len(out)} positives | "
            f"{out['time'].min()} to {out['time'].max()} | {fp.name}"
        )

    if len(rows) == 0:
        raise ValueError(f"No positive centroids loaded for {dataset_name}.")

    return pd.concat(rows, ignore_index=True)


def setup_map(ax):
    if HAS_CARTOPY:
        ax.set_extent([LON_MIN, LON_MAX, LAT_MIN, LAT_MAX], crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.OCEAN, facecolor="white", zorder=0)
        ax.add_feature(cfeature.LAND, facecolor="lightgray", alpha=0.4, zorder=0)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.8, zorder=1)
        ax.add_feature(cfeature.BORDERS, linewidth=0.4, alpha=0.6, zorder=1)

        gl = ax.gridlines(
            draw_labels=True,
            linewidth=0.35,
            color="gray",
            alpha=0.5,
            linestyle="--"
        )
        gl.top_labels = False
        gl.right_labels = False
    else:
        ax.set_xlim(LON_MIN, LON_MAX)
        ax.set_ylim(LAT_MIN, LAT_MAX)
        ax.grid(True, alpha=0.3)

    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")


def density_field(df_sub):
    """
    Returns smoothed 2D histogram and grid centers.
    """
    H, xedges, yedges = np.histogram2d(
        df_sub["centroid_lon"].values,
        df_sub["centroid_lat"].values,
        bins=[LON_BINS, LAT_BINS]
    )

    # histogram2d gives shape [xbin, ybin], transpose for contour plotting
    H = H.T

    if HAS_SCIPY:
        H = gaussian_filter(H, sigma=SMOOTH_SIGMA)

    xcent = 0.5 * (xedges[:-1] + xedges[1:])
    ycent = 0.5 * (yedges[:-1] + yedges[1:])

    Xc, Yc = np.meshgrid(xcent, ycent)

    return Xc, Yc, H


def contour_levels(H, n_levels=6):
    valid = H[np.isfinite(H) & (H > 0)]

    if len(valid) == 0:
        return None

    vmin = valid.min()
    vmax = valid.max()

    if np.isclose(vmin, vmax):
        return np.array([vmin])

    return np.linspace(vmin, vmax, n_levels)


# ============================================================
# LOAD DATA
# ============================================================

print("\n" + "=" * 80)
print("LOADING POSITIVE CENTROIDS")
print("=" * 80)

old_pos = load_positive_centroids(OLD_CASE_DIR, "old")
new_pos = load_positive_centroids(NEW_CASE_DIR, "new")

old_pos.to_csv(OUTDIR / "old_positive_centroids_2009_2025.csv", index=False)
new_pos.to_csv(OUTDIR / "new_positive_centroids_2009_2025.csv", index=False)

# summary
summary_rows = []
for lead in LEADS:
    old_n = int((old_pos["lead_h"] == lead).sum())
    new_n = int((new_pos["lead_h"] == lead).sum())
    summary_rows.append({
        "lead_h": lead,
        "old_positive_n": old_n,
        "new_positive_n": new_n,
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTDIR / "summary_positive_counts_old_vs_new_2009_2025.csv", index=False)

print("\nPositive counts by lead:")
display(summary_df)

# ============================================================
# 2x2 DENSITY-CONTOUR FIGURE
# ============================================================

if HAS_CARTOPY:
    fig, axes = plt.subplots(
        2, 2,
        figsize=(14, 9),
        subplot_kw={"projection": ccrs.PlateCarree()},
        constrained_layout=True
    )
else:
    fig, axes = plt.subplots(
        2, 2,
        figsize=(14, 9),
        constrained_layout=True
    )

axes = axes.flatten()

for ax, lead in zip(axes, LEADS):
    old_sub = old_pos[old_pos["lead_h"] == lead].copy()
    new_sub = new_pos[new_pos["lead_h"] == lead].copy()

    setup_map(ax)

    # OLD density
    Xo, Yo, Ho = density_field(old_sub)
    levels_old = contour_levels(Ho, N_CONTOUR_LEVELS)

    # NEW density
    Xn, Yn, Hn = density_field(new_sub)
    levels_new = contour_levels(Hn, N_CONTOUR_LEVELS)

    if HAS_CARTOPY:
        transform = ccrs.PlateCarree()
    else:
        transform = None

    if levels_old is not None:
        cs_old = ax.contour(
            Xo, Yo, Ho,
            levels=levels_old,
            linewidths=1.4,
            linestyles="-",
            colors="blue",
            transform=transform if HAS_CARTOPY else None,
            zorder=3
        )
    else:
        cs_old = None

    if levels_new is not None:
        cs_new = ax.contour(
            Xn, Yn, Hn,
            levels=levels_new,
            linewidths=1.4,
            linestyles="--",
            colors="red",
            transform=transform if HAS_CARTOPY else None,
            zorder=4
        )
    else:
        cs_new = None

    ax.set_title(
        f"{lead} h lead\nOLD n={len(old_sub)} | NEW n={len(new_sub)}",
        fontsize=11
    )

    # manual legend
    from matplotlib.lines import Line2D
    handles = [
        Line2D([0], [0], color="blue", lw=1.8, linestyle="-", label=f"OLD (n={len(old_sub)})"),
        Line2D([0], [0], color="red", lw=1.8, linestyle="--", label=f"NEW (n={len(new_sub)})"),
    ]
    ax.legend(handles=handles, loc="lower left", fontsize=8, frameon=True)

fig.suptitle(
    "OLD vs NEW positive MCS centroid density, 2009–2025\n"
    "box2deg exact-lead labels",
    fontsize=15
)

out_png = OUTDIR / "old_vs_new_positive_centroid_density_map_2x2_2009_2025.png"
plt.savefig(out_png, dpi=300, bbox_inches="tight")
plt.show()

print("\nSaved figure:")
print(out_png)

In [ ]:
# ============================================================
# METHOD SCHEMATICS FOR CONFERENCE
# 1. ERA5 four-grid-point averaging schematic
# 2. Simple decision-tree / AdaBoost concept schematic
#
# Output:
#   - method_era5_four_point_averaging.png
#   - method_decision_tree_concept.png
#   - method_schematic_combined.png
# ============================================================

from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import FancyArrowPatch
import numpy as np

# ============================================================
# CONFIG
# ============================================================

OUT_DIR = Path(r"C:\Users\_s2218026\Documents\lab\conference_method_figures")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# FIGURE 1 — ERA5 FOUR-POINT AVERAGING SCHEMATIC
# ============================================================

fig, ax = plt.subplots(figsize=(6.5, 6.2))

# grid domain
ax.set_xlim(0, 4)
ax.set_ylim(0, 4)
ax.set_aspect("equal")

# draw grid lines
for x in range(5):
    ax.plot([x, x], [0, 4], linestyle="--", linewidth=1.2, color="gray", alpha=0.6)
for y in range(5):
    ax.plot([0, 4], [y, y], linestyle="--", linewidth=1.2, color="gray", alpha=0.6)

# selected four grid points around MCS point
grid_points = np.array([
    [1.5, 1.5],
    [1.5, 2.5],
    [2.5, 1.5],
    [2.5, 2.5],
])

ax.scatter(
    grid_points[:, 0],
    grid_points[:, 1],
    s=430,
    facecolors="white",
    edgecolors="black",
    linewidths=2,
    zorder=4,
    label="Surrounding ERA5 grid points"
)

# connect grid points
ax.plot(
    [1.5, 2.5, 2.5, 1.5, 1.5],
    [1.5, 1.5, 2.5, 2.5, 1.5],
    color="black",
    linewidth=1.6,
    zorder=3
)

# MCS / genesis point
ax.scatter(
    [2.05],
    [2.0],
    marker="*",
    s=750,
    color="black",
    zorder=5,
    label="MCS / genesis point"
)

# annotation arrows for resolution
ax.annotate(
    "",
    xy=(1, 3.55),
    xytext=(2, 3.55),
    arrowprops=dict(arrowstyle="<->", linewidth=1.6, color="black")
)
ax.text(1.5, 3.68, "0.25°", ha="center", va="bottom", fontsize=13)

ax.annotate(
    "",
    xy=(0.45, 2),
    xytext=(0.45, 3),
    arrowprops=dict(arrowstyle="<->", linewidth=1.6, color="black")
)
ax.text(0.28, 2.5, "0.25°", ha="center", va="center", rotation=90, fontsize=13)

# explanatory text
ax.text(
    2.0,
    0.35,
    "Environmental predictors are computed\n"
    "by averaging the four surrounding ERA5 grid points",
    ha="center",
    va="center",
    fontsize=12,
    bbox=dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor="black", alpha=0.9)
)

ax.set_title(
    "Extraction of large-scale environmental predictors",
    fontsize=15,
    pad=15
)

ax.set_xticks([])
ax.set_yticks([])
ax.legend(loc="upper right", fontsize=10, frameon=True)

plt.tight_layout()

out1 = OUT_DIR / "method_era5_four_point_averaging.png"
plt.savefig(out1, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", out1)

# ============================================================
# FIGURE 2 — SIMPLE DECISION TREE / ADABOOST CONCEPT
# ============================================================

fig, ax = plt.subplots(figsize=(9, 5.6))
ax.set_xlim(0, 10)
ax.set_ylim(0, 7)
ax.axis("off")

def add_node(x, y, text, w=2.2, h=0.75, fontsize=11):
    rect = patches.FancyBboxPatch(
        (x - w/2, y - h/2),
        w,
        h,
        boxstyle="round,pad=0.15",
        linewidth=1.5,
        edgecolor="black",
        facecolor="white"
    )
    ax.add_patch(rect)
    ax.text(x, y, text, ha="center", va="center", fontsize=fontsize)
    return rect

def add_leaf(x, y, text, r=0.36, fontsize=10):
    circ = patches.Circle(
        (x, y),
        radius=r,
        linewidth=1.5,
        edgecolor="black",
        facecolor="white"
    )
    ax.add_patch(circ)
    ax.text(x, y, text, ha="center", va="center", fontsize=fontsize)
    return circ

def add_arrow(x1, y1, x2, y2, label=None, label_dx=0, label_dy=0):
    arr = FancyArrowPatch(
        (x1, y1),
        (x2, y2),
        arrowstyle="->",
        mutation_scale=13,
        linewidth=1.3,
        color="black"
    )
    ax.add_patch(arr)
    if label:
        ax.text(
            (x1 + x2) / 2 + label_dx,
            (y1 + y2) / 2 + label_dy,
            label,
            fontsize=11,
            ha="center",
            va="center"
        )

# nodes
add_node(5, 6.1, "GPI < c1")
add_node(2.5, 4.55, "Vorticity < c2")
add_node(7.5, 4.55, "Wind shear < c3")
add_node(6.4, 3.0, "RH600 < c4")

# leaves
add_leaf(1.4, 3.0, "Non-dev")
add_leaf(3.6, 3.0, "Dev")
add_leaf(8.6, 3.0, "Non-dev")
add_leaf(5.45, 1.55, "Non-dev")
add_leaf(7.35, 1.55, "Dev")

# arrows
add_arrow(5, 5.72, 2.5, 4.98, "yes", label_dx=-0.15, label_dy=0.22)
add_arrow(5, 5.72, 7.5, 4.98, "no", label_dx=0.15, label_dy=0.22)

add_arrow(2.5, 4.15, 1.4, 3.38, "yes", label_dx=-0.18, label_dy=0.05)
add_arrow(2.5, 4.15, 3.6, 3.38, "no", label_dx=0.18, label_dy=0.05)

add_arrow(7.5, 4.15, 6.4, 3.38, "yes", label_dx=-0.18, label_dy=0.05)
add_arrow(7.5, 4.15, 8.6, 3.38, "no", label_dx=0.18, label_dy=0.05)

add_arrow(6.4, 2.62, 5.45, 1.92, "yes", label_dx=-0.18, label_dy=0.02)
add_arrow(6.4, 2.62, 7.35, 1.92, "no", label_dx=0.18, label_dy=0.02)

# AdaBoost explanation box
ax.text(
    5,
    0.55,
    "AdaBoost combines many weak decision trees.\n"
    "Feature importance reflects how much each predictor contributes to classification.",
    ha="center",
    va="center",
    fontsize=12,
    bbox=dict(boxstyle="round,pad=0.45", facecolor="white", edgecolor="black")
)

ax.set_title(
    "Concept of tree-based classification",
    fontsize=15,
    pad=12
)

plt.tight_layout()

out2 = OUT_DIR / "method_decision_tree_concept.png"
plt.savefig(out2, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", out2)

# ============================================================
# FIGURE 3 — COMBINED VERSION FOR ONE METHOD SLIDE
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5.8))

# ------------------------------
# Left: ERA5 averaging
# ------------------------------
ax = axes[0]
ax.set_xlim(0, 4)
ax.set_ylim(0, 4)
ax.set_aspect("equal")

for x in range(5):
    ax.plot([x, x], [0, 4], linestyle="--", linewidth=1.0, color="gray", alpha=0.6)
for y in range(5):
    ax.plot([0, 4], [y, y], linestyle="--", linewidth=1.0, color="gray", alpha=0.6)

ax.scatter(
    grid_points[:, 0],
    grid_points[:, 1],
    s=330,
    facecolors="white",
    edgecolors="black",
    linewidths=1.8,
    zorder=4
)

ax.plot(
    [1.5, 2.5, 2.5, 1.5, 1.5],
    [1.5, 1.5, 2.5, 2.5, 1.5],
    color="black",
    linewidth=1.4,
    zorder=3
)

ax.scatter(
    [2.05],
    [2.0],
    marker="*",
    s=620,
    color="black",
    zorder=5
)

ax.annotate(
    "",
    xy=(1, 3.55),
    xytext=(2, 3.55),
    arrowprops=dict(arrowstyle="<->", linewidth=1.4, color="black")
)
ax.text(1.5, 3.68, "0.25°", ha="center", va="bottom", fontsize=12)

ax.annotate(
    "",
    xy=(0.45, 2),
    xytext=(0.45, 3),
    arrowprops=dict(arrowstyle="<->", linewidth=1.4, color="black")
)
ax.text(0.28, 2.5, "0.25°", ha="center", va="center", rotation=90, fontsize=12)

ax.set_title("ERA5 predictor extraction", fontsize=14)
ax.set_xticks([])
ax.set_yticks([])

ax.text(
    2.0,
    0.35,
    "Average four surrounding grid points",
    ha="center",
    va="center",
    fontsize=11,
    bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor="black")
)

# ------------------------------
# Right: tree concept
# ------------------------------
ax = axes[1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 7)
ax.axis("off")

def add_node2(x, y, text, w=2.2, h=0.75, fontsize=10.5):
    rect = patches.FancyBboxPatch(
        (x - w/2, y - h/2),
        w,
        h,
        boxstyle="round,pad=0.13",
        linewidth=1.4,
        edgecolor="black",
        facecolor="white"
    )
    ax.add_patch(rect)
    ax.text(x, y, text, ha="center", va="center", fontsize=fontsize)

def add_leaf2(x, y, text, r=0.35, fontsize=9.5):
    circ = patches.Circle(
        (x, y),
        radius=r,
        linewidth=1.4,
        edgecolor="black",
        facecolor="white"
    )
    ax.add_patch(circ)
    ax.text(x, y, text, ha="center", va="center", fontsize=fontsize)

def add_arrow2(x1, y1, x2, y2, label=None, label_dx=0, label_dy=0):
    ax.add_patch(FancyArrowPatch(
        (x1, y1),
        (x2, y2),
        arrowstyle="->",
        mutation_scale=12,
        linewidth=1.2,
        color="black"
    ))
    if label:
        ax.text(
            (x1 + x2) / 2 + label_dx,
            (y1 + y2) / 2 + label_dy,
            label,
            fontsize=10,
            ha="center",
            va="center"
        )

add_node2(5, 6.1, "GPI < c1")
add_node2(2.5, 4.55, "Vorticity < c2")
add_node2(7.5, 4.55, "Wind shear < c3")
add_leaf2(1.4, 3.0, "Non-dev")
add_leaf2(3.6, 3.0, "Dev")
add_leaf2(6.4, 3.0, "Dev")
add_leaf2(8.6, 3.0, "Non-dev")

add_arrow2(5, 5.72, 2.5, 4.98, "yes", label_dx=-0.15, label_dy=0.2)
add_arrow2(5, 5.72, 7.5, 4.98, "no", label_dx=0.15, label_dy=0.2)
add_arrow2(2.5, 4.15, 1.4, 3.38, "yes", label_dx=-0.18, label_dy=0.05)
add_arrow2(2.5, 4.15, 3.6, 3.38, "no", label_dx=0.18, label_dy=0.05)
add_arrow2(7.5, 4.15, 6.4, 3.38, "yes", label_dx=-0.18, label_dy=0.05)
add_arrow2(7.5, 4.15, 8.6, 3.38, "no", label_dx=0.18, label_dy=0.05)

ax.text(
    5,
    0.95,
    "Tree-based models split samples using predictor thresholds",
    ha="center",
    va="center",
    fontsize=11,
    bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor="black")
)

ax.set_title("Tree-based classification concept", fontsize=14)

fig.suptitle(
    "Method schematic: environmental predictor extraction and ML classification",
    fontsize=15
)

plt.tight_layout()

out3 = OUT_DIR / "method_schematic_combined.png"
plt.savefig(out3, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", out3)

In [ ]:
# ============================================================
# COLORED COMPACT METHOD FIGURE
# ERA5 predictor extraction around MCS centroid
# Less similar to Zhang, slide-friendly
# ============================================================

from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

# ============================================================
# OUTPUT
# ============================================================

OUT_DIR = Path(r"C:\Users\_s2218026\Documents\lab\conference_method_figures")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# FIGURE
# ============================================================

fig, ax = plt.subplots(figsize=(9.5, 5.2))
ax.set_xlim(0, 10)
ax.set_ylim(0, 6)
ax.axis("off")

# ------------------------------------------------------------
# Left panel: ERA5 grid sampling schematic
# ------------------------------------------------------------

# background panel
panel = patches.FancyBboxPatch(
    (0.35, 0.45), 5.3, 5.0,
    boxstyle="round,pad=0.12",
    linewidth=1.5,
    edgecolor="#2E4057",
    facecolor="#F7FBFF"
)
ax.add_patch(panel)

ax.text(
    3.0, 5.15,
    "ERA5 predictor extraction",
    ha="center",
    va="center",
    fontsize=15,
    weight="bold",
    color="#1B263B"
)

# grid area
grid_x0, grid_y0 = 1.0, 1.25
cell = 1.35

# draw 3x3 colored grid
for i in range(3):
    for j in range(3):
        x = grid_x0 + i * cell
        y = grid_y0 + j * cell

        face = "#EAF4F4"
        edge = "#8AA6A3"

        # highlight selected surrounding area
        if i in [0, 1] and j in [0, 1]:
            face = "#D8F3DC"

        rect = patches.Rectangle(
            (x, y),
            cell,
            cell,
            linewidth=1.2,
            edgecolor=edge,
            facecolor=face
        )
        ax.add_patch(rect)

# four ERA5 grid points
points = np.array([
    [grid_x0 + cell, grid_y0 + cell],
    [grid_x0 + 2 * cell, grid_y0 + cell],
    [grid_x0 + cell, grid_y0 + 2 * cell],
    [grid_x0 + 2 * cell, grid_y0 + 2 * cell],
])

point_labels = ["P1", "P2", "P3", "P4"]

for (x, y), label in zip(points, point_labels):
    ax.scatter(
        x, y,
        s=360,
        facecolor="#457B9D",
        edgecolor="white",
        linewidth=2.0,
        zorder=5
    )
    ax.text(
        x, y,
        label,
        ha="center",
        va="center",
        fontsize=11,
        color="white",
        weight="bold",
        zorder=6
    )

# MCS centroid / sampled point
mcs_x = grid_x0 + 1.58 * cell
mcs_y = grid_y0 + 1.45 * cell

ax.scatter(
    mcs_x,
    mcs_y,
    marker="*",
    s=700,
    color="#E63946",
    edgecolor="white",
    linewidth=1.5,
    zorder=7
)

ax.text(
    mcs_x + 0.25,
    mcs_y - 0.38,
    "MCS centroid",
    ha="left",
    va="center",
    fontsize=11,
    color="#B00020",
    weight="bold"
)

# resolution annotation
ax.annotate(
    "ERA5 grid",
    xy=(grid_x0 + 0.5 * cell, grid_y0 + 3.05 * cell),
    xytext=(grid_x0 + 0.5 * cell, grid_y0 + 3.42 * cell),
    ha="center",
    fontsize=10,
    color="#2E4057",
    arrowprops=dict(arrowstyle="-", color="#2E4057", linewidth=1.2)
)

ax.text(
    grid_x0 + 2.3 * cell,
    grid_y0 + 3.22 * cell,
    "0.25° × 0.25°",
    ha="center",
    va="center",
    fontsize=11,
    color="#2E4057",
    bbox=dict(
        boxstyle="round,pad=0.25",
        facecolor="white",
        edgecolor="#8AA6A3"
    )
)

# ------------------------------------------------------------
# Arrow to right panel
# ------------------------------------------------------------

ax.annotate(
    "",
    xy=(6.55, 3.1),
    xytext=(5.8, 3.1),
    arrowprops=dict(
        arrowstyle="->",
        linewidth=2.2,
        color="#1D3557"
    )
)

# ------------------------------------------------------------
# Right panel: feature construction
# ------------------------------------------------------------

right_panel = patches.FancyBboxPatch(
    (6.65, 0.8), 3.0, 4.55,
    boxstyle="round,pad=0.15",
    linewidth=1.5,
    edgecolor="#2E4057",
    facecolor="#FFF8E8"
)
ax.add_patch(right_panel)

ax.text(
    8.15, 5.0,
    "Feature value",
    ha="center",
    va="center",
    fontsize=15,
    weight="bold",
    color="#1B263B"
)

# formula box
formula_box = patches.FancyBboxPatch(
    (7.0, 3.65), 2.3, 0.8,
    boxstyle="round,pad=0.12",
    linewidth=1.4,
    edgecolor="#E76F51",
    facecolor="#FFE5D9"
)
ax.add_patch(formula_box)

ax.text(
    8.15, 4.05,
    "mean(P1, P2, P3, P4)",
    ha="center",
    va="center",
    fontsize=12,
    color="#6D2E1F",
    weight="bold"
)

# feature list
feature_box = patches.FancyBboxPatch(
    (7.0, 1.45), 2.3, 1.75,
    boxstyle="round,pad=0.12",
    linewidth=1.3,
    edgecolor="#588157",
    facecolor="#E9F5DB"
)
ax.add_patch(feature_box)

feature_text = (
    "Environmental predictors\n"
    "• 850-hPa vorticity\n"
    "• RH600\n"
    "• 200–850 hPa shear\n"
    "• Potential intensity\n"
    "• GPI"
)

ax.text(
    8.15, 2.32,
    feature_text,
    ha="center",
    va="center",
    fontsize=10.5,
    color="#243B2A"
)

# bottom note
ax.text(
    5.0, 0.18,
    "For each MCS snapshot, large-scale ERA5 predictors are sampled around the MCS centroid and used as ML input features.",
    ha="center",
    va="bottom",
    fontsize=11,
    color="#333333"
)

# title
fig.suptitle(
    "Construction of environmental predictors for MCS-based TC genesis prediction",
    fontsize=16,
    weight="bold",
    y=0.98
)

plt.tight_layout()

out_png = OUT_DIR / "colored_compact_era5_predictor_extraction.png"
plt.savefig(out_png, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", out_png)

In [ ]:
# ============================================================
# CONFERENCE METHOD FIGURE
# Hybrid figure:
# (1) MERGIR MCS detection/tracking
# (2) ERA5 environmental predictor extraction around centroid
# (3) Final predictor groups used in ML
# ============================================================

from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyBboxPatch, Circle, FancyArrowPatch

# ------------------------------------------------------------
# output
# ------------------------------------------------------------
OUT_DIR = Path(r"C:\Users\_s2218026\Documents\lab\conference_method_figures")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PNG = OUT_DIR / "conference_method_hybrid_predictor_construction.png"

# ------------------------------------------------------------
# helper functions
# ------------------------------------------------------------
def rounded_box(ax, x, y, w, h, text="", fc="#f7f7f7", ec="#333333",
                lw=1.6, fontsize=12, weight="normal", color="#222222",
                radius=0.08, z=2, ha="center", va="center"):
    box = FancyBboxPatch(
        (x, y), w, h,
        boxstyle=f"round,pad=0.02,rounding_size={radius}",
        linewidth=lw, edgecolor=ec, facecolor=fc, zorder=z
    )
    ax.add_patch(box)
    if text:
        ax.text(x + w/2, y + h/2, text, ha=ha, va=va,
                fontsize=fontsize, weight=weight, color=color, zorder=z+1)
    return box

def arrow(ax, x1, y1, x2, y2, color="#333333", lw=2.0, ms=16, z=3):
    arr = FancyArrowPatch(
        (x1, y1), (x2, y2),
        arrowstyle="->", mutation_scale=ms,
        linewidth=lw, color=color, zorder=z
    )
    ax.add_patch(arr)
    return arr

def label_text(ax, x, y, s, fontsize=11, weight="normal",
               color="#222222", ha="center", va="center", z=5):
    ax.text(x, y, s, fontsize=fontsize, weight=weight,
            color=color, ha=ha, va=va, zorder=z)

# ------------------------------------------------------------
# figure
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(16, 8))
ax.set_xlim(0, 16)
ax.set_ylim(0, 8.5)
ax.axis("off")

# ------------------------------------------------------------
# title
# ------------------------------------------------------------
label_text(
    ax, 8, 8.15,
    "Construction of predictors for MCS-based tropical cyclone genesis prediction",
    fontsize=20, weight="bold"
)

# ============================================================
# LEFT PANEL: MERGIR / MCS properties
# ============================================================
rounded_box(ax, 0.5, 1.0, 4.2, 6.2, fc="#EAF4FF", ec="#3B5B7E", lw=1.8)

label_text(ax, 2.6, 7.0, "MERGIR MCS information", fontsize=17, weight="bold", color="#22324A")

# simple IR image placeholder
ir = Rectangle((1.0, 4.2), 3.2, 1.8, facecolor="#D9E8F6", edgecolor="#6C8AA6", linewidth=1.2)
ax.add_patch(ir)
label_text(ax, 2.6, 5.65, "Infrared brightness temperature", fontsize=11, weight="bold", color="#2C3E50")
label_text(ax, 2.6, 5.18, "Cold-cloud shield detection\nTb ≤ 233 K, area ≥ 5000 km²", fontsize=10.5, color="#2C3E50")

# MCS object / tracking
rounded_box(ax, 1.0, 2.45, 3.2, 1.2,
            text="MCS detection and tracking",
            fc="#D6ECFF", ec="#4E7CA1", fontsize=12.5, weight="bold")

# structural predictors box
rounded_box(ax, 1.0, 1.25, 3.2, 0.85,
            text="MCS structural / motion predictors\narea_km2, tb_mean, tb_min, speed_kmh, direction_deg",
            fc="#FFF1CC", ec="#C9952E", fontsize=10.3, weight="bold", color="#4D3A10")

arrow(ax, 2.6, 4.2, 2.6, 3.65, color="#4E7CA1")
arrow(ax, 2.6, 2.45, 2.6, 2.1, color="#C9952E")

# ============================================================
# CENTER PANEL: ERA5 grid extraction
# ============================================================
rounded_box(ax, 5.3, 1.0, 5.0, 6.2, fc="#F2F7F4", ec="#496B58", lw=1.8)

label_text(ax, 7.8, 7.0, "ERA5 environmental predictor extraction", fontsize=17, weight="bold", color="#254235")

# grid frame
grid_x, grid_y = 6.0, 2.1
grid_w, grid_h = 3.8, 4.5
grid_rect = Rectangle((grid_x, grid_y), grid_w, grid_h,
                      facecolor="#DDE8E2", edgecolor="#7A9A8A", linewidth=1.5)
ax.add_patch(grid_rect)

# shaded inner area
shade = Rectangle((6.0, 2.1), 2.53, 3.0, facecolor="#C7DFC8", edgecolor="none", alpha=0.75)
ax.add_patch(shade)

# draw 3x3 grid
for i in range(1, 3):
    ax.plot([grid_x + i*grid_w/3, grid_x + i*grid_w/3],
            [grid_y, grid_y + grid_h], color="#7A9A8A", lw=1.2)
for j in range(1, 3):
    ax.plot([grid_x, grid_x + grid_w],
            [grid_y + j*grid_h/3, grid_y + j*grid_h/3], color="#7A9A8A", lw=1.2)

# grid labels
label_text(ax, 6.75, 7.35, "ERA5 grid", fontsize=11, color="#3B4D63")
ax.plot([6.75, 6.75], [7.15, 6.85], color="#3B4D63", lw=1.2)
rounded_box(ax, 8.65, 6.95, 1.3, 0.35, text="0.25° × 0.25°",
            fc="#F3F6F8", ec="#9AA8B2", fontsize=9.5, color="#3B4D63", radius=0.04, lw=1.0)

# centroid star
ax.scatter([8.05], [4.55], marker="*", s=700, color="#E63946", edgecolor="white", linewidth=1.4, zorder=6)
label_text(ax, 8.35, 4.08, "MCS centroid", fontsize=12.5, weight="bold", color="#C1121F", ha="left")

# P1-P4 points
pts = {
    "P1": (7.1, 3.75),
    "P2": (8.55, 3.75),
    "P3": (7.1, 5.5),
    "P4": (8.55, 5.5),
}
for name, (px, py) in pts.items():
    c = Circle((px, py), 0.22, facecolor="#5D89B3", edgecolor="white", linewidth=1.5, zorder=7)
    ax.add_patch(c)
    label_text(ax, px, py, name, fontsize=12, weight="bold", color="white", z=8)

# feature value box
rounded_box(ax, 10.75, 2.35, 2.5, 3.95, fc="#F4EEDC", ec="#7A6A46", lw=1.6)
label_text(ax, 12.0, 5.95, "Feature value", fontsize=14, weight="bold", color="#24324A")

rounded_box(ax, 11.15, 4.98, 1.7, 0.62,
            text="mean(P1, P2, P3, P4)",
            fc="#F8E5DA", ec="#D56B47", fontsize=10.5, weight="bold", color="#6A2E1A", radius=0.08)

rounded_box(ax, 11.15, 3.0, 1.7, 1.55,
            text="Environmental predictors\n• eta850_abs\n• rh600\n• wshear_200_850\n• pi_vmax_ms\n• gpi",
            fc="#DAE9CF", ec="#5D8A58", fontsize=10.2, color="#2B4B2A", radius=0.08)

arrow(ax, 9.8, 4.45, 10.75, 4.45, color="#304E75", lw=2.2)

# ============================================================
# RIGHT PANEL: final ML input
# ============================================================
rounded_box(ax, 13.55, 1.0, 1.85, 6.2, fc="#FBF7EF", ec="#7A6650", lw=1.8)

label_text(ax, 14.48, 6.95, "Final ML input", fontsize=16, weight="bold", color="#43382C")

rounded_box(ax, 13.8, 5.75, 1.35, 0.78,
            text="Spatiotemporal\nlon, lat, month",
            fc="#E9D9F6", ec="#8D6CAB", fontsize=10.5, weight="bold", color="#4B3560")

rounded_box(ax, 13.8, 4.45, 1.35, 0.95,
            text="MCS structure / motion\narea, Tb, speed, direction",
            fc="#FFF1CC", ec="#C9952E", fontsize=9.8, weight="bold", color="#4D3A10")

rounded_box(ax, 13.8, 3.0, 1.35, 1.1,
            text="Environment\neta850_abs, rh600,\nwshear_200_850,\nPI, GPI",
            fc="#DAE9CF", ec="#5D8A58", fontsize=9.8, weight="bold", color="#2B4B2A")

rounded_box(ax, 13.8, 1.6, 1.35, 0.78,
            text="Classifier\n6 / 12 / 24 / 48 h",
            fc="#DCEAF8", ec="#5C7FA5", fontsize=10.8, weight="bold", color="#24364D")

rounded_box(ax, 13.8, 0.45, 1.35, 0.75,
            text="Output\nDeveloping / Non-developing",
            fc="#FFE1E1", ec="#C95B5B", fontsize=10.3, weight="bold", color="#6B2222")

# arrows into right panel
arrow(ax, 4.7, 1.68, 13.72, 5.95, color="#8D6CAB", lw=1.7)
arrow(ax, 4.2, 1.68, 13.72, 4.9, color="#C9952E", lw=1.7)
arrow(ax, 13.25, 3.78, 13.72, 3.55, color="#5D8A58", lw=1.7)

arrow(ax, 14.48, 5.75, 14.48, 5.4, color="#444444", lw=1.4)
arrow(ax, 14.48, 4.45, 14.48, 4.1, color="#444444", lw=1.4)
arrow(ax, 14.48, 3.0, 14.48, 2.4, color="#444444", lw=1.4)
arrow(ax, 14.48, 1.6, 14.48, 1.2, color="#444444", lw=1.4)

# ------------------------------------------------------------
# bottom note
# ------------------------------------------------------------
label_text(
    ax, 8.0, 0.12,
    "For each MCS snapshot, structural, environmental, and spatiotemporal predictors are combined and used as ML input features.",
    fontsize=13, color="#444444"
)

plt.tight_layout()
plt.savefig(OUT_PNG, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", OUT_PNG)

In [ ]:
# ============================================================
# CLEAN CONFERENCE FIGURE
# Predictors for MCS-based TC genesis prediction
# ============================================================

import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle

# ------------------------------------------------------------
# helper functions
# ------------------------------------------------------------
def add_round_box(ax, xy, w, h, title="", body="", fc="#f7f7f7", ec="#333333",
                  title_color="#1f1f1f", body_color="#333333",
                  lw=1.8, rounding=0.02, title_size=18, body_size=12,
                  title_weight="bold", z=2):
    x, y = xy
    box = FancyBboxPatch(
        (x, y), w, h,
        boxstyle=f"round,pad=0.01,rounding_size={rounding}",
        linewidth=lw, edgecolor=ec, facecolor=fc, zorder=z
    )
    ax.add_patch(box)

    if title:
        ax.text(
            x + w/2, y + h*0.86, title,
            ha="center", va="center",
            fontsize=title_size, fontweight=title_weight,
            color=title_color, zorder=z+1
        )

    if body:
        ax.text(
            x + w/2, y + h*0.50, body,
            ha="center", va="center",
            fontsize=body_size, color=body_color,
            zorder=z+1, linespacing=1.35
        )
    return box


def add_arrow(ax, start, end, color="#334e75", lw=2.3, ms=18, style='->', z=3):
    arr = FancyArrowPatch(
        start, end,
        arrowstyle=style,
        mutation_scale=ms,
        linewidth=lw,
        color=color,
        zorder=z
    )
    ax.add_patch(arr)
    return arr


def add_small_group_box(ax, xy, w, h, text, fc, ec, text_color="#1f1f1f",
                        fs=12.5, lw=1.6, rounding=0.015, z=3):
    x, y = xy
    box = FancyBboxPatch(
        (x, y), w, h,
        boxstyle=f"round,pad=0.008,rounding_size={rounding}",
        linewidth=lw, edgecolor=ec, facecolor=fc, zorder=z
    )
    ax.add_patch(box)
    ax.text(
        x + w/2, y + h/2, text,
        ha="center", va="center",
        fontsize=fs, color=text_color, zorder=z+1,
        linespacing=1.25, fontweight="bold"
    )
    return box


# ------------------------------------------------------------
# figure canvas
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(16, 9))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")

fig.patch.set_facecolor("white")
ax.set_facecolor("white")

# ------------------------------------------------------------
# title
# ------------------------------------------------------------
ax.text(
    0.5, 0.955,
    "Construction of predictors for MCS-based tropical cyclone genesis prediction",
    ha="center", va="center",
    fontsize=24, fontweight="bold", color="#222222"
)

# ------------------------------------------------------------
# PANEL 1 — MERGIR MCS information
# ------------------------------------------------------------
add_round_box(
    ax, (0.04, 0.18), 0.22, 0.66,
    title="1. MERGIR MCS information",
    body="Infrared brightness temperature\n\n"
         "Cold-cloud shield detection\n"
         "Tb ≤ 233 K, area ≥ 5000 km²",
    fc="#dfeaf6", ec="#5b84b1", title_color="#23364d",
    body_color="#31465a", title_size=20, body_size=14
)

# internal process box
add_small_group_box(
    ax, (0.075, 0.34), 0.15, 0.10,
    "MCS detection\nand tracking",
    fc="#c9def0", ec="#5b84b1", fs=16
)

# internal output box
add_small_group_box(
    ax, (0.075, 0.21), 0.15, 0.09,
    "MCS structure / motion\narea, Tb, speed, direction",
    fc="#f8ead0", ec="#c9921d", fs=13
)

add_arrow(ax, (0.15, 0.46), (0.15, 0.44))
add_arrow(ax, (0.15, 0.34), (0.15, 0.30), color="#c9921d")

# ------------------------------------------------------------
# PANEL 2 — ERA5 extraction
# ------------------------------------------------------------
add_round_box(
    ax, (0.34, 0.18), 0.30, 0.66,
    title="2. ERA5 environmental predictor extraction",
    body="",
    fc="#e6f1ea", ec="#5c7f69", title_color="#2b4b38",
    title_size=20
)

# grid inside panel 2
gx0, gy0 = 0.39, 0.31
gw, gh = 0.22, 0.43

# filled 2x2 area
for i in range(2):
    for j in range(2):
        ax.add_patch(Rectangle(
            (gx0 + i*gw/2, gy0 + j*gh/2), gw/2, gh/2,
            facecolor="#d8ead9", edgecolor="none", zorder=1
        ))

# outer grid frame
ax.add_patch(Rectangle((gx0, gy0), gw, gh, fill=False, linewidth=2, edgecolor="#6c8f7b"))
# internal grid lines
ax.plot([gx0 + gw/2, gx0 + gw/2], [gy0, gy0 + gh], color="#7ea08b", lw=1.4)
ax.plot([gx0, gx0 + gw], [gy0 + gh/2, gy0 + gh/2], color="#7ea08b", lw=1.4)

# label above grid
ax.text(gx0 + gw/2, gy0 + gh + 0.035, "ERA5 grid", ha="center", va="bottom",
        fontsize=14, color="#40556f")
ax.text(gx0 + gw/2 + 0.14, gy0 + gh + 0.01, "0.25° × 0.25°",
        ha="center", va="bottom", fontsize=13, color="#40556f",
        bbox=dict(boxstyle="round,pad=0.2", fc="#f3f3f3", ec="#b8b8b8"))

# centroid star
ax.scatter([gx0 + gw*0.62], [gy0 + gh*0.53], marker="*", s=900,
           color="#e63946", edgecolor="white", linewidth=1.3, zorder=5)
ax.text(gx0 + gw*0.65, gy0 + gh*0.44, "MCS centroid",
        color="#c1121f", fontsize=16, fontweight="bold", zorder=6)

# four points
pts = {
    "P1": (gx0 + gw*0.28, gy0 + gh*0.37),
    "P2": (gx0 + gw*0.64, gy0 + gh*0.37),
    "P3": (gx0 + gw*0.28, gy0 + gh*0.75),
    "P4": (gx0 + gw*0.64, gy0 + gh*0.75),
}
for name, (px, py) in pts.items():
    ax.scatter([px], [py], s=800, color="#5f88b0", edgecolor="white", linewidth=1.5, zorder=5)
    ax.text(px, py, name, ha="center", va="center", color="white",
            fontsize=15, fontweight="bold", zorder=6)

# feature value note
add_small_group_box(
    ax, (0.68, 0.49), 0.13, 0.10,
    "Environmental predictors\n= mean(P1, P2, P3, P4)",
    fc="#f9eee6", ec="#dd6b3f", text_color="#7a3a22", fs=14
)

add_arrow(ax, (0.61, 0.52), (0.68, 0.54))

# ------------------------------------------------------------
# PANEL 3 — final predictor groups
# ------------------------------------------------------------
add_round_box(
    ax, (0.72, 0.18), 0.17, 0.66,
    title="3. Final predictor groups",
    body="",
    fc="#f5efe3", ec="#8d765d", title_color="#3c3126",
    title_size=20
)

add_small_group_box(
    ax, (0.745, 0.66), 0.12, 0.08,
    "Spatiotemporal\nlon, lat, month",
    fc="#e9ddf8", ec="#8b6bb1", text_color="#4b3b63", fs=13
)

add_small_group_box(
    ax, (0.745, 0.51), 0.12, 0.09,
    "MCS properties\narea, Tb, speed,\ndirection",
    fc="#f8ead0", ec="#c9921d", text_color="#5d4610", fs=13
)

add_small_group_box(
    ax, (0.745, 0.34), 0.12, 0.12,
    "Environment\neta850_abs, rh600,\nwshear_200_850,\nPI, GPI",
    fc="#dce9d6", ec="#5f9956", text_color="#2d5630", fs=13
)

# arrows entering predictor groups
add_arrow(ax, (0.225, 0.255), (0.745, 0.555), color="#c9921d", lw=1.9, ms=16)
add_arrow(ax, (0.81, 0.49), (0.805, 0.40), color="#5f9956", lw=1.9, ms=16)

# ------------------------------------------------------------
# PANEL 4 — ML classifier output
# ------------------------------------------------------------
add_round_box(
    ax, (0.91, 0.22), 0.07, 0.58,
    title="4. ML output",
    body="",
    fc="#eef3fb", ec="#6b88b6", title_color="#2a4061",
    title_size=18
)

add_small_group_box(
    ax, (0.922, 0.54), 0.045, 0.09,
    "Classifier",
    fc="#d7e5f7", ec="#6b88b6", fs=12.5
)

add_small_group_box(
    ax, (0.922, 0.40), 0.045, 0.09,
    "Lead time\n6 / 12 / 24 / 48 h",
    fc="#d7e5f7", ec="#6b88b6", fs=12.5
)

add_small_group_box(
    ax, (0.922, 0.26), 0.045, 0.09,
    "Output\nDeveloping /\nNon-developing",
    fc="#f8dddd", ec="#d56a6a", text_color="#7a2d2d", fs=12.5
)

# vertical arrows in output panel
add_arrow(ax, (0.9445, 0.54), (0.9445, 0.49), color="#555555", lw=1.8, ms=14)
add_arrow(ax, (0.9445, 0.40), (0.9445, 0.35), color="#555555", lw=1.8, ms=14)

# arrow from predictor groups to output
add_arrow(ax, (0.89, 0.52), (0.922, 0.585), color="#334e75", lw=2.2, ms=17)

# ------------------------------------------------------------
# footer
# ------------------------------------------------------------
ax.text(
    0.5, 0.08,
    "For each MCS snapshot, structural, environmental, and spatiotemporal predictors are combined and used as ML input features.",
    ha="center", va="center",
    fontsize=16, color="#444444"
)

# ------------------------------------------------------------
# save + show
# ------------------------------------------------------------
out_png = "conference_predictor_construction_clean.png"
plt.savefig(out_png, dpi=250, bbox_inches="tight", facecolor="white")
plt.show()

print("Saved:", out_png)

In [ ]:
# ============================================================
# CLEAN METHOD FIGURE — CONFERENCE VERSION
# Predictor construction for MCS-based TC genesis prediction
# ============================================================

from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle, Circle
import numpy as np

# ============================================================
# OUTPUT
# ============================================================

OUT_DIR = Path(r"C:\Users\_s2218026\Documents\lab\conference_method_figures")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_PNG = OUT_DIR / "clean_predictor_construction_workflow.png"

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def box(ax, x, y, w, h, title="", body="", fc="#F7F7F7", ec="#333333",
        title_color="#222222", body_color="#333333",
        lw=1.8, title_size=15, body_size=11.5, radius=0.025):
    b = FancyBboxPatch(
        (x, y), w, h,
        boxstyle=f"round,pad=0.012,rounding_size={radius}",
        linewidth=lw,
        edgecolor=ec,
        facecolor=fc
    )
    ax.add_patch(b)

    if title:
        ax.text(
            x + w / 2,
            y + h - 0.055,
            title,
            ha="center",
            va="center",
            fontsize=title_size,
            fontweight="bold",
            color=title_color
        )

    if body:
        ax.text(
            x + w / 2,
            y + h / 2 - 0.015,
            body,
            ha="center",
            va="center",
            fontsize=body_size,
            color=body_color,
            linespacing=1.35
        )

    return b


def small_box(ax, x, y, w, h, text, fc, ec, color="#222222",
              fontsize=10.5, lw=1.4, radius=0.018):
    b = FancyBboxPatch(
        (x, y), w, h,
        boxstyle=f"round,pad=0.008,rounding_size={radius}",
        linewidth=lw,
        edgecolor=ec,
        facecolor=fc
    )
    ax.add_patch(b)
    ax.text(
        x + w / 2,
        y + h / 2,
        text,
        ha="center",
        va="center",
        fontsize=fontsize,
        fontweight="bold",
        color=color,
        linespacing=1.25
    )
    return b


def arrow(ax, x1, y1, x2, y2, color="#444444", lw=2.2):
    arr = FancyArrowPatch(
        (x1, y1), (x2, y2),
        arrowstyle="->",
        mutation_scale=18,
        linewidth=lw,
        color=color
    )
    ax.add_patch(arr)
    return arr


# ============================================================
# FIGURE CANVAS
# ============================================================

fig, ax = plt.subplots(figsize=(16, 9))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
fig.patch.set_facecolor("white")

# ============================================================
# TITLE
# ============================================================

ax.text(
    0.5, 0.94,
    "Predictor construction for MCS-based tropical cyclone genesis prediction",
    ha="center",
    va="center",
    fontsize=19,
    fontweight="bold",
    color="#222222"
)

# ============================================================
# PANEL POSITIONS
# ============================================================

y0 = 0.19
h = 0.63

x1, w1 = 0.045, 0.205
x2, w2 = 0.305, 0.235
x3, w3 = 0.595, 0.225
x4, w4 = 0.865, 0.095

# ============================================================
# PANEL 1 — MERGIR MCS
# ============================================================

box(
    ax, x1, y0, w1, h,
    title="1. MERGIR MCS",
    body="Infrared brightness temperature\n\n"
         "Cold-cloud shield detection\n"
         "Tb ≤ 233 K\n"
         "Area ≥ 5000 km²",
    fc="#E4EEF8",
    ec="#5B84B1",
    title_color="#23364D",
    body_color="#31465A",
    title_size=15,
    body_size=11.5
)

small_box(
    ax, x1 + 0.035, y0 + 0.17, w1 - 0.07, 0.085,
    "MCS detection\nand tracking",
    fc="#CFE2F3",
    ec="#5B84B1",
    color="#23364D",
    fontsize=11
)

small_box(
    ax, x1 + 0.025, y0 + 0.045, w1 - 0.05, 0.075,
    "MCS properties\narea, Tb, speed, direction",
    fc="#F8EAD0",
    ec="#C9921D",
    color="#5D4610",
    fontsize=10
)

arrow(ax, x1 + w1/2, y0 + 0.37, x1 + w1/2, y0 + 0.255, color="#5B84B1", lw=1.8)
arrow(ax, x1 + w1/2, y0 + 0.17, x1 + w1/2, y0 + 0.12, color="#C9921D", lw=1.8)

# ============================================================
# PANEL 2 — ERA5 SAMPLING
# ============================================================

box(
    ax, x2, y0, w2, h,
    title="2. ERA5 sampling",
    body="",
    fc="#E8F2EA",
    ec="#5C7F69",
    title_color="#2B4B38",
    title_size=15
)

# Grid drawing
gx = x2 + 0.055
gy = y0 + 0.20
gw = w2 - 0.11
gh = 0.34

# grid background
ax.add_patch(Rectangle(
    (gx, gy), gw, gh,
    facecolor="#DDEBE0",
    edgecolor="#6C8F7B",
    linewidth=1.5
))

# shaded lower-left area
ax.add_patch(Rectangle(
    (gx, gy), gw / 2, gh / 2,
    facecolor="#CFE8D2",
    edgecolor="none",
    alpha=0.9
))

# internal grid lines
ax.plot([gx + gw/2, gx + gw/2], [gy, gy + gh], color="#6C8F7B", lw=1.3)
ax.plot([gx, gx + gw], [gy + gh/2, gy + gh/2], color="#6C8F7B", lw=1.3)

# grid label
ax.text(
    gx + gw/2,
    gy + gh + 0.04,
    "0.25° × 0.25° ERA5 grid",
    ha="center",
    va="center",
    fontsize=10.5,
    color="#40556F",
    bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="#B0B0B0")
)

# grid points
points = {
    "P1": (gx + gw*0.28, gy + gh*0.30),
    "P2": (gx + gw*0.72, gy + gh*0.30),
    "P3": (gx + gw*0.28, gy + gh*0.74),
    "P4": (gx + gw*0.72, gy + gh*0.74),
}

for name, (px, py) in points.items():
    circ = Circle(
        (px, py),
        radius=0.018,
        facecolor="#5F88B0",
        edgecolor="white",
        linewidth=1.4
    )
    ax.add_patch(circ)
    ax.text(
        px, py,
        name,
        ha="center",
        va="center",
        fontsize=10.5,
        color="white",
        fontweight="bold"
    )

# MCS centroid
star_x = gx + gw*0.57
star_y = gy + gh*0.47
ax.scatter(
    [star_x], [star_y],
    marker="*",
    s=600,
    color="#E63946",
    edgecolor="white",
    linewidth=1.2,
    zorder=5
)

ax.text(
    star_x + 0.025,
    star_y - 0.035,
    "MCS centroid",
    ha="left",
    va="center",
    fontsize=11.5,
    fontweight="bold",
    color="#C1121F"
)

# averaging note
small_box(
    ax, x2 + 0.035, y0 + 0.055, w2 - 0.07, 0.085,
    "Environmental predictors\n= mean(P1, P2, P3, P4)",
    fc="#F9EEE6",
    ec="#DD6B3F",
    color="#7A3A22",
    fontsize=10.5
)

# ============================================================
# PANEL 3 — FEATURE VECTOR
# ============================================================

box(
    ax, x3, y0, w3, h,
    title="3. Feature vector",
    body="",
    fc="#F5EFE3",
    ec="#8D765D",
    title_color="#3C3126",
    title_size=15
)

small_box(
    ax, x3 + 0.035, y0 + 0.47, w3 - 0.07, 0.09,
    "Spatiotemporal\nlon, lat, month",
    fc="#E9DDF8",
    ec="#8B6BB1",
    color="#4B3B63",
    fontsize=10.5
)

small_box(
    ax, x3 + 0.035, y0 + 0.335, w3 - 0.07, 0.105,
    "MCS properties\narea, Tb, speed, direction",
    fc="#F8EAD0",
    ec="#C9921D",
    color="#5D4610",
    fontsize=10.2
)

small_box(
    ax, x3 + 0.035, y0 + 0.165, w3 - 0.07, 0.13,
    "Environment\neta850_abs, rh600,\nwshear, PI, GPI",
    fc="#DCE9D6",
    ec="#5F9956",
    color="#2D5630",
    fontsize=10.2
)

ax.text(
    x3 + w3/2,
    y0 + 0.07,
    "one row per MCS snapshot",
    ha="center",
    va="center",
    fontsize=10.5,
    color="#555555"
)

# ============================================================
# PANEL 4 — ML OUTPUT
# ============================================================

box(
    ax, x4, y0, w4, h,
    title="4. ML output",
    body="",
    fc="#EEF3FB",
    ec="#6B88B6",
    title_color="#2A4061",
    title_size=13.5
)

small_box(
    ax, x4 + 0.015, y0 + 0.43, w4 - 0.03, 0.08,
    "Classifier",
    fc="#D7E5F7",
    ec="#6B88B6",
    color="#2A4061",
    fontsize=9.5
)

small_box(
    ax, x4 + 0.015, y0 + 0.30, w4 - 0.03, 0.09,
    "Lead time\n6 / 12 / 24 / 48 h",
    fc="#D7E5F7",
    ec="#6B88B6",
    color="#2A4061",
    fontsize=8.7
)

small_box(
    ax, x4 + 0.015, y0 + 0.13, w4 - 0.03, 0.11,
    "Output\nDeveloping /\nNon-developing",
    fc="#F8DDDD",
    ec="#D56A6A",
    color="#7A2D2D",
    fontsize=8.7
)

arrow(ax, x4 + w4/2, y0 + 0.43, x4 + w4/2, y0 + 0.39, color="#555555", lw=1.5)
arrow(ax, x4 + w4/2, y0 + 0.30, x4 + w4/2, y0 + 0.24, color="#555555", lw=1.5)

# ============================================================
# MAIN HORIZONTAL ARROWS
# ============================================================

arrow(ax, x1 + w1 + 0.015, 0.50, x2 - 0.015, 0.50, color="#334E75", lw=2.3)
arrow(ax, x2 + w2 + 0.015, 0.50, x3 - 0.015, 0.50, color="#334E75", lw=2.3)
arrow(ax, x3 + w3 + 0.015, 0.50, x4 - 0.015, 0.50, color="#334E75", lw=2.3)

# ============================================================
# FOOTER
# ============================================================

ax.text(
    0.5,
    0.08,
    "Structural, environmental, and spatiotemporal predictors are combined for each MCS snapshot.",
    ha="center",
    va="center",
    fontsize=13.5,
    color="#444444"
)

# ============================================================
# SAVE
# ============================================================

plt.savefig(OUT_PNG, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

print("Saved:", OUT_PNG)

In [ ]:

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Optional map background
USE_CARTOPY = True
try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
except Exception:
    USE_CARTOPY = False

# ============================================================
# USER CONFIG
# ============================================================

LAB = Path(r"C:\Users\_s2218026\Documents\lab")

OLD_LABEL = "Conventional"
NEW_LABEL = "MR"

COLOR_OLD = "teal"
COLOR_NEW = "salmon"

LEADS = [6, 12, 24, 48]

# Requested confusion-matrix models
CONVENTIONAL_CM_MODELS = {
    6:  "AdaBoost",
    12: "Naive Bayes",
    24: "AdaBoost",
    48: "MLP",
}

PROPOSED_CM_MODELS = {
    6:  "AdaBoost",
    12: "Naive Bayes",
    24: "AdaBoost",
    48: "Naive Bayes",
}

# ------------------------------------------------------------
# OPTIONAL: manual paths
# If auto-search fails, set these directly.
# ------------------------------------------------------------

MANUAL_OLD_OOF = None
MANUAL_NEW_OOF = None

MANUAL_OLD_MASTER = None
MANUAL_NEW_MASTER = None

MANUAL_OLD_SHAP = None
MANUAL_NEW_SHAP = None

MANUAL_OLD_ABLATION = None
MANUAL_NEW_ABLATION = None

# Output
OUT_DIR = LAB / "PLOTS_CONVENTIONAL_PROPOSED_RELABEL"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CM_DIR = OUT_DIR / "confusion_matrices"
SHAP_DIR = OUT_DIR / "shap_heatmaps"
ABL_DIR = OUT_DIR / "feature_ablation"
MAP_DIR = OUT_DIR / "positive_maps"

for d in [CM_DIR, SHAP_DIR, ABL_DIR, MAP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ============================================================
# HELPERS
# ============================================================

def find_first_existing(candidates, label="file"):
    for fp in candidates:
        if fp is not None and Path(fp).exists():
            return Path(fp)
    raise FileNotFoundError(
        f"Could not find {label}. Checked:\n" + "\n".join(str(x) for x in candidates if x is not None)
    )

def recursive_search(base_dir, include_keywords, suffixes=(".csv", ".xlsx", ".xls")):
    base_dir = Path(base_dir)
    hits = []
    for fp in base_dir.rglob("*"):
        if not fp.is_file():
            continue
        if fp.suffix.lower() not in suffixes:
            continue
        name = fp.name.lower()
        if all(k.lower() in name for k in include_keywords):
            hits.append(fp)
    return sorted(hits)

def clean_df(df):
    return df.loc[:, ~df.columns.astype(str).str.startswith("Unnamed")].copy()

def find_model_name(available_models, target_name):
    available_models = list(available_models)

    if target_name in available_models:
        return target_name

    canon = lambda s: str(s).lower().replace(" ", "").replace("_", "").replace("-", "")

    target_c = canon(target_name)
    amap = {canon(m): m for m in available_models}

    if target_c in amap:
        return amap[target_c]

    # aliases
    aliases = {
        "naivebayes": ["nb", "gaussiannb", "naivebayes", "naivebayesclassifier", "naivebayesmodel"],
        "adaboost": ["ada", "adaboost"],
        "randomforest": ["rf", "randomforest"],
        "logisticregression": ["logistic", "lr", "logisticregression"],
        "decisiontree": ["dt", "decisiontree"],
        "knn": ["knn", "knearestneighbors"],
        "mlp": ["mlp", "neuralnet", "neuralnetwork"],
        "qda": ["qda"],
        "svm": ["svm"],
    }

    for key, vals in aliases.items():
        if target_c == key or target_c in vals:
            for c, original in amap.items():
                if c == key or c in vals:
                    return original

    raise ValueError(
        f"Could not find model '{target_name}'. Available models:\n{sorted(available_models)}"
    )

def choose_pred_column(sub):
    if "y_pred_default" in sub.columns:
        return sub["y_pred_default"].astype(int).values
    elif "y_pred" in sub.columns:
        return sub["y_pred"].astype(int).values
    elif "y_score" in sub.columns:
        return (sub["y_score"].astype(float).values >= 0.5).astype(int)
    else:
        raise ValueError("No prediction column found. Need y_pred_default, y_pred, or y_score.")

def normalize_lead_name(c):
    s = str(c).strip().lower().replace("hours", "h").replace("hour", "h").replace(" ", "")
    if s in ["6", "6h"]:
        return "6 h"
    if s in ["12", "12h"]:
        return "12 h"
    if s in ["24", "24h"]:
        return "24 h"
    if s in ["48", "48h"]:
        return "48 h"
    return str(c)

def make_matrix_from_df(df, exclude_features=("lon", "lat", "month")):
    """
    Accepts either:
      1) wide matrix-style CSV
      2) long-form CSV with predictor/lead/value columns
    Returns a DataFrame with predictors as rows and [6 h, 12 h, 24 h, 48 h] as cols.
    """
    df = clean_df(df).copy()
    df.columns = [str(c) for c in df.columns]

    # ------------------------
    # Case A: already wide
    # ------------------------
    possible_index_cols = [
        "predictor", "feature", "removed predictor", "removed_predictor",
        "variable", "name"
    ]

    lead_cols = []
    for c in df.columns:
        cc = normalize_lead_name(c)
        if cc in ["6 h", "12 h", "24 h", "48 h"]:
            lead_cols.append(c)

    index_col = None
    for c in possible_index_cols:
        if c in df.columns:
            index_col = c
            break

    if index_col is not None and len(lead_cols) >= 1:
        mat = df.set_index(index_col)[lead_cols].copy()
        mat.columns = [normalize_lead_name(c) for c in mat.columns]

    else:
        # ------------------------
        # Case B: long form
        # ------------------------
        lower_map = {c.lower(): c for c in df.columns}

        pred_candidates = [
            "predictor", "feature", "removed_predictor", "removed predictor",
            "variable", "name"
        ]
        lead_candidates = ["lead_h", "lead", "lead time", "lead_time", "hours"]
        value_candidates = [
            "value", "delta_f1", "f1_drop", "mean_abs_shap", "mean_shap",
            "shap", "importance", "score"
        ]

        pred_col = None
        lead_col = None
        val_col = None

        for c in pred_candidates:
            if c in lower_map:
                pred_col = lower_map[c]
                break

        for c in lead_candidates:
            if c in lower_map:
                lead_col = lower_map[c]
                break

        for c in value_candidates:
            if c in lower_map:
                val_col = lower_map[c]
                break

        if pred_col is None or lead_col is None or val_col is None:
            raise ValueError(
                "Could not interpret SHAP / ablation file. "
                f"Columns found: {df.columns.tolist()}"
            )

        tmp = df[[pred_col, lead_col, val_col]].copy()
        tmp[pred_col] = tmp[pred_col].astype(str)
        tmp[lead_col] = tmp[lead_col].apply(
            lambda x: f"{int(x)} h" if str(x).replace(".0", "").isdigit() else normalize_lead_name(x)
        )

        mat = tmp.pivot_table(
            index=pred_col,
            columns=lead_col,
            values=val_col,
            aggfunc="mean"
        )

    # reorder columns
    desired_cols = ["6 h", "12 h", "24 h", "48 h"]
    keep_cols = [c for c in desired_cols if c in mat.columns]
    mat = mat[keep_cols].copy()

    # remove lon/lat/month
    idx_keep = [idx for idx in mat.index if str(idx).strip().lower() not in exclude_features]
    mat = mat.loc[idx_keep].copy()

    # predictor order
    desired_order = [
        "eta850_abs", "rh600", "wshear_200_850", "pi_vmax_ms", "gpi",
        "area_km2", "tb_mean", "tb_min", "speed_kmh", "direction_deg"
    ]
    row_order = [r for r in desired_order if r in mat.index] + [r for r in mat.index if r not in desired_order]
    mat = mat.loc[row_order]

    return mat

def plot_annot_heatmap(ax, mat, title, cmap="coolwarm", force_symmetric=False):
    vals = mat.values.astype(float)

    if force_symmetric or np.nanmin(vals) < 0:
        vmax = np.nanmax(np.abs(vals))
        vmin = -vmax
    else:
        vmin = np.nanmin(vals)
        vmax = np.nanmax(vals)

    im = ax.imshow(vals, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)

    ax.set_xticks(np.arange(mat.shape[1]))
    ax.set_yticks(np.arange(mat.shape[0]))
    ax.set_xticklabels(mat.columns, fontsize=12)
    ax.set_yticklabels(mat.index, fontsize=12)
    ax.set_title(title, fontsize=15, fontweight="bold")
    ax.set_xlabel("Lead time", fontsize=13)

    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            val = mat.iloc[i, j]
            if pd.isna(val):
                txt = "NaN"
            else:
                txt = f"{val:.3f}"
            color = "white" if abs(val) > (0.55 * np.nanmax(np.abs(vals))) else "black"
            ax.text(j, i, txt, ha="center", va="center", fontsize=11, color=color, fontweight="bold")

    return im

def load_oof_old_new():
    old_candidates = [
        MANUAL_OLD_OOF,
        LAB / "mcs_output" / "ml_old_box2deg_2009_2025_final" / "oof_predictions_all_models.csv",
        LAB / "mcs_output" / "ml_old_box2deg_2009_2025_final" / "oof_predictions_allmodels_old.csv",
        LAB / "mcs_output" / "ml_old_box2deg_2009_2025_UPDATED" / "oof_predictions_all_models.csv",
        LAB / "mcs_output" / "ml_old_box2deg_2009_2025_UPDATED" / "oof_predictions_allmodels_old.csv",
    ]
    new_candidates = [
        MANUAL_NEW_OOF,
        LAB / "mcs_output_softtest" / "ml_new_box2deg_2009_2025_UPDATED" / "oof_predictions_all_models.csv",
        LAB / "mcs_output_softtest" / "ml_new_box2deg_2009_2025_UPDATED" / "oof_predictions_allmodels_new.csv",
        LAB / "mcs_output_softtest" / "ml_new_box2deg_2009_2025_UPDATED" / "oof_predictions_allmodels_soft.csv",
    ]

    old_fp = find_first_existing(old_candidates, "OLD OOF")
    new_fp = find_first_existing(new_candidates, "NEW OOF")

    old = clean_df(pd.read_csv(old_fp, low_memory=False))
    new = clean_df(pd.read_csv(new_fp, low_memory=False))

    print("Loaded OLD OOF:", old_fp)
    print("Loaded NEW OOF:", new_fp)

    return old, new

def load_master_old_new():
    old_candidates = [
        MANUAL_OLD_MASTER,
        LAB / "mcs_output" / "mcs_master_old_exactlead_ready_2009_2025_UPDATED.csv",
        LAB / "mcs_output" / "mcs_master_old_zhang_ready_2009_2025_JJASO.csv",
    ]
    new_candidates = [
        MANUAL_NEW_MASTER,
        LAB / "mcs_output_softtest" / "mcs_master_new_exactlead_ready_2009_2025_UPDATED.csv",
        LAB / "mcs_output_softtest" / "mcs_master_new_zhang_ready_2009_2025_JJASO.csv",
        LAB / "mcs_output_softtest" / "mcs_master_soft_zhang_ready_2009_2025_JJASO.csv",
    ]

    old_fp = find_first_existing(old_candidates, "OLD master")
    new_fp = find_first_existing(new_candidates, "NEW master")

    old = clean_df(pd.read_csv(old_fp, low_memory=False))
    new = clean_df(pd.read_csv(new_fp, low_memory=False))

    print("Loaded OLD master:", old_fp)
    print("Loaded NEW master:", new_fp)

    return old, new

def find_heatmap_file(dataset="old", kind="shap"):
    if kind == "shap":
        manual = MANUAL_OLD_SHAP if dataset == "old" else MANUAL_NEW_SHAP
    else:
        manual = MANUAL_OLD_ABLATION if dataset == "old" else MANUAL_NEW_ABLATION

    if manual is not None and Path(manual).exists():
        return Path(manual)

    base = LAB / "mcs_output" if dataset == "old" else LAB / "mcs_output_softtest"

    candidates = []
    if kind == "shap":
        # prefer AdaBoost SHAP if available
        patterns = [
            ["shap", "adaboost"],
            ["shap"],
        ]
    else:
        # prefer feature ablation, especially AdaBoost
        patterns = [
            ["ablation", "adaboost"],
            ["feature", "ablation", "adaboost"],
            ["ablation"],
        ]

    for keys in patterns:
        hits = recursive_search(base, keys)
        if hits:
            candidates.extend(hits)

    # Remove obvious image files since we want tables
    candidates = [fp for fp in candidates if fp.suffix.lower() in [".csv", ".xlsx", ".xls"]]

    if not candidates:
        raise FileNotFoundError(f"Could not auto-find {kind} file for dataset={dataset} under {base}")

    print(f"Auto-selected {kind} file for {dataset.upper()}: {candidates[0]}")
    return candidates[0]

# ============================================================
# 1) CONFUSION MATRICES
# ============================================================

old_oof, new_oof = load_oof_old_new()

old_models_available = sorted(old_oof["model"].dropna().unique().tolist())
new_models_available = sorted(new_oof["model"].dropna().unique().tolist())

fig, axes = plt.subplots(2, 4, figsize=(18, 9), constrained_layout=True)
cm_rows = []

for j, lead in enumerate(LEADS):
    # ---------------- OLD / Conventional ----------------
    target_model = CONVENTIONAL_CM_MODELS[lead]
    actual_model = find_model_name(old_models_available, target_model)

    sub = old_oof[(old_oof["lead_h"].astype(int) == lead) & (old_oof["model"] == actual_model)].copy()
    y_true = sub["y_true"].astype(int).values
    y_pred = choose_pred_column(sub)

    cm = pd.crosstab(
        pd.Series(y_true, name="True"),
        pd.Series(y_pred, name="Pred"),
        dropna=False
    ).reindex(index=[0,1], columns=[0,1], fill_value=0).values

    cm_norm = cm / cm.sum(axis=1, keepdims=True)

    ax = axes[0, j]
    im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(["Non-dev", "Dev"], fontsize=11)
    ax.set_yticklabels(["Non-dev", "Dev"], fontsize=11)
    ax.set_xlabel("Predicted outcome", fontsize=12)
    ax.set_ylabel("True outcome", fontsize=12)
    ax.set_title(f"{OLD_LABEL} | {lead} h\n{target_model}", fontsize=14, fontweight="bold")

    for i in range(2):
        for k in range(2):
            val = cm_norm[i, k]
            col = "white" if val > 0.55 else "black"
            ax.text(k, i, f"{val:.2f}", ha="center", va="center", fontsize=15, fontweight="bold", color=col)

    cm_rows.append({
        "dataset": OLD_LABEL,
        "lead_h": lead,
        "model": target_model,
        "tn": cm[0,0], "fp": cm[0,1], "fn": cm[1,0], "tp": cm[1,1]
    })

    # ---------------- NEW / Proposed ----------------
    target_model = PROPOSED_CM_MODELS[lead]
    actual_model = find_model_name(new_models_available, target_model)

    sub = new_oof[(new_oof["lead_h"].astype(int) == lead) & (new_oof["model"] == actual_model)].copy()
    y_true = sub["y_true"].astype(int).values
    y_pred = choose_pred_column(sub)

    cm = pd.crosstab(
        pd.Series(y_true, name="True"),
        pd.Series(y_pred, name="Pred"),
        dropna=False
    ).reindex(index=[0,1], columns=[0,1], fill_value=0).values

    cm_norm = cm / cm.sum(axis=1, keepdims=True)

    ax = axes[1, j]
    im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(["Non-dev", "Dev"], fontsize=11)
    ax.set_yticklabels(["Non-dev", "Dev"], fontsize=11)
    ax.set_xlabel("Predicted outcome", fontsize=12)
    ax.set_ylabel("True outcome", fontsize=12)
    ax.set_title(f"{NEW_LABEL} | {lead} h\n{target_model}", fontsize=14, fontweight="bold")

    for i in range(2):
        for k in range(2):
            val = cm_norm[i, k]
            col = "white" if val > 0.55 else "black"
            ax.text(k, i, f"{val:.2f}", ha="center", va="center", fontsize=15, fontweight="bold", color=col)

    cm_rows.append({
        "dataset": NEW_LABEL,
        "lead_h": lead,
        "model": target_model,
        "tn": cm[0,0], "fp": cm[0,1], "fn": cm[1,0], "tp": cm[1,1]
    })

fig.suptitle(
    "Selected confusion matrices — row-normalized\nConventional vs MR | default threshold",
    fontsize=18,
    fontweight="bold",
    y=1.02
)
cm_out = CM_DIR / "confusion_matrix_conventional_vs_MR_selected_models.png"
plt.savefig(cm_out, dpi=250, bbox_inches="tight")
plt.show()

pd.DataFrame(cm_rows).to_csv(CM_DIR / "confusion_matrix_conventional_vs_MR_selected_models_values.csv", index=False)
print("Saved:", cm_out)

# ============================================================
# 2) SHAP HEATMAPS
# ============================================================

old_shap_fp = find_heatmap_file(dataset="old", kind="shap")
new_shap_fp = find_heatmap_file(dataset="new", kind="shap")

old_shap_raw = pd.read_excel(old_shap_fp) if old_shap_fp.suffix.lower() in [".xlsx", ".xls"] else pd.read_csv(old_shap_fp)
new_shap_raw = pd.read_excel(new_shap_fp) if new_shap_fp.suffix.lower() in [".xlsx", ".xls"] else pd.read_csv(new_shap_fp)

old_shap = make_matrix_from_df(old_shap_raw, exclude_features=("lon", "lat", "month"))
new_shap = make_matrix_from_df(new_shap_raw, exclude_features=("lon", "lat", "month"))

fig, axes = plt.subplots(1, 2, figsize=(16, 8), constrained_layout=True)

# if all nonnegative, use sequential cmap; if not, use diverging
old_nonneg = np.nanmin(old_shap.values) >= 0
new_nonneg = np.nanmin(new_shap.values) >= 0
cmap_shap = "YlOrRd" if (old_nonneg and new_nonneg) else "coolwarm"
force_sym = not (old_nonneg and new_nonneg)

im1 = plot_annot_heatmap(
    axes[0], old_shap,
    title=f"{OLD_LABEL} SHAP heatmap",
    cmap=cmap_shap,
    force_symmetric=force_sym
)
im2 = plot_annot_heatmap(
    axes[1], new_shap,
    title=f"{NEW_LABEL} SHAP heatmap",
    cmap=cmap_shap,
    force_symmetric=force_sym
)

fig.colorbar(im2, ax=axes.ravel().tolist(), shrink=0.8, label="SHAP value")
shap_out = SHAP_DIR / "shap_heatmap_conventional_vs_MR.png"
plt.savefig(shap_out, dpi=250, bbox_inches="tight")
plt.show()

print("Saved:", shap_out)

# ============================================================
# 3) FEATURE ABLATION HEATMAPS
# ============================================================

old_abl_fp = find_heatmap_file(dataset="old", kind="ablation")
new_abl_fp = find_heatmap_file(dataset="new", kind="ablation")

old_abl_raw = pd.read_excel(old_abl_fp) if old_abl_fp.suffix.lower() in [".xlsx", ".xls"] else pd.read_csv(old_abl_fp)
new_abl_raw = pd.read_excel(new_abl_fp) if new_abl_fp.suffix.lower() in [".xlsx", ".xls"] else pd.read_csv(new_abl_fp)

old_abl = make_matrix_from_df(old_abl_raw, exclude_features=("lon", "lat", "month"))
new_abl = make_matrix_from_df(new_abl_raw, exclude_features=("lon", "lat", "month"))

fig, axes = plt.subplots(1, 2, figsize=(16, 8), constrained_layout=True)

im1 = plot_annot_heatmap(
    axes[0], old_abl,
    title=f"{OLD_LABEL} feature ablation",
    cmap="coolwarm",
    force_symmetric=True
)
im2 = plot_annot_heatmap(
    axes[1], new_abl,
    title=f"{NEW_LABEL} feature ablation",
    cmap="coolwarm",
    force_symmetric=True
)

fig.colorbar(im2, ax=axes.ravel().tolist(), shrink=0.8, label="F1 drop")
abl_out = ABL_DIR / "feature_ablation_conventional_vs_MR.png"
plt.savefig(abl_out, dpi=250, bbox_inches="tight")
plt.show()

print("Saved:", abl_out)

# ============================================================
# 4) POSITIVE MAPS (salmon and teal)
# ============================================================

old_master, new_master = load_master_old_new()

# make sure time is parsed
for df in [old_master, new_master]:
    if "time" in df.columns:
        df["time"] = pd.to_datetime(df["time"], errors="coerce", utc=True).dt.tz_convert(None)

# detect lon/lat columns
def get_lonlat_cols(df):
    lon_candidates = ["centroid_lon", "lon", "longitude"]
    lat_candidates = ["centroid_lat", "lat", "latitude"]

    lon_col = None
    lat_col = None

    for c in lon_candidates:
        if c in df.columns:
            lon_col = c
            break
    for c in lat_candidates:
        if c in df.columns:
            lat_col = c
            break

    if lon_col is None or lat_col is None:
        raise ValueError(f"Could not find lon/lat columns. Columns: {df.columns.tolist()}")

    return lon_col, lat_col

old_lon, old_lat = get_lonlat_cols(old_master)
new_lon, new_lat = get_lonlat_cols(new_master)

# label columns
def get_label_col(df, lead):
    candidates = [
        f"box2deg_y_{lead}h",
        f"zhang_y_{lead}h",
    ]
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError(f"Could not find positive-label column for {lead}h in columns:\n{df.columns.tolist()}")

# Filter thesis period just in case
if "time" in old_master.columns:
    old_master = old_master[
        (old_master["time"].dt.year >= 2009) &
        (old_master["time"].dt.year <= 2025)
    ].copy()
if "time" in new_master.columns:
    new_master = new_master[
        (new_master["time"].dt.year >= 2009) &
        (new_master["time"].dt.year <= 2025)
    ].copy()

# Plot
if USE_CARTOPY:
    fig, axes = plt.subplots(
        2, 2, figsize=(16, 10),
        subplot_kw={"projection": ccrs.PlateCarree()},
        constrained_layout=True
    )
else:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10), constrained_layout=True)

axes = axes.flatten()

for ax, lead in zip(axes, LEADS):
    old_lab_col = get_label_col(old_master, lead)
    new_lab_col = get_label_col(new_master, lead)

    old_pos = old_master[old_master[old_lab_col].astype(int) == 1].copy()
    new_pos = new_master[new_master[new_lab_col].astype(int) == 1].copy()

    if USE_CARTOPY:
        ax.set_extent([100, 180, 0, 30], crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.LAND, facecolor="whitesmoke", edgecolor="none", zorder=0)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.7, zorder=1)
        ax.add_feature(cfeature.BORDERS, linewidth=0.4, zorder=1)
        gl = ax.gridlines(draw_labels=True, linewidth=0.3, color="gray", alpha=0.5, linestyle="--")
        gl.top_labels = False
        gl.right_labels = False
    else:
        ax.set_xlim(100, 180)
        ax.set_ylim(0, 30)
        ax.grid(True, linestyle="--", linewidth=0.4, alpha=0.5)

    ax.scatter(
        old_pos[old_lon], old_pos[old_lat],
        s=36, c=COLOR_OLD, marker="o", alpha=0.75,
        label=f"{OLD_LABEL} (n={len(old_pos)})", zorder=3
    )
    ax.scatter(
        new_pos[new_lon], new_pos[new_lat],
        s=42, c=COLOR_NEW, marker="x", alpha=0.90,
        label=f"{NEW_LABEL} (n={len(new_pos)})", zorder=4
    )

    ax.set_title(f"{lead} h lead", fontsize=15, fontweight="bold")
    ax.legend(loc="lower left", fontsize=10, frameon=True)

fig.suptitle(
    f"{OLD_LABEL} vs {NEW_LABEL} positive MCS centroid locations, 2009–2025\nbox2deg exact-lead labels",
    fontsize=18,
    fontweight="bold"
)

map_out = MAP_DIR / "positive_centroid_maps_conventional_vs_MR_teal_salmon.png"
plt.savefig(map_out, dpi=250, bbox_inches="tight")
plt.show()

print("Saved:", map_out)

print("\nDONE.")
print("All outputs saved in:")
print(OUT_DIR)

In [ ]:
# ============================================================
# FINAL PLOTS — CONVENTIONAL vs PROPOSED
#
# Fixes:
#   1. No overlapping figure titles
#   2. OLD -> Conventional
#   3. NEW -> Proposed
#   4. Positive maps use teal / salmon
#   5. SHAP / feature ablation only loads valid summary files
#
# Outputs:
#   1. Selected row-normalized confusion matrices
#   2. SHAP heatmap: Conventional vs Proposed
#   3. Feature ablation heatmap: Conventional vs Proposed
#   4. Positive centroid maps: Conventional vs Proposed
# ============================================================

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAS_CARTOPY = True
except Exception:
    HAS_CARTOPY = False

try:
    from IPython.display import display
except Exception:
    display = print

# ============================================================
# CONFIG
# ============================================================

LAB = Path(r"C:\Users\_s2218026\Documents\lab")

OLD_NAME = "Conventional"
NEW_NAME = "MR"

COLOR_OLD = "#2A9D8F"   # teal
COLOR_NEW = "#E76F51"   # salmon

LEADS = [6, 12, 24, 48]
LABEL_NAME = "box2deg"

START_YEAR = 2009
END_YEAR = 2025

# WNP map extent
LON_MIN, LON_MAX = 100, 180
LAT_MIN, LAT_MAX = 0, 30

# Selected models for confusion matrix
CONVENTIONAL_CM_MODELS = {
    6:  "AdaBoost",
    12: "Naive Bayes",
    24: "AdaBoost",
    48: "MLP",
}

PROPOSED_CM_MODELS = {
    6:  "AdaBoost",
    12: "Naive Bayes",
    24: "AdaBoost",
    48: "Naive Bayes",
}

# Input paths
OLD_CASE_DIR = LAB / "mcs_output" / "exactlead_box2deg_case_tables_old_2009_2025_UPDATED"
NEW_CASE_DIR = LAB / "mcs_output_softtest" / "exactlead_box2deg_case_tables_new_2009_2025_UPDATED"

OLD_OOF_CANDIDATES = [
    LAB / "mcs_output" / "ml_old_box2deg_2009_2025_final" / "oof_predictions_all_models.csv",
    LAB / "mcs_output" / "ml_old_box2deg_2009_2025_final" / "oof_predictions_allmodels_old.csv",
    LAB / "mcs_output" / "ml_old_box2deg_2009_2025_UPDATED" / "oof_predictions_all_models.csv",
    LAB / "mcs_output" / "ml_old_box2deg_2009_2025_UPDATED" / "oof_predictions_allmodels_old.csv",
]

NEW_OOF_CANDIDATES = [
    LAB / "mcs_output_softtest" / "ml_new_box2deg_2009_2025_UPDATED" / "oof_predictions_allmodels_new.csv",
    LAB / "mcs_output_softtest" / "ml_new_box2deg_2009_2025_UPDATED" / "oof_predictions_all_models.csv",
    LAB / "mcs_output_softtest" / "ml_new_box2deg_2009_2025_UPDATED" / "oof_predictions_allmodels_soft.csv",
]

# Output
OUT_DIR = LAB / "FINAL_CONVENTIONAL_PROPOSED_PLOTS"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CM_DIR = OUT_DIR / "confusion_matrices"
SHAP_DIR = OUT_DIR / "shap_heatmap"
ABL_DIR = OUT_DIR / "feature_ablation"
MAP_DIR = OUT_DIR / "positive_maps"
TABLE_DIR = OUT_DIR / "tables"

for d in [CM_DIR, SHAP_DIR, ABL_DIR, MAP_DIR, TABLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def clean_df(df):
    return df.loc[:, ~df.columns.astype(str).str.startswith("Unnamed")].copy()


def first_existing(paths, label):
    for fp in paths:
        if fp.exists():
            return fp
    raise FileNotFoundError(
        f"No file found for {label}. Checked:\n"
        + "\n".join(str(fp) for fp in paths)
    )


def find_model_name(models_available, target_name):
    models_available = list(models_available)

    if target_name in models_available:
        return target_name

    def canon(s):
        return str(s).lower().replace(" ", "").replace("_", "").replace("-", "")

    target = canon(target_name)
    cmap = {canon(m): m for m in models_available}

    if target in cmap:
        return cmap[target]

    aliases = {
        "naivebayes": ["naivebayes", "nb", "gaussiannb", "gaussiannaivebayes"],
        "adaboost": ["adaboost", "ada", "adaboostclassifier"],
        "mlp": ["mlp", "multilayerperceptron", "neuralnetwork"],
    }

    for key, alias_list in aliases.items():
        if target == key or target in alias_list:
            for c, original in cmap.items():
                if c == key or c in alias_list:
                    return original

    raise ValueError(
        f"Could not find model '{target_name}'. Available models:\n"
        f"{sorted(models_available)}"
    )


def choose_default_prediction(sub):
    if "y_pred_default" in sub.columns:
        return sub["y_pred_default"].astype(int).values
    if "y_pred" in sub.columns:
        return sub["y_pred"].astype(int).values
    if "y_score" in sub.columns:
        return (sub["y_score"].astype(float).values >= 0.5).astype(int)
    if "y_prob" in sub.columns:
        return (sub["y_prob"].astype(float).values >= 0.5).astype(int)

    raise ValueError(
        "No prediction column found. Need y_pred_default, y_pred, y_score, or y_prob."
    )


def row_normalize_cm(cm):
    cm = np.asarray(cm, dtype=float)
    row_sum = cm.sum(axis=1, keepdims=True)
    return np.divide(cm, row_sum, out=np.zeros_like(cm), where=row_sum != 0)


def draw_normalized_cm(ax, cm_norm, title):
    ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])

    ax.set_xticklabels(["Non-dev", "Dev"], fontsize=11)
    ax.set_yticklabels(["Non-dev", "Dev"], fontsize=11)

    ax.set_xlabel("Predicted outcome", fontsize=11)
    ax.set_ylabel("True outcome", fontsize=11)

    ax.set_title(title, fontsize=13, fontweight="bold", pad=8)

    for i in range(2):
        for j in range(2):
            val = cm_norm[i, j]
            ax.text(
                j,
                i,
                f"{val:.2f}",
                ha="center",
                va="center",
                fontsize=15,
                fontweight="bold",
                color="white" if val > 0.55 else "black"
            )


def find_case_file(case_dir, dataset_name, lead):
    candidates = [
        case_dir / f"{LABEL_NAME}_case_table_{dataset_name}_{lead}h.csv",
        case_dir / f"zhang_case_table_{dataset_name}_{lead}h.csv",
        case_dir / f"zhang_case_table_{lead}h.csv",
        case_dir / f"case_table_{lead}h.csv",
    ]

    for fp in candidates:
        if fp.exists():
            return fp

    raise FileNotFoundError(
        f"No case-table file found for {dataset_name}, lead={lead}h.\n"
        + "\n".join(str(fp) for fp in candidates)
    )


def detect_lon_lat_cols(df):
    lon_candidates = ["centroid_lon", "lon", "longitude", "mcs_lon"]
    lat_candidates = ["centroid_lat", "lat", "latitude", "mcs_lat"]

    lon_col = next((c for c in lon_candidates if c in df.columns), None)
    lat_col = next((c for c in lat_candidates if c in df.columns), None)

    if lon_col is None or lat_col is None:
        raise ValueError(f"Could not find lon/lat columns. Available:\n{df.columns.tolist()}")

    return lon_col, lat_col


def load_positive_case_points(case_dir, dataset_name):
    rows = []

    for lead in LEADS:
        fp = find_case_file(case_dir, dataset_name, lead)
        df = clean_df(pd.read_csv(fp, low_memory=False))

        if "time" in df.columns:
            df["time"] = pd.to_datetime(df["time"], errors="coerce", utc=True).dt.tz_convert(None)
            df = df.dropna(subset=["time"]).copy()
            df = df[
                (df["time"].dt.year >= START_YEAR)
                & (df["time"].dt.year <= END_YEAR)
            ].copy()

        if "y" not in df.columns:
            raise ValueError(f"{fp} has no y column.")

        df = df[df["y"].astype(int) == 1].copy()

        lon_col, lat_col = detect_lon_lat_cols(df)

        out = pd.DataFrame({
            "dataset": dataset_name,
            "lead_h": lead,
            "lon": pd.to_numeric(df[lon_col], errors="coerce"),
            "lat": pd.to_numeric(df[lat_col], errors="coerce"),
        }).dropna(subset=["lon", "lat"])

        rows.append(out)

        print(f"{dataset_name.upper()} {lead}h positives: {len(out)} | {fp.name}")

    return pd.concat(rows, ignore_index=True)


def setup_map(ax):
    if HAS_CARTOPY:
        ax.set_extent([LON_MIN, LON_MAX, LAT_MIN, LAT_MAX], crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.OCEAN, facecolor="white", zorder=0)
        ax.add_feature(cfeature.LAND, facecolor="whitesmoke", zorder=0)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.7, zorder=1)
        ax.add_feature(cfeature.BORDERS, linewidth=0.4, alpha=0.7, zorder=1)

        gl = ax.gridlines(
            draw_labels=True,
            linewidth=0.35,
            color="gray",
            alpha=0.45,
            linestyle="--"
        )
        gl.top_labels = False
        gl.right_labels = False
    else:
        ax.set_xlim(LON_MIN, LON_MAX)
        ax.set_ylim(LAT_MIN, LAT_MAX)
        ax.grid(True, linestyle="--", alpha=0.4)

    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")


def recursive_csv_search(base_dir, keywords):
    base_dir = Path(base_dir)
    hits = []

    for fp in base_dir.rglob("*.csv"):
        name = fp.name.lower()
        if all(k.lower() in name for k in keywords):
            hits.append(fp)

    return sorted(hits)


def normalize_lead(x):
    try:
        return f"{int(float(x))} h"
    except Exception:
        s = str(x).strip().lower().replace("hours", "h").replace("hour", "h").replace(" ", "")
        if s in ["6", "6h"]:
            return "6 h"
        if s in ["12", "12h"]:
            return "12 h"
        if s in ["24", "24h"]:
            return "24 h"
        if s in ["48", "48h"]:
            return "48 h"
        return str(x)


def matrix_from_valid_summary(df, dataset_key, kind):
    """
    Converts a valid SHAP / ablation summary table into heatmap matrix.

    Accepted SHAP format:
      dataset, lead_h, feature, mean_abs_shap

    Accepted ablation format:
      dataset, lead_h, removed_feature, delta_default_f1 / delta_tuned_f1 / f1_drop

    Also accepts wide matrix format with predictor rows and 6h/12h/24h/48h columns.
    """

    df = clean_df(df).copy()
    df.columns = [str(c) for c in df.columns]

    # Filter dataset if possible
    if "dataset" in df.columns:
        tmp = df["dataset"].astype(str).str.lower()
        df = df[tmp.isin([dataset_key.lower(), OLD_NAME.lower(), NEW_NAME.lower()])].copy()

        if dataset_key == "old":
            df = df[df["dataset"].astype(str).str.lower().isin(["old", OLD_NAME.lower(), "conventional"])].copy()
        else:
            df = df[df["dataset"].astype(str).str.lower().isin(["new", "soft", NEW_NAME.lower(), "proposed"])].copy()

    lower = {c.lower(): c for c in df.columns}

    # Long-format candidates
    pred_candidates = ["feature", "predictor", "removed_feature", "removed predictor", "variable", "name"]
    lead_candidates = ["lead_h", "lead", "lead time", "lead_time"]

    if kind == "shap":
        value_candidates = ["mean_abs_shap", "mean_shap", "shap_value", "shap", "importance", "value"]
    else:
        value_candidates = ["delta_default_f1", "f1_drop", "delta_f1", "drop_f1", "delta_tuned_f1", "value"]

    pred_col = next((lower[c] for c in pred_candidates if c in lower), None)
    lead_col = next((lower[c] for c in lead_candidates if c in lower), None)
    value_col = next((lower[c] for c in value_candidates if c in lower), None)

    if pred_col is not None and lead_col is not None and value_col is not None:
        use = df[[pred_col, lead_col, value_col]].copy()
        use[pred_col] = use[pred_col].astype(str)
        use[lead_col] = use[lead_col].apply(normalize_lead)
        use[value_col] = pd.to_numeric(use[value_col], errors="coerce")

        mat = use.pivot_table(
            index=pred_col,
            columns=lead_col,
            values=value_col,
            aggfunc="mean"
        )

    else:
        # Wide format
        possible_lead_cols = []
        for c in df.columns:
            nc = normalize_lead(c)
            if nc in ["6 h", "12 h", "24 h", "48 h"]:
                possible_lead_cols.append(c)

        index_col = next((lower[c] for c in pred_candidates if c in lower), None)

        if index_col is None or len(possible_lead_cols) == 0:
            raise ValueError(
                "Invalid SHAP / ablation summary table. "
                f"Columns found: {df.columns.tolist()}"
            )

        mat = df.set_index(index_col)[possible_lead_cols].copy()
        mat.columns = [normalize_lead(c) for c in mat.columns]

    # Remove spatiotemporal predictors
    remove = {"lon", "lat", "month", "mon"}
    mat = mat.loc[[r for r in mat.index if str(r).lower() not in remove]].copy()

    # Order rows
    order = [
        "eta850_abs",
        "rh600",
        "wshear_200_850",
        "pi_vmax_ms",
        "gpi",
        "area_km2",
        "tb_mean",
        "tb_min",
        "speed_kmh",
        "direction_deg",
    ]

    row_order = [r for r in order if r in mat.index] + [r for r in mat.index if r not in order]
    mat = mat.loc[row_order]

    # Order columns
    col_order = [c for c in ["6 h", "12 h", "24 h", "48 h"] if c in mat.columns]
    mat = mat[col_order]

    if mat.empty:
        raise ValueError("Parsed matrix is empty.")

    return mat


def find_valid_heatmap_source(kind):
    """
    Prioritizes summary files generated by previous master plotting cells.
    Avoids raw prediction files, which caused your previous error.
    """

    if kind == "shap":
        candidate_paths = [
            LAB / "OLD_NEW_MASTER_PLOTS_2009_2025_TEAL_SALMON" / "tables" / "shap_importance_adaboost_old_vs_new_no_geo.csv",
            LAB / "shap_adaboost_old_vs_new_all_leads_no_geo" / "shap_importance_adaboost_old_vs_new_all_leads_no_geo.csv",
        ]

        search_dirs = [
            LAB / "OLD_NEW_MASTER_PLOTS_2009_2025_TEAL_SALMON",
            LAB / "shap_adaboost_old_vs_new_all_leads_no_geo",
            LAB / "mcs_output",
            LAB / "mcs_output_softtest",
        ]

        keyword_groups = [
            ["shap", "importance"],
            ["shap", "adaboost"],
            ["shap"],
        ]

    else:
        candidate_paths = [
            LAB / "OLD_NEW_MASTER_PLOTS_2009_2025_TEAL_SALMON" / "tables" / "adaboost_feature_ablation_old_vs_new_no_geo.csv",
            LAB / "feature_ablation_adaboost_old_vs_new_all_leads_no_geo" / "adaboost_feature_ablation_old_vs_new_no_geo.csv",
        ]

        search_dirs = [
            LAB / "OLD_NEW_MASTER_PLOTS_2009_2025_TEAL_SALMON",
            LAB / "feature_ablation_adaboost_old_vs_new_all_leads_no_geo",
            LAB / "mcs_output",
            LAB / "mcs_output_softtest",
        ]

        keyword_groups = [
            ["ablation", "adaboost"],
            ["feature", "ablation"],
            ["ablation"],
        ]

    candidates = [fp for fp in candidate_paths if fp.exists()]

    for sdir in search_dirs:
        if not Path(sdir).exists():
            continue
        for keys in keyword_groups:
            candidates.extend(recursive_csv_search(sdir, keys))

    # Deduplicate
    seen = set()
    candidates = [x for x in candidates if not (str(x) in seen or seen.add(str(x)))]

    valid = []

    for fp in candidates:
        try:
            df = pd.read_csv(fp, low_memory=False)
            _old = matrix_from_valid_summary(df, "old", kind)
            _new = matrix_from_valid_summary(df, "new", kind)
            valid.append(fp)
        except Exception:
            continue

    if not valid:
        raise FileNotFoundError(
            f"No valid {kind} summary table found.\n"
            f"Important: raw prediction tables are not valid {kind} heatmap inputs.\n"
            f"Checked {len(candidates)} candidate files."
        )

    print(f"Using {kind} summary table:")
    print(valid[0])

    return valid[0]


def plot_heatmap_pair(old_mat, new_mat, title, colorbar_label, out_fp, cmap="coolwarm", symmetric=True):
    if symmetric:
        vmax = max(
            np.nanmax(np.abs(old_mat.values.astype(float))),
            np.nanmax(np.abs(new_mat.values.astype(float)))
        )
        vmin = -vmax
    else:
        vmin = 0
        vmax = max(
            np.nanmax(old_mat.values.astype(float)),
            np.nanmax(new_mat.values.astype(float))
        )

    fig, axes = plt.subplots(1, 2, figsize=(15.5, 7.2))
    plt.subplots_adjust(top=0.83, wspace=0.35)

    for ax, mat, label in [
        (axes[0], old_mat, OLD_NAME),
        (axes[1], new_mat, NEW_NAME),
    ]:
        im = ax.imshow(mat.values.astype(float), aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)

        ax.set_title(label, fontsize=16, fontweight="bold", pad=10)
        ax.set_xlabel("Lead time", fontsize=13)
        ax.set_ylabel("Predictor", fontsize=13)

        ax.set_xticks(np.arange(mat.shape[1]))
        ax.set_xticklabels(mat.columns, fontsize=12)

        ax.set_yticks(np.arange(mat.shape[0]))
        ax.set_yticklabels(mat.index, fontsize=12)

        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                val = float(mat.iloc[i, j])
                ax.text(
                    j,
                    i,
                    f"{val:.3f}",
                    ha="center",
                    va="center",
                    fontsize=9,
                    color="white" if abs(val) > 0.55 * max(abs(vmin), abs(vmax)) else "black"
                )

    cbar = fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.85)
    cbar.set_label(colorbar_label, fontsize=12)

    fig.suptitle(title, fontsize=18, fontweight="bold", y=0.96)

    plt.savefig(out_fp, dpi=250, bbox_inches="tight")
    plt.show()

    print("Saved:", out_fp)


# ============================================================
# 1. CONFUSION MATRICES
# ============================================================

old_oof_fp = first_existing(OLD_OOF_CANDIDATES, "Conventional OOF")
new_oof_fp = first_existing(NEW_OOF_CANDIDATES, "Proposed OOF")

old_oof = clean_df(pd.read_csv(old_oof_fp, low_memory=False))
new_oof = clean_df(pd.read_csv(new_oof_fp, low_memory=False))

old_oof["lead_h"] = old_oof["lead_h"].astype(int)
new_oof["lead_h"] = new_oof["lead_h"].astype(int)

old_models = sorted(old_oof["model"].dropna().unique().tolist())
new_models = sorted(new_oof["model"].dropna().unique().tolist())

print("Loaded Conventional OOF:", old_oof_fp)
print("Loaded Proposed OOF:", new_oof_fp)

fig, axes = plt.subplots(2, 4, figsize=(18, 8.7))
plt.subplots_adjust(
    left=0.055,
    right=0.985,
    bottom=0.075,
    top=0.79,
    wspace=0.32,
    hspace=0.68
)

cm_rows = []

for col, lead in enumerate(LEADS):
    # ---------------- Conventional ----------------
    model_display = CONVENTIONAL_CM_MODELS[lead]
    model_actual = find_model_name(old_models, model_display)

    sub = old_oof[
        (old_oof["lead_h"] == lead)
        & (old_oof["model"] == model_actual)
    ].copy()

    y_true = sub["y_true"].astype(int).values
    y_pred = choose_default_prediction(sub)

    cm = pd.crosstab(
        pd.Series(y_true, name="True"),
        pd.Series(y_pred, name="Pred"),
        dropna=False
    ).reindex(index=[0, 1], columns=[0, 1], fill_value=0).values

    cm_norm = row_normalize_cm(cm)

    draw_normalized_cm(
        axes[0, col],
        cm_norm,
        f"{OLD_NAME} | {lead} h\n{model_display}"
    )

    cm_rows.append({
        "dataset": OLD_NAME,
        "lead_h": lead,
        "model": model_display,
        "tn": int(cm[0, 0]),
        "fp": int(cm[0, 1]),
        "fn": int(cm[1, 0]),
        "tp": int(cm[1, 1]),
        "tn_norm": cm_norm[0, 0],
        "fp_norm": cm_norm[0, 1],
        "fn_norm": cm_norm[1, 0],
        "tp_norm": cm_norm[1, 1],
    })

    # ---------------- Proposed ----------------
    model_display = PROPOSED_CM_MODELS[lead]
    model_actual = find_model_name(new_models, model_display)

    sub = new_oof[
        (new_oof["lead_h"] == lead)
        & (new_oof["model"] == model_actual)
    ].copy()

    y_true = sub["y_true"].astype(int).values
    y_pred = choose_default_prediction(sub)

    cm = pd.crosstab(
        pd.Series(y_true, name="True"),
        pd.Series(y_pred, name="Pred"),
        dropna=False
    ).reindex(index=[0, 1], columns=[0, 1], fill_value=0).values

    cm_norm = row_normalize_cm(cm)

    draw_normalized_cm(
        axes[1, col],
        cm_norm,
        f"{NEW_NAME} | {lead} h\n{model_display}"
    )

    cm_rows.append({
        "dataset": NEW_NAME,
        "lead_h": lead,
        "model": model_display,
        "tn": int(cm[0, 0]),
        "fp": int(cm[0, 1]),
        "fn": int(cm[1, 0]),
        "tp": int(cm[1, 1]),
        "tn_norm": cm_norm[0, 0],
        "fp_norm": cm_norm[0, 1],
        "fn_norm": cm_norm[1, 0],
        "tp_norm": cm_norm[1, 1],
    })

fig.suptitle(
    "Selected confusion matrices — row-normalized\n"
    "Conventional vs Proposed | default threshold",
    fontsize=18,
    fontweight="bold",
    y=0.965
)

cm_out = CM_DIR / "selected_confusion_matrices_conventional_vs_proposed_default.png"
plt.savefig(cm_out, dpi=250, bbox_inches="tight")
plt.show()

cm_table = pd.DataFrame(cm_rows)
cm_table.to_csv(CM_DIR / "selected_confusion_matrices_conventional_vs_proposed_default_values.csv", index=False)

print("Saved:", cm_out)
display(cm_table)

# ============================================================
# 2. SHAP HEATMAP
# ============================================================

shap_fp = find_valid_heatmap_source("shap")
shap_df = pd.read_csv(shap_fp, low_memory=False)

old_shap = matrix_from_valid_summary(shap_df, "old", "shap")
new_shap = matrix_from_valid_summary(shap_df, "new", "shap")

plot_heatmap_pair(
    old_shap,
    new_shap,
    title="SHAP heatmap: Conventional vs Proposed",
    colorbar_label="Mean absolute SHAP value",
    out_fp=SHAP_DIR / "shap_heatmap_conventional_vs_proposed.png",
    cmap="YlOrRd",
    symmetric=False
)

# ============================================================
# 3. FEATURE ABLATION HEATMAP
# ============================================================

abl_fp = find_valid_heatmap_source("ablation")
abl_df = pd.read_csv(abl_fp, low_memory=False)

old_abl = matrix_from_valid_summary(abl_df, "old", "ablation")
new_abl = matrix_from_valid_summary(abl_df, "new", "ablation")

plot_heatmap_pair(
    old_abl,
    new_abl,
    title="Feature ablation heatmap: Conventional vs Proposed",
    colorbar_label="F1 drop",
    out_fp=ABL_DIR / "feature_ablation_conventional_vs_proposed.png",
    cmap="RdBu_r",
    symmetric=True
)

# ============================================================
# 4. POSITIVE MAPS
# ============================================================

old_pos = load_positive_case_points(OLD_CASE_DIR, "old")
new_pos = load_positive_case_points(NEW_CASE_DIR, "new")

old_pos.to_csv(TABLE_DIR / "positive_points_conventional.csv", index=False)
new_pos.to_csv(TABLE_DIR / "positive_points_proposed.csv", index=False)

if HAS_CARTOPY:
    fig, axes = plt.subplots(
        2,
        2,
        figsize=(15.5, 8.8),
        subplot_kw={"projection": ccrs.PlateCarree()}
    )
else:
    fig, axes = plt.subplots(2, 2, figsize=(15.5, 8.8))

plt.subplots_adjust(
    left=0.05,
    right=0.98,
    bottom=0.08,
    top=0.84,
    wspace=0.20,
    hspace=0.32
)

axes = axes.flatten()

for ax, lead in zip(axes, LEADS):
    old_sub = old_pos[old_pos["lead_h"] == lead].copy()
    new_sub = new_pos[new_pos["lead_h"] == lead].copy()

    setup_map(ax)

    scatter_kwargs_old = dict(
        s=32,
        c=COLOR_OLD,
        marker="o",
        alpha=0.76,
        label=f"{OLD_NAME} (n={len(old_sub)})",
        zorder=3
    )

    scatter_kwargs_new = dict(
        s=42,
        c=COLOR_NEW,
        marker="x",
        alpha=0.92,
        linewidths=1.25,
        label=f"{NEW_NAME} (n={len(new_sub)})",
        zorder=4
    )

    if HAS_CARTOPY:
        scatter_kwargs_old["transform"] = ccrs.PlateCarree()
        scatter_kwargs_new["transform"] = ccrs.PlateCarree()

    ax.scatter(old_sub["lon"], old_sub["lat"], **scatter_kwargs_old)
    ax.scatter(new_sub["lon"], new_sub["lat"], **scatter_kwargs_new)

    ax.set_title(f"{lead} h lead", fontsize=13, fontweight="bold")
    ax.legend(loc="lower left", fontsize=9, frameon=True)

fig.suptitle(
    "Positive MCS centroid locations, 2009–2025\n"
    "Conventional vs Proposed | box2deg exact-lead labels",
    fontsize=17,
    fontweight="bold",
    y=0.965
)

map_out = MAP_DIR / "positive_centroid_maps_conventional_vs_proposed_teal_salmon.png"
plt.savefig(map_out, dpi=250, bbox_inches="tight")
plt.show()

print("Saved:", map_out)

print("\nDONE.")
print("Output directory:")
print(OUT_DIR)

In [ ]:
# ============================================================
# FIXED SHAP + FEATURE ABLATION HEATMAPS
#
# Labels:
#   OLD -> Conventional
#   NEW -> Proposed
#
# Fixes:
#   1. Avoids raw SHAP prediction tables
#   2. Uses previous heatmap style
#   3. Keeps month / lat / lon in SHAP if available
#   4. Removes overlapping Predictor label on the right subplot
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

try:
    from IPython.display import display
except Exception:
    display = print

# ============================================================
# CONFIG
# ============================================================

LAB = Path(r"C:\Users\_s2218026\Documents\lab")

OLD_NAME = "Conventional"
NEW_NAME = "Proposed"

OUT_DIR = LAB / "FINAL_CONVENTIONAL_PROPOSED_PLOTS_FIXED_HEATMAPS"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SHAP_DIR = OUT_DIR / "shap_heatmap"
ABL_DIR = OUT_DIR / "feature_ablation"

SHAP_DIR.mkdir(parents=True, exist_ok=True)
ABL_DIR.mkdir(parents=True, exist_ok=True)

LEADS = [6, 12, 24, 48]
LEAD_LABELS = ["6 h", "12 h", "24 h", "48 h"]

# ------------------------------------------------------------
# Important:
# Set these manually if needed.
# If None, the code will search likely previous output folders.
# ------------------------------------------------------------

MANUAL_SHAP_SUMMARY = None
MANUAL_ABLATION_SUMMARY = None

# If you want to remove month/lat/lon from SHAP too, set this to True.
# Your screenshot includes month/lat/lon, so default is False.
REMOVE_GEO_FROM_SHAP = False

# Feature ablation usually should exclude month/lat/lon for physical interpretation.
REMOVE_GEO_FROM_ABLATION = True

# For ablation: choose default or tuned F1 drop if both exist.
ABLATION_VALUE_COL_PRIORITY = [
    "delta_default_f1",
    "f1_drop",
    "delta_f1",
    "drop_f1",
    "delta_tuned_f1",
    "value",
]

# ============================================================
# HELPERS
# ============================================================

def clean_df(df):
    return df.loc[:, ~df.columns.astype(str).str.startswith("Unnamed")].copy()


def recursive_csv_search(base_dir, keywords):
    base_dir = Path(base_dir)
    hits = []

    if not base_dir.exists():
        return hits

    for fp in base_dir.rglob("*.csv"):
        name = fp.name.lower()
        if all(k.lower() in name for k in keywords):
            hits.append(fp)

    return sorted(hits)


def normalize_lead(x):
    try:
        return f"{int(float(x))} h"
    except Exception:
        s = str(x).strip().lower()
        s = s.replace("hours", "h").replace("hour", "h").replace(" ", "")
        if s in ["6", "6h"]:
            return "6 h"
        if s in ["12", "12h"]:
            return "12 h"
        if s in ["24", "24h"]:
            return "24 h"
        if s in ["48", "48h"]:
            return "48 h"
        return str(x)


def is_raw_prediction_table(df):
    """
    Rejects tables that are raw prediction/explanation input tables,
    not summary heatmap tables.
    """
    cols = set(df.columns.astype(str))

    raw_markers = {
        "y_true", "y_prob", "y_score", "y_pred",
        "y_pred_default", "y_pred_tuned"
    }

    feature_cols = {
        "lon", "lat", "month", "area_km2", "tb_mean", "tb_min",
        "speed_kmh", "direction_deg", "eta850_abs", "rh600",
        "wshear_200_850", "pi_vmax_ms", "gpi"
    }

    if len(cols.intersection(raw_markers)) > 0 and len(cols.intersection(feature_cols)) >= 5:
        return True

    return False


def choose_existing_col(df, candidates):
    lower = {str(c).lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lower:
            return lower[c.lower()]
    return None


def matrix_from_summary(
    df,
    dataset_key,
    kind,
    remove_geo=False,
):
    """
    Converts long or wide summary table to heatmap matrix.

    Accepted SHAP long format:
      dataset, lead_h, feature, mean_abs_shap

    Accepted ablation long format:
      dataset, lead_h, removed_feature, delta_default_f1

    Accepted wide format:
      feature/predictor rows + 6 h / 12 h / 24 h / 48 h columns
    """

    df = clean_df(df).copy()
    df.columns = [str(c) for c in df.columns]

    if is_raw_prediction_table(df):
        raise ValueError(
            "This is a raw prediction/explanation table, not a SHAP/ablation summary table."
        )

    # ------------------------------------------------------------
    # Filter dataset if available
    # ------------------------------------------------------------

    if "dataset" in df.columns:
        ds = df["dataset"].astype(str).str.lower()

        if dataset_key == "old":
            valid_names = ["old", "conventional"]
        else:
            valid_names = ["new", "soft", "proposed"]

        df = df[ds.isin(valid_names)].copy()

    # ------------------------------------------------------------
    # Long-format parser
    # ------------------------------------------------------------

    pred_col = choose_existing_col(
        df,
        [
            "feature",
            "predictor",
            "removed_feature",
            "removed predictor",
            "variable",
            "name",
        ]
    )

    lead_col = choose_existing_col(
        df,
        [
            "lead_h",
            "lead",
            "lead time",
            "lead_time",
        ]
    )

    if kind == "shap":
        value_col = choose_existing_col(
            df,
            [
                "mean_abs_shap",
                "mean(|SHAP value|)",
                "mean_shap",
                "shap_value",
                "shap",
                "importance",
                "value",
            ]
        )
    else:
        value_col = choose_existing_col(df, ABLATION_VALUE_COL_PRIORITY)

    if pred_col is not None and lead_col is not None and value_col is not None:
        tmp = df[[pred_col, lead_col, value_col]].copy()

        tmp[pred_col] = tmp[pred_col].astype(str)
        tmp[lead_col] = tmp[lead_col].apply(normalize_lead)
        tmp[value_col] = pd.to_numeric(tmp[value_col], errors="coerce")

        mat = tmp.pivot_table(
            index=pred_col,
            columns=lead_col,
            values=value_col,
            aggfunc="mean"
        )

    else:
        # ------------------------------------------------------------
        # Wide-format parser
        # ------------------------------------------------------------
        lead_cols = []
        for c in df.columns:
            if normalize_lead(c) in LEAD_LABELS:
                lead_cols.append(c)

        if pred_col is None or len(lead_cols) == 0:
            raise ValueError(
                "Could not interpret this as a valid SHAP/ablation summary table.\n"
                f"Columns found: {df.columns.tolist()}"
            )

        mat = df.set_index(pred_col)[lead_cols].copy()
        mat.columns = [normalize_lead(c) for c in mat.columns]

    # ------------------------------------------------------------
    # Keep lead order
    # ------------------------------------------------------------

    mat = mat[[c for c in LEAD_LABELS if c in mat.columns]].copy()

    # ------------------------------------------------------------
    # Remove geo features if requested
    # ------------------------------------------------------------

    if remove_geo:
        remove = {"lon", "lat", "month", "mon"}
        keep_rows = [r for r in mat.index if str(r).strip().lower() not in remove]
        mat = mat.loc[keep_rows].copy()

    # ------------------------------------------------------------
    # Row order
    # ------------------------------------------------------------

    full_order = [
        "eta850_abs",
        "rh600",
        "wshear_200_850",
        "pi_vmax_ms",
        "gpi",
        "area_km2",
        "tb_mean",
        "tb_min",
        "speed_kmh",
        "direction_deg",
        "month",
        "lat",
        "lon",
    ]

    row_order = [r for r in full_order if r in mat.index] + [
        r for r in mat.index if r not in full_order
    ]

    mat = mat.loc[row_order].copy()

    if mat.empty:
        raise ValueError("Parsed matrix is empty.")

    return mat


def find_valid_summary_file(kind):
    """
    Finds a valid SHAP or feature-ablation summary file.
    This rejects raw prediction tables automatically.
    """

    if kind == "shap" and MANUAL_SHAP_SUMMARY is not None:
        return Path(MANUAL_SHAP_SUMMARY)

    if kind == "ablation" and MANUAL_ABLATION_SUMMARY is not None:
        return Path(MANUAL_ABLATION_SUMMARY)

    if kind == "shap":
        direct_candidates = [
            LAB / "OLD_NEW_MASTER_PLOTS_2009_2025_TEAL_SALMON" / "tables" / "shap_importance_adaboost_old_vs_new_no_geo.csv",
            LAB / "OLD_NEW_MASTER_PLOTS_2009_2025_TEAL_SALMON" / "tables" / "shap_importance_adaboost_old_vs_new_all_leads.csv",
            LAB / "shap_adaboost_old_vs_new_all_leads_no_geo" / "shap_importance_adaboost_old_vs_new_all_leads_no_geo.csv",
        ]

        search_roots = [
            LAB / "OLD_NEW_MASTER_PLOTS_2009_2025_TEAL_SALMON",
            LAB / "shap_adaboost_old_vs_new_all_leads_no_geo",
            LAB / "mcs_output",
            LAB / "mcs_output_softtest",
        ]

        keyword_groups = [
            ["shap", "importance", "adaboost"],
            ["shap", "importance"],
            ["shap", "adaboost"],
        ]

    else:
        direct_candidates = [
            LAB / "OLD_NEW_MASTER_PLOTS_2009_2025_TEAL_SALMON" / "tables" / "adaboost_feature_ablation_old_vs_new_no_geo.csv",
            LAB / "feature_ablation_adaboost_old_vs_new_all_leads_no_geo" / "adaboost_feature_ablation_old_vs_new_no_geo.csv",
        ]

        search_roots = [
            LAB / "OLD_NEW_MASTER_PLOTS_2009_2025_TEAL_SALMON",
            LAB / "feature_ablation_adaboost_old_vs_new_all_leads_no_geo",
            LAB / "mcs_output",
            LAB / "mcs_output_softtest",
        ]

        keyword_groups = [
            ["ablation", "adaboost"],
            ["feature", "ablation"],
        ]

    candidates = []

    for fp in direct_candidates:
        if fp.exists():
            candidates.append(fp)

    for root in search_roots:
        for keys in keyword_groups:
            candidates.extend(recursive_csv_search(root, keys))

    # Deduplicate
    seen = set()
    candidates = [fp for fp in candidates if not (str(fp) in seen or seen.add(str(fp)))]

    valid = []

    for fp in candidates:
        try:
            df = pd.read_csv(fp, low_memory=False)

            if is_raw_prediction_table(df):
                continue

            _old = matrix_from_summary(
                df,
                dataset_key="old",
                kind=kind,
                remove_geo=(REMOVE_GEO_FROM_SHAP if kind == "shap" else REMOVE_GEO_FROM_ABLATION)
            )

            _new = matrix_from_summary(
                df,
                dataset_key="new",
                kind=kind,
                remove_geo=(REMOVE_GEO_FROM_SHAP if kind == "shap" else REMOVE_GEO_FROM_ABLATION)
            )

            valid.append(fp)

        except Exception:
            continue

    if not valid:
        raise FileNotFoundError(
            f"No valid {kind} summary table found.\n\n"
            f"The code rejected raw prediction tables because they are not heatmap summaries.\n"
            f"If you know the correct file, set MANUAL_{kind.upper()}_SUMMARY at the top."
        )

    print(f"Using valid {kind} summary file:")
    print(valid[0])

    return valid[0]


def plot_heatmap_pair(
    old_mat,
    new_mat,
    title,
    cbar_label,
    out_fp,
    cmap="viridis",
    symmetric=False,
):
    old_vals = old_mat.values.astype(float)
    new_vals = new_mat.values.astype(float)

    if symmetric:
        vmax = max(np.nanmax(np.abs(old_vals)), np.nanmax(np.abs(new_vals)))
        vmin = -vmax
    else:
        vmin = min(np.nanmin(old_vals), np.nanmin(new_vals))
        vmax = max(np.nanmax(old_vals), np.nanmax(new_vals))

        if vmin > 0:
            vmin = 0

    fig, axes = plt.subplots(1, 2, figsize=(16.5, 7.5))

    # More space between subplots so "Predictor" does not overlap
    plt.subplots_adjust(
        left=0.10,
        right=0.88,
        bottom=0.12,
        top=0.82,
        wspace=0.45
    )

    for ax, mat, panel_title, show_ylabel in [
        (axes[0], old_mat, OLD_NAME, True),
        (axes[1], new_mat, NEW_NAME, False),
    ]:
        vals = mat.values.astype(float)

        im = ax.imshow(
            vals,
            aspect="auto",
            cmap=cmap,
            vmin=vmin,
            vmax=vmax
        )

        ax.set_title(panel_title, fontsize=16, fontweight="bold", pad=10)
        ax.set_xlabel("Lead time", fontsize=13)

        if show_ylabel:
            ax.set_ylabel("Predictor", fontsize=13, labelpad=12)
        else:
            ax.set_ylabel("")

        ax.set_xticks(np.arange(mat.shape[1]))
        ax.set_xticklabels(mat.columns, fontsize=12)

        ax.set_yticks(np.arange(mat.shape[0]))
        ax.set_yticklabels(mat.index, fontsize=12)

        threshold = 0.55 * max(abs(vmin), abs(vmax))

        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                val = vals[i, j]
                color = "white" if abs(val) > threshold else "black"

                ax.text(
                    j,
                    i,
                    f"{val:.3f}",
                    ha="center",
                    va="center",
                    fontsize=8.5,
                    color=color
                )

    cbar_ax = fig.add_axes([0.905, 0.18, 0.018, 0.58])
    cbar = fig.colorbar(im, cax=cbar_ax)
    cbar.set_label(cbar_label, fontsize=12)

    fig.suptitle(title, fontsize=18, fontweight="bold", y=0.95)

    plt.savefig(out_fp, dpi=250, bbox_inches="tight")
    plt.show()

    print("Saved:", out_fp)


# ============================================================
# SHAP HEATMAP
# ============================================================

shap_fp = find_valid_summary_file("shap")
shap_df = pd.read_csv(shap_fp, low_memory=False)

old_shap = matrix_from_summary(
    shap_df,
    dataset_key="old",
    kind="shap",
    remove_geo=REMOVE_GEO_FROM_SHAP
)

new_shap = matrix_from_summary(
    shap_df,
    dataset_key="new",
    kind="shap",
    remove_geo=REMOVE_GEO_FROM_SHAP
)

plot_heatmap_pair(
    old_mat=old_shap,
    new_mat=new_shap,
    title="SHAP heatmap: Conventional vs Proposed",
    cbar_label="mean(|SHAP value|)",
    out_fp=SHAP_DIR / "shap_heatmap_conventional_vs_proposed_fixed.png",
    cmap="viridis",
    symmetric=False,
)

display(old_shap)
display(new_shap)

# ============================================================
# FEATURE ABLATION HEATMAP
# ============================================================

abl_fp = find_valid_summary_file("ablation")
abl_df = pd.read_csv(abl_fp, low_memory=False)

old_abl = matrix_from_summary(
    abl_df,
    dataset_key="old",
    kind="ablation",
    remove_geo=REMOVE_GEO_FROM_ABLATION
)

new_abl = matrix_from_summary(
    abl_df,
    dataset_key="new",
    kind="ablation",
    remove_geo=REMOVE_GEO_FROM_ABLATION
)

plot_heatmap_pair(
    old_mat=old_abl,
    new_mat=new_abl,
    title="Feature ablation heatmap: Conventional vs Proposed",
    cbar_label="F1 drop",
    out_fp=ABL_DIR / "feature_ablation_conventional_vs_proposed_fixed.png",
    cmap="RdBu_r",
    symmetric=True,
)

display(old_abl)
display(new_abl)

print("\nDONE.")
print("Output directory:")
print(OUT_DIR)

In [ ]:
# ============================================================
# FINAL HEATMAPS — CONVENTIONAL vs PROPOSED
#
# Produces:
#   1. Conventional vs Proposed AdaBoost SHAP heatmap
#   2. Conventional vs Proposed AdaBoost feature ablation heatmap
#
# Style follows your previous figures:
#   - dataset | box2deg | AdaBoost in panel title
#   - "No lon/lat/month" in figure title
#   - Lead time on x-axis
#   - Predictor / Removed predictor on y-axis
#   - no overlapping y-axis labels
# ============================================================

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    display = print

# ============================================================
# CONFIG
# ============================================================

LAB = Path(r"C:\Users\_s2218026\Documents\lab")

OLD_NAME = "Conventional"
NEW_NAME = "Proposed"

LABEL_NAME = "box2deg"
MODEL_NAME = "AdaBoost"

LEAD_LABELS = ["6 h", "12 h", "24 h", "48 h"]

OUT_DIR = LAB / "FINAL_HEATMAPS_CONVENTIONAL_PROPOSED_ADABOOST"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SHAP_DIR = OUT_DIR / "shap"
ABL_DIR = OUT_DIR / "feature_ablation"

SHAP_DIR.mkdir(parents=True, exist_ok=True)
ABL_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# If auto-search fails, manually put the correct CSV path here.
# ------------------------------------------------------------

MANUAL_SHAP_SUMMARY = None
MANUAL_ABLATION_SUMMARY = None

# Remove lon/lat/month for both plots
REMOVE_SPATIOTEMPORAL = True

FEATURE_ORDER = [
    "eta850_abs",
    "rh600",
    "wshear_200_850",
    "pi_vmax_ms",
    "gpi",
    "area_km2",
    "tb_mean",
    "tb_min",
    "speed_kmh",
    "direction_deg",
]

# ============================================================
# HELPERS
# ============================================================

def clean_df(df):
    return df.loc[:, ~df.columns.astype(str).str.startswith("Unnamed")].copy()


def recursive_csv_search(base_dir, keywords):
    base_dir = Path(base_dir)
    hits = []

    if not base_dir.exists():
        return hits

    for fp in base_dir.rglob("*.csv"):
        name = fp.name.lower()
        if all(k.lower() in name for k in keywords):
            hits.append(fp)

    return sorted(hits)


def normalize_lead(x):
    try:
        return f"{int(float(x))} h"
    except Exception:
        s = str(x).strip().lower()
        s = s.replace("hours", "h").replace("hour", "h").replace(" ", "")
        if s in ["6", "6h"]:
            return "6 h"
        if s in ["12", "12h"]:
            return "12 h"
        if s in ["24", "24h"]:
            return "24 h"
        if s in ["48", "48h"]:
            return "48 h"
        return str(x)


def is_raw_prediction_table(df):
    cols = set(df.columns.astype(str))

    raw_cols = {
        "y_true",
        "y_prob",
        "y_score",
        "y_pred",
        "y_pred_default",
        "y_pred_tuned",
    }

    predictor_cols = {
        "lon",
        "lat",
        "month",
        "area_km2",
        "tb_mean",
        "tb_min",
        "speed_kmh",
        "direction_deg",
        "eta850_abs",
        "rh600",
        "wshear_200_850",
        "pi_vmax_ms",
        "gpi",
    }

    return bool(cols.intersection(raw_cols)) and len(cols.intersection(predictor_cols)) >= 5


def choose_existing_col(df, candidates):
    lower = {str(c).lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lower:
            return lower[c.lower()]
    return None


def matrix_from_summary(df, dataset_key, kind):
    """
    Converts a valid long-format or wide-format heatmap table into matrix.

    SHAP long format expected:
      dataset, lead_h, feature, mean_abs_shap

    Ablation long format expected:
      dataset, lead_h, removed_feature, delta_default_f1
    """

    df = clean_df(df).copy()
    df.columns = [str(c) for c in df.columns]

    if is_raw_prediction_table(df):
        raise ValueError(
            "This is a raw prediction table, not a SHAP/ablation summary table."
        )

    # --------------------------------------------------------
    # Filter dataset
    # --------------------------------------------------------

    if "dataset" in df.columns:
        ds = df["dataset"].astype(str).str.lower()

        if dataset_key == "old":
            valid_names = ["old", "conventional"]
        else:
            valid_names = ["new", "soft", "proposed"]

        df = df[ds.isin(valid_names)].copy()

    # --------------------------------------------------------
    # Detect columns
    # --------------------------------------------------------

    pred_col = choose_existing_col(
        df,
        [
            "feature",
            "predictor",
            "removed_feature",
            "removed predictor",
            "variable",
            "name",
        ]
    )

    lead_col = choose_existing_col(
        df,
        [
            "lead_h",
            "lead",
            "lead_time",
            "lead time",
        ]
    )

    if kind == "shap":
        value_col = choose_existing_col(
            df,
            [
                "mean_abs_shap",
                "mean(|SHAP value|)",
                "mean_shap",
                "shap_value",
                "shap",
                "importance",
                "value",
            ]
        )
    else:
        value_col = choose_existing_col(
            df,
            [
                "delta_default_f1",
                "f1_drop",
                "delta_f1",
                "drop_f1",
                "delta_tuned_f1",
                "value",
            ]
        )

    # --------------------------------------------------------
    # Long format
    # --------------------------------------------------------

    if pred_col is not None and lead_col is not None and value_col is not None:
        tmp = df[[pred_col, lead_col, value_col]].copy()

        tmp[pred_col] = tmp[pred_col].astype(str)
        tmp[lead_col] = tmp[lead_col].apply(normalize_lead)
        tmp[value_col] = pd.to_numeric(tmp[value_col], errors="coerce")

        mat = tmp.pivot_table(
            index=pred_col,
            columns=lead_col,
            values=value_col,
            aggfunc="mean"
        )

    # --------------------------------------------------------
    # Wide format
    # --------------------------------------------------------

    else:
        lead_cols = []

        for c in df.columns:
            if normalize_lead(c) in LEAD_LABELS:
                lead_cols.append(c)

        if pred_col is None or len(lead_cols) == 0:
            raise ValueError(
                "Could not interpret table as heatmap summary.\n"
                f"Columns found: {df.columns.tolist()}"
            )

        mat = df.set_index(pred_col)[lead_cols].copy()
        mat.columns = [normalize_lead(c) for c in mat.columns]

    # --------------------------------------------------------
    # Keep lead order
    # --------------------------------------------------------

    mat = mat[[c for c in LEAD_LABELS if c in mat.columns]].copy()

    # --------------------------------------------------------
    # Remove lon/lat/month
    # --------------------------------------------------------

    if REMOVE_SPATIOTEMPORAL:
        remove = {"lon", "lat", "month", "mon"}
        mat = mat.loc[
            [r for r in mat.index if str(r).strip().lower() not in remove]
        ].copy()

    # --------------------------------------------------------
    # Row order
    # --------------------------------------------------------

    row_order = [r for r in FEATURE_ORDER if r in mat.index]
    row_order += [r for r in mat.index if r not in row_order]

    mat = mat.loc[row_order].copy()

    if mat.empty:
        raise ValueError("Parsed heatmap matrix is empty.")

    return mat


def find_valid_summary_file(kind):
    """
    Finds valid SHAP / ablation summary CSV.
    Rejects raw prediction tables automatically.
    """

    if kind == "shap" and MANUAL_SHAP_SUMMARY is not None:
        return Path(MANUAL_SHAP_SUMMARY)

    if kind == "ablation" and MANUAL_ABLATION_SUMMARY is not None:
        return Path(MANUAL_ABLATION_SUMMARY)

    if kind == "shap":
        direct_candidates = [
            LAB / "OLD_NEW_MASTER_PLOTS_2009_2025_TEAL_SALMON" / "tables" / "shap_importance_adaboost_old_vs_new_no_geo.csv",
            LAB / "shap_adaboost_old_vs_new_all_leads_no_geo" / "shap_importance_adaboost_old_vs_new_all_leads_no_geo.csv",
        ]

        search_roots = [
            LAB / "OLD_NEW_MASTER_PLOTS_2009_2025_TEAL_SALMON",
            LAB / "shap_adaboost_old_vs_new_all_leads_no_geo",
            LAB / "mcs_output",
            LAB / "mcs_output_softtest",
        ]

        keyword_groups = [
            ["shap", "importance", "adaboost"],
            ["shap", "importance"],
        ]

    else:
        direct_candidates = [
            LAB / "OLD_NEW_MASTER_PLOTS_2009_2025_TEAL_SALMON" / "tables" / "adaboost_feature_ablation_old_vs_new_no_geo.csv",
            LAB / "feature_ablation_adaboost_old_vs_new_all_leads_no_geo" / "adaboost_feature_ablation_old_vs_new_no_geo.csv",
        ]

        search_roots = [
            LAB / "OLD_NEW_MASTER_PLOTS_2009_2025_TEAL_SALMON",
            LAB / "feature_ablation_adaboost_old_vs_new_all_leads_no_geo",
            LAB / "mcs_output",
            LAB / "mcs_output_softtest",
        ]

        keyword_groups = [
            ["ablation", "adaboost"],
            ["feature", "ablation"],
        ]

    candidates = []

    for fp in direct_candidates:
        if fp.exists():
            candidates.append(fp)

    for root in search_roots:
        for keys in keyword_groups:
            candidates.extend(recursive_csv_search(root, keys))

    # Deduplicate
    seen = set()
    candidates = [fp for fp in candidates if not (str(fp) in seen or seen.add(str(fp)))]

    valid = []

    for fp in candidates:
        try:
            df = pd.read_csv(fp, low_memory=False)

            old_mat = matrix_from_summary(df, dataset_key="old", kind=kind)
            new_mat = matrix_from_summary(df, dataset_key="new", kind=kind)

            if old_mat.shape[0] > 0 and new_mat.shape[0] > 0:
                valid.append(fp)

        except Exception:
            continue

    if not valid:
        raise FileNotFoundError(
            f"No valid {kind} summary table found.\n"
            f"Set MANUAL_{kind.upper()}_SUMMARY at the top if needed."
        )

    print(f"Using {kind} summary file:")
    print(valid[0])

    return valid[0]


def plot_shap_pair(old_mat, new_mat):
    old_vals = old_mat.values.astype(float)
    new_vals = new_mat.values.astype(float)

    vmin = 0
    vmax = max(np.nanmax(old_vals), np.nanmax(new_vals))

    fig, axes = plt.subplots(1, 2, figsize=(14.5, 7.2))

    plt.subplots_adjust(
        left=0.08,
        right=0.88,
        bottom=0.12,
        top=0.80,
        wspace=0.38
    )

    for ax, mat, dataset_title, show_ylabel in [
        (axes[0], old_mat, OLD_NAME, True),
        (axes[1], new_mat, NEW_NAME, False),
    ]:
        vals = mat.values.astype(float)

        im = ax.imshow(
            vals,
            aspect="auto",
            cmap="viridis",
            vmin=vmin,
            vmax=vmax
        )

        ax.set_title(
            f"{dataset_title} {LABEL_NAME} | {MODEL_NAME}\n"
            f"SHAP heatmap, mean |SHAP value|",
            fontsize=12,
            pad=8
        )

        ax.set_xlabel("Lead time", fontsize=12)

        if show_ylabel:
            ax.set_ylabel("Predictor", fontsize=12, labelpad=12)
        else:
            ax.set_ylabel("")

        ax.set_xticks(np.arange(mat.shape[1]))
        ax.set_xticklabels(mat.columns, fontsize=11)

        ax.set_yticks(np.arange(mat.shape[0]))
        ax.set_yticklabels(mat.index, fontsize=11)

        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                val = vals[i, j]
                text_color = "white" if val > 0.55 * vmax else "black"

                ax.text(
                    j,
                    i,
                    f"{val:.3f}",
                    ha="center",
                    va="center",
                    fontsize=7.5,
                    color=text_color
                )

    cbar_ax = fig.add_axes([0.90, 0.18, 0.018, 0.56])
    cbar = fig.colorbar(im, cax=cbar_ax)
    cbar.set_label("Mean absolute SHAP value", fontsize=11)

    fig.suptitle(
        f"{OLD_NAME} vs {NEW_NAME} {MODEL_NAME} SHAP heatmap\n"
        f"No lon/lat/month",
        fontsize=16,
        y=0.95
    )

    out_png = SHAP_DIR / "conventional_vs_proposed_adaboost_shap_heatmap_no_geo.png"
    plt.savefig(out_png, dpi=250, bbox_inches="tight")
    plt.show()

    print("Saved:", out_png)


def plot_ablation_pair(old_mat, new_mat):
    old_vals = old_mat.values.astype(float)
    new_vals = new_mat.values.astype(float)

    vmax = max(np.nanmax(np.abs(old_vals)), np.nanmax(np.abs(new_vals)))
    vmin = -vmax

    fig, axes = plt.subplots(1, 2, figsize=(14.5, 7.2))

    plt.subplots_adjust(
        left=0.08,
        right=0.88,
        bottom=0.12,
        top=0.80,
        wspace=0.38
    )

    for ax, mat, dataset_title, show_ylabel in [
        (axes[0], old_mat, OLD_NAME, True),
        (axes[1], new_mat, NEW_NAME, False),
    ]:
        vals = mat.values.astype(float)

        im = ax.imshow(
            vals,
            aspect="auto",
            cmap="RdBu_r",
            vmin=vmin,
            vmax=vmax
        )

        ax.set_title(
            f"{dataset_title} {LABEL_NAME} | {MODEL_NAME}\n"
            f"Feature ablation (default threshold)",
            fontsize=12,
            pad=8
        )

        ax.set_xlabel("Lead time", fontsize=12)

        if show_ylabel:
            ax.set_ylabel("Removed predictor", fontsize=12, labelpad=12)
        else:
            ax.set_ylabel("")

        ax.set_xticks(np.arange(mat.shape[1]))
        ax.set_xticklabels(mat.columns, fontsize=11)

        ax.set_yticks(np.arange(mat.shape[0]))
        ax.set_yticklabels(mat.index, fontsize=11)

        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                val = vals[i, j]
                text_color = "white" if abs(val) > 0.55 * vmax else "black"

                ax.text(
                    j,
                    i,
                    f"{val:.3f}",
                    ha="center",
                    va="center",
                    fontsize=7.5,
                    color=text_color
                )

    cbar_ax = fig.add_axes([0.90, 0.18, 0.018, 0.56])
    cbar = fig.colorbar(im, cax=cbar_ax)
    cbar.set_label("F1 drop", fontsize=11)

    fig.suptitle(
        f"{OLD_NAME} vs {NEW_NAME} {MODEL_NAME} feature ablation heatmap\n"
        f"No lon/lat/month | value = baseline F1 − ablated F1",
        fontsize=16,
        y=0.95
    )

    out_png = ABL_DIR / "conventional_vs_proposed_adaboost_feature_ablation_no_geo.png"
    plt.savefig(out_png, dpi=250, bbox_inches="tight")
    plt.show()

    print("Saved:", out_png)


# ============================================================
# RUN SHAP
# ============================================================

shap_fp = find_valid_summary_file("shap")
shap_df = pd.read_csv(shap_fp, low_memory=False)

old_shap = matrix_from_summary(shap_df, dataset_key="old", kind="shap")
new_shap = matrix_from_summary(shap_df, dataset_key="new", kind="shap")

plot_shap_pair(old_shap, new_shap)

print("\nSHAP matrix — Conventional:")
display(old_shap)

print("\nSHAP matrix — Proposed:")
display(new_shap)

# ============================================================
# RUN FEATURE ABLATION
# ============================================================

abl_fp = find_valid_summary_file("ablation")
abl_df = pd.read_csv(abl_fp, low_memory=False)

old_abl = matrix_from_summary(abl_df, dataset_key="old", kind="ablation")
new_abl = matrix_from_summary(abl_df, dataset_key="new", kind="ablation")

plot_ablation_pair(old_abl, new_abl)

print("\nFeature ablation matrix — Conventional:")
display(old_abl)

print("\nFeature ablation matrix — Proposed:")
display(new_abl)

print("\nDONE.")
print("Output directory:")
print(OUT_DIR)

In [ ]:
# ============================================================
# POSITIVE LABEL SUMMARY + MAPS
# Conventional vs Proposed, 2009–2025
#
# Labels:
#   OLD -> Conventional
#   NEW -> Proposed
#
# Colors:
#   Conventional = teal
#   Proposed     = salmon
#
# Outputs:
#   1. Positive count bar plot
#   2. Positive rate bar plot
#   3. Positive centroid maps, 2x2 by lead time
#   4. Summary CSV
# ============================================================

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAS_CARTOPY = True
except Exception:
    HAS_CARTOPY = False
    print("[WARNING] Cartopy not available. Maps will use plain matplotlib axes.")

try:
    from IPython.display import display
except Exception:
    display = print

# ============================================================
# CONFIG
# ============================================================

LAB = Path(r"C:\Users\_s2218026\Documents\lab")

OLD_NAME = "Conventional"
NEW_NAME = "MR"

COLOR_OLD = "#2A9D8F"   # teal
COLOR_NEW = "#E76F51"   # salmon

LEADS = [6, 12, 24, 48]
LABEL_NAME = "box2deg"

START_YEAR = 2009
END_YEAR = 2025

# WNP map extent
LON_MIN, LON_MAX = 100, 180
LAT_MIN, LAT_MAX = 0, 30

OLD_CASE_DIR = (
    LAB
    / "mcs_output"
    / "exactlead_box2deg_case_tables_old_2009_2025_UPDATED"
)

NEW_CASE_DIR = (
    LAB
    / "mcs_output_softtest"
    / "exactlead_box2deg_case_tables_new_2009_2025_UPDATED"
)

OUT_DIR = LAB / "FINAL_POSITIVE_MAPS_AND_BARS_CONVENTIONAL_PROPOSED"
OUT_DIR.mkdir(parents=True, exist_ok=True)

BAR_DIR = OUT_DIR / "bar_plots"
MAP_DIR = OUT_DIR / "positive_maps"
TABLE_DIR = OUT_DIR / "tables"

for d in [BAR_DIR, MAP_DIR, TABLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ============================================================
# HELPERS
# ============================================================

def clean_df(df):
    return df.loc[:, ~df.columns.astype(str).str.startswith("Unnamed")].copy()


def find_case_file(case_dir, dataset_name, lead):
    candidates = [
        case_dir / f"{LABEL_NAME}_case_table_{dataset_name}_{lead}h.csv",
        case_dir / f"zhang_case_table_{dataset_name}_{lead}h.csv",
        case_dir / f"zhang_case_table_{lead}h.csv",
        case_dir / f"case_table_{lead}h.csv",
    ]

    for fp in candidates:
        if fp.exists():
            return fp

    raise FileNotFoundError(
        f"No case-table file found for {dataset_name}, lead={lead}h.\n"
        "Checked:\n"
        + "\n".join(str(fp) for fp in candidates)
    )


def detect_lon_lat_cols(df):
    lon_candidates = [
        "centroid_lon",
        "lon",
        "longitude",
        "mcs_lon",
        "genesis_lon",
    ]

    lat_candidates = [
        "centroid_lat",
        "lat",
        "latitude",
        "mcs_lat",
        "genesis_lat",
    ]

    lon_col = next((c for c in lon_candidates if c in df.columns), None)
    lat_col = next((c for c in lat_candidates if c in df.columns), None)

    if lon_col is None or lat_col is None:
        raise ValueError(
            "Could not find longitude/latitude columns.\n"
            f"Available columns:\n{df.columns.tolist()}"
        )

    return lon_col, lat_col


def parse_and_filter_time(df):
    if "time" in df.columns:
        df["time"] = pd.to_datetime(
            df["time"],
            errors="coerce",
            utc=True
        ).dt.tz_convert(None)

        df = df.dropna(subset=["time"]).copy()

        df = df[
            (df["time"].dt.year >= START_YEAR)
            & (df["time"].dt.year <= END_YEAR)
        ].copy()

    return df


def load_case_positive_points(case_dir, dataset_name, display_name):
    all_points = []
    summary_rows = []

    for lead in LEADS:
        fp = find_case_file(case_dir, dataset_name, lead)

        df = pd.read_csv(fp, low_memory=False)
        df = clean_df(df)
        df = parse_and_filter_time(df)

        if "y" not in df.columns:
            raise ValueError(f"{fp} has no 'y' column.")

        df["y"] = df["y"].astype(int)

        total_n = len(df)
        positive_n = int(df["y"].sum())
        negative_n = int((df["y"] == 0).sum())
        positive_rate = positive_n / total_n if total_n > 0 else np.nan

        lon_col, lat_col = detect_lon_lat_cols(df)

        pos = df[df["y"] == 1].copy()

        points = pd.DataFrame({
            "dataset": dataset_name,
            "display_name": display_name,
            "lead_h": lead,
            "lon": pd.to_numeric(pos[lon_col], errors="coerce"),
            "lat": pd.to_numeric(pos[lat_col], errors="coerce"),
        }).dropna(subset=["lon", "lat"])

        if "time" in pos.columns:
            points["time"] = pos["time"].values

        if "track_id" in pos.columns:
            points["track_id"] = pos["track_id"].astype(str).values

        all_points.append(points)

        summary_rows.append({
            "dataset": dataset_name,
            "display_name": display_name,
            "lead_h": lead,
            "total_n": total_n,
            "positive_n": positive_n,
            "negative_n": negative_n,
            "positive_rate": positive_rate,
            "file": str(fp),
        })

        print(
            f"{display_name} | {lead}h | "
            f"total={total_n}, positive={positive_n}, "
            f"rate={positive_rate:.4f} | {fp.name}"
        )

    points_all = pd.concat(all_points, ignore_index=True)
    summary = pd.DataFrame(summary_rows)

    return points_all, summary


def setup_map(ax):
    if HAS_CARTOPY:
        ax.set_extent(
            [LON_MIN, LON_MAX, LAT_MIN, LAT_MAX],
            crs=ccrs.PlateCarree()
        )

        ax.add_feature(cfeature.OCEAN, facecolor="white", zorder=0)
        ax.add_feature(cfeature.LAND, facecolor="whitesmoke", zorder=0)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.7, zorder=1)
        ax.add_feature(cfeature.BORDERS, linewidth=0.35, alpha=0.7, zorder=1)

        gl = ax.gridlines(
            draw_labels=True,
            linewidth=0.35,
            color="gray",
            alpha=0.45,
            linestyle="--"
        )

        gl.top_labels = False
        gl.right_labels = False

    else:
        ax.set_xlim(LON_MIN, LON_MAX)
        ax.set_ylim(LAT_MIN, LAT_MAX)
        ax.grid(True, linestyle="--", alpha=0.35)

    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")


# ============================================================
# LOAD DATA
# ============================================================

print("\n" + "=" * 90)
print("LOADING POSITIVE CASE TABLES")
print("=" * 90)

old_points, old_summary = load_case_positive_points(
    case_dir=OLD_CASE_DIR,
    dataset_name="old",
    display_name=OLD_NAME
)

new_points, new_summary = load_case_positive_points(
    case_dir=NEW_CASE_DIR,
    dataset_name="new",
    display_name=NEW_NAME
)

points_all = pd.concat([old_points, new_points], ignore_index=True)
summary_all = pd.concat([old_summary, new_summary], ignore_index=True)

summary_wide = summary_all.pivot(
    index="lead_h",
    columns="display_name",
    values=["positive_n", "positive_rate", "total_n"]
).reset_index()

summary_all.to_csv(
    TABLE_DIR / "positive_summary_conventional_vs_proposed_long.csv",
    index=False
)

points_all.to_csv(
    TABLE_DIR / "positive_points_conventional_vs_proposed.csv",
    index=False
)

print("\nSummary:")
display(summary_all)

# ============================================================
# BAR PLOT 1 — POSITIVE COUNTS
# ============================================================

plot_df = summary_all.copy()

old_bar = plot_df[plot_df["display_name"] == OLD_NAME].sort_values("lead_h")
new_bar = plot_df[plot_df["display_name"] == NEW_NAME].sort_values("lead_h")

x = np.arange(len(LEADS))
width = 0.36

fig, ax = plt.subplots(figsize=(9.5, 5.4))

ax.bar(
    x - width / 2,
    old_bar["positive_n"],
    width=width,
    label=OLD_NAME,
    color=COLOR_OLD,
    edgecolor="black",
    linewidth=0.6
)

ax.bar(
    x + width / 2,
    new_bar["positive_n"],
    width=width,
    label=NEW_NAME,
    color=COLOR_NEW,
    edgecolor="black",
    linewidth=0.6
)

ax.set_xticks(x)
ax.set_xticklabels([f"{L} h" for L in LEADS])
ax.set_xlabel("Lead time")
ax.set_ylabel("Number of positive MCS snapshots")
ax.set_title(
    "Positive MCS label counts, 2009–2025\n"
    "Conventional vs Proposed"
)
ax.legend(title="Dataset")
ax.grid(True, axis="y", alpha=0.3)

for i, (old_v, new_v) in enumerate(zip(old_bar["positive_n"], new_bar["positive_n"])):
    ax.text(i - width / 2, old_v, f"{int(old_v)}", ha="center", va="bottom", fontsize=10)
    ax.text(i + width / 2, new_v, f"{int(new_v)}", ha="center", va="bottom", fontsize=10)

plt.tight_layout()

out_bar_counts = BAR_DIR / "positive_counts_conventional_vs_proposed_teal_salmon.png"
plt.savefig(out_bar_counts, dpi=250, bbox_inches="tight")
plt.show()

print("Saved:", out_bar_counts)

# ============================================================
# BAR PLOT 2 — POSITIVE RATE
# ============================================================

fig, ax = plt.subplots(figsize=(9.5, 5.4))

ax.bar(
    x - width / 2,
    old_bar["positive_rate"],
    width=width,
    label=OLD_NAME,
    color=COLOR_OLD,
    edgecolor="black",
    linewidth=0.6
)

ax.bar(
    x + width / 2,
    new_bar["positive_rate"],
    width=width,
    label=NEW_NAME,
    color=COLOR_NEW,
    edgecolor="black",
    linewidth=0.6
)

ax.set_xticks(x)
ax.set_xticklabels([f"{L} h" for L in LEADS])
ax.set_xlabel("Lead time")
ax.set_ylabel("Positive label rate")
ax.set_title(
    "Positive MCS label rate, 2009–2025\n"
    "Conventional vs Proposed"
)
ax.legend(title="Dataset")
ax.grid(True, axis="y", alpha=0.3)

for i, (old_v, new_v) in enumerate(zip(old_bar["positive_rate"], new_bar["positive_rate"])):
    ax.text(i - width / 2, old_v, f"{old_v:.3f}", ha="center", va="bottom", fontsize=10)
    ax.text(i + width / 2, new_v, f"{new_v:.3f}", ha="center", va="bottom", fontsize=10)

plt.tight_layout()

out_bar_rate = BAR_DIR / "positive_rate_conventional_vs_proposed_teal_salmon.png"
plt.savefig(out_bar_rate, dpi=250, bbox_inches="tight")
plt.show()

print("Saved:", out_bar_rate)

# ============================================================
# POSITIVE CENTROID MAP — 2x2 BY LEAD
# ============================================================

if HAS_CARTOPY:
    fig, axes = plt.subplots(
        2,
        2,
        figsize=(15.5, 8.8),
        subplot_kw={"projection": ccrs.PlateCarree()}
    )
else:
    fig, axes = plt.subplots(
        2,
        2,
        figsize=(15.5, 8.8)
    )

plt.subplots_adjust(
    left=0.05,
    right=0.98,
    bottom=0.08,
    top=0.84,
    wspace=0.20,
    hspace=0.32
)

axes = axes.flatten()

for ax, lead in zip(axes, LEADS):
    old_sub = points_all[
        (points_all["display_name"] == OLD_NAME)
        & (points_all["lead_h"] == lead)
    ].copy()

    new_sub = points_all[
        (points_all["display_name"] == NEW_NAME)
        & (points_all["lead_h"] == lead)
    ].copy()

    setup_map(ax)

    old_kwargs = dict(
        s=32,
        c=COLOR_OLD,
        marker="o",
        alpha=0.76,
        label=f"{OLD_NAME} (n={len(old_sub)})",
        zorder=3
    )

    new_kwargs = dict(
        s=42,
        c=COLOR_NEW,
        marker="x",
        alpha=0.92,
        linewidths=1.25,
        label=f"{NEW_NAME} (n={len(new_sub)})",
        zorder=4
    )

    if HAS_CARTOPY:
        old_kwargs["transform"] = ccrs.PlateCarree()
        new_kwargs["transform"] = ccrs.PlateCarree()

    ax.scatter(old_sub["lon"], old_sub["lat"], **old_kwargs)
    ax.scatter(new_sub["lon"], new_sub["lat"], **new_kwargs)

    ax.set_title(f"{lead} h lead", fontsize=13, fontweight="bold")
    ax.legend(loc="lower left", fontsize=9, frameon=True)

fig.suptitle(
    "Positive MCS centroid locations, 2009–2025\n"
    "Conventional vs Proposed | box2deg exact-lead labels",
    fontsize=17,
    fontweight="bold",
    y=0.965
)

out_map = MAP_DIR / "positive_centroid_maps_conventional_vs_proposed_teal_salmon.png"
plt.savefig(out_map, dpi=250, bbox_inches="tight")
plt.show()

print("Saved:", out_map)

# ============================================================
# DONE
# ============================================================

print("\nDONE.")
print("Output directory:")
print(OUT_DIR)

In [ ]:
# ============================================================
# CELL 1 — CONFUSION MATRIX + SHAP + FEATURE ABLATION
# Conventional vs MR
#
# OLD -> Conventional
# NEW -> MR
# MR = morphology-refined segmentation
# ============================================================

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    display = print

# ============================================================
# CONFIG
# ============================================================

LAB = Path(r"C:\Users\_s2218026\Documents\lab")

OLD_NAME = "Conventional"
NEW_NAME = "MR"

LABEL_NAME = "box2deg"
MODEL_NAME = "AdaBoost"

LEADS = [6, 12, 24, 48]
LEAD_LABELS = ["6 h", "12 h", "24 h", "48 h"]

# Selected models for confusion matrix
CONVENTIONAL_CM_MODELS = {
    6:  "AdaBoost",
    12: "Naive Bayes",
    24: "AdaBoost",
    48: "MLP",
}

MR_CM_MODELS = {
    6:  "AdaBoost",
    12: "Naive Bayes",
    24: "AdaBoost",
    48: "Naive Bayes",
}

OLD_OOF_CANDIDATES = [
    LAB / "mcs_output" / "ml_old_box2deg_2009_2025_final" / "oof_predictions_all_models.csv",
    LAB / "mcs_output" / "ml_old_box2deg_2009_2025_final" / "oof_predictions_allmodels_old.csv",
    LAB / "mcs_output" / "ml_old_box2deg_2009_2025_UPDATED" / "oof_predictions_all_models.csv",
    LAB / "mcs_output" / "ml_old_box2deg_2009_2025_UPDATED" / "oof_predictions_allmodels_old.csv",
]

NEW_OOF_CANDIDATES = [
    LAB / "mcs_output_softtest" / "ml_new_box2deg_2009_2025_UPDATED" / "oof_predictions_allmodels_new.csv",
    LAB / "mcs_output_softtest" / "ml_new_box2deg_2009_2025_UPDATED" / "oof_predictions_all_models.csv",
    LAB / "mcs_output_softtest" / "ml_new_box2deg_2009_2025_UPDATED" / "oof_predictions_allmodels_soft.csv",
]

# Optional manual paths if auto-search picks wrong file
MANUAL_SHAP_SUMMARY = None
MANUAL_ABLATION_SUMMARY = None

REMOVE_SPATIOTEMPORAL = True

FEATURE_ORDER = [
    "eta850_abs",
    "rh600",
    "wshear_200_850",
    "pi_vmax_ms",
    "gpi",
    "area_km2",
    "tb_mean",
    "tb_min",
    "speed_kmh",
    "direction_deg",
]

OUT_DIR = LAB / "FINAL_PLOTS_CONVENTIONAL_vs_MR"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CM_DIR = OUT_DIR / "confusion_matrices"
SHAP_DIR = OUT_DIR / "shap_heatmap"
ABL_DIR = OUT_DIR / "feature_ablation"
TABLE_DIR = OUT_DIR / "tables"

for d in [CM_DIR, SHAP_DIR, ABL_DIR, TABLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ============================================================
# BASIC HELPERS
# ============================================================

def clean_df(df):
    return df.loc[:, ~df.columns.astype(str).str.startswith("Unnamed")].copy()


def first_existing(paths, label):
    for fp in paths:
        if fp.exists():
            return fp
    raise FileNotFoundError(
        f"No file found for {label}. Checked:\n"
        + "\n".join(str(fp) for fp in paths)
    )


def recursive_csv_search(base_dir, keywords):
    base_dir = Path(base_dir)
    hits = []

    if not base_dir.exists():
        return hits

    for fp in base_dir.rglob("*.csv"):
        name = fp.name.lower()
        if all(k.lower() in name for k in keywords):
            hits.append(fp)

    return sorted(hits)


def normalize_lead(x):
    try:
        return f"{int(float(x))} h"
    except Exception:
        s = str(x).strip().lower()
        s = s.replace("hours", "h").replace("hour", "h").replace(" ", "")
        if s in ["6", "6h"]:
            return "6 h"
        if s in ["12", "12h"]:
            return "12 h"
        if s in ["24", "24h"]:
            return "24 h"
        if s in ["48", "48h"]:
            return "48 h"
        return str(x)


def find_model_name(models_available, target_name):
    models_available = list(models_available)

    if target_name in models_available:
        return target_name

    def canon(s):
        return str(s).lower().replace(" ", "").replace("_", "").replace("-", "")

    target = canon(target_name)
    model_map = {canon(m): m for m in models_available}

    if target in model_map:
        return model_map[target]

    aliases = {
        "naivebayes": ["naivebayes", "nb", "gaussiannb", "gaussiannaivebayes"],
        "adaboost": ["adaboost", "ada", "adaboostclassifier"],
        "mlp": ["mlp", "multilayerperceptron", "neuralnetwork"],
        "randomforest": ["randomforest", "rf"],
        "svm": ["svm"],
    }

    for key, alias_list in aliases.items():
        if target == key or target in alias_list:
            for c, original in model_map.items():
                if c == key or c in alias_list:
                    return original

    raise ValueError(
        f"Could not find model '{target_name}'. Available models:\n"
        f"{sorted(models_available)}"
    )


def choose_default_prediction(sub):
    if "y_pred_default" in sub.columns:
        return sub["y_pred_default"].astype(int).values
    if "y_pred" in sub.columns:
        return sub["y_pred"].astype(int).values
    if "y_score" in sub.columns:
        return (sub["y_score"].astype(float).values >= 0.5).astype(int)
    if "y_prob" in sub.columns:
        return (sub["y_prob"].astype(float).values >= 0.5).astype(int)

    raise ValueError(
        "No prediction column found. Need y_pred_default, y_pred, y_score, or y_prob."
    )


def row_normalize_cm(cm):
    cm = np.asarray(cm, dtype=float)
    row_sum = cm.sum(axis=1, keepdims=True)
    return np.divide(cm, row_sum, out=np.zeros_like(cm), where=row_sum != 0)


def draw_normalized_cm(ax, cm_norm, title):
    ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])

    ax.set_xticklabels(["Non-dev", "Dev"], fontsize=11)
    ax.set_yticklabels(["Non-dev", "Dev"], fontsize=11)

    ax.set_xlabel("Predicted outcome", fontsize=11)
    ax.set_ylabel("True outcome", fontsize=11)

    ax.set_title(title, fontsize=13, fontweight="bold", pad=8)

    for i in range(2):
        for j in range(2):
            val = cm_norm[i, j]
            ax.text(
                j,
                i,
                f"{val:.2f}",
                ha="center",
                va="center",
                fontsize=15,
                fontweight="bold",
                color="white" if val > 0.55 else "black"
            )

# ============================================================
# HEATMAP HELPERS
# ============================================================

def is_raw_prediction_table(df):
    cols = set(df.columns.astype(str))

    raw_cols = {
        "y_true",
        "y_prob",
        "y_score",
        "y_pred",
        "y_pred_default",
        "y_pred_tuned",
    }

    predictor_cols = {
        "lon",
        "lat",
        "month",
        "area_km2",
        "tb_mean",
        "tb_min",
        "speed_kmh",
        "direction_deg",
        "eta850_abs",
        "rh600",
        "wshear_200_850",
        "pi_vmax_ms",
        "gpi",
    }

    return bool(cols.intersection(raw_cols)) and len(cols.intersection(predictor_cols)) >= 5


def choose_existing_col(df, candidates):
    lower = {str(c).lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lower:
            return lower[c.lower()]
    return None


def matrix_from_summary(df, dataset_key, kind):
    df = clean_df(df).copy()
    df.columns = [str(c) for c in df.columns]

    if is_raw_prediction_table(df):
        raise ValueError(
            "This is a raw prediction table, not a SHAP/ablation summary table."
        )

    if "dataset" in df.columns:
        ds = df["dataset"].astype(str).str.lower()

        if dataset_key == "old":
            valid_names = ["old", "conventional"]
        else:
            valid_names = ["new", "soft", "mr", "morphology-refined", "morphology refined"]

        df = df[ds.isin(valid_names)].copy()

    pred_col = choose_existing_col(
        df,
        [
            "feature",
            "predictor",
            "removed_feature",
            "removed predictor",
            "variable",
            "name",
        ]
    )

    lead_col = choose_existing_col(
        df,
        [
            "lead_h",
            "lead",
            "lead_time",
            "lead time",
        ]
    )

    if kind == "shap":
        value_col = choose_existing_col(
            df,
            [
                "mean_abs_shap",
                "mean(|SHAP value|)",
                "mean_shap",
                "shap_value",
                "shap",
                "importance",
                "value",
            ]
        )
    else:
        value_col = choose_existing_col(
            df,
            [
                "delta_default_f1",
                "f1_drop",
                "delta_f1",
                "drop_f1",
                "delta_tuned_f1",
                "value",
            ]
        )

    if pred_col is not None and lead_col is not None and value_col is not None:
        tmp = df[[pred_col, lead_col, value_col]].copy()

        tmp[pred_col] = tmp[pred_col].astype(str)
        tmp[lead_col] = tmp[lead_col].apply(normalize_lead)
        tmp[value_col] = pd.to_numeric(tmp[value_col], errors="coerce")

        mat = tmp.pivot_table(
            index=pred_col,
            columns=lead_col,
            values=value_col,
            aggfunc="mean"
        )

    else:
        lead_cols = []

        for c in df.columns:
            if normalize_lead(c) in LEAD_LABELS:
                lead_cols.append(c)

        if pred_col is None or len(lead_cols) == 0:
            raise ValueError(
                "Could not interpret table as heatmap summary.\n"
                f"Columns found: {df.columns.tolist()}"
            )

        mat = df.set_index(pred_col)[lead_cols].copy()
        mat.columns = [normalize_lead(c) for c in mat.columns]

    mat = mat[[c for c in LEAD_LABELS if c in mat.columns]].copy()

    if REMOVE_SPATIOTEMPORAL:
        remove = {"lon", "lat", "month", "mon"}
        mat = mat.loc[
            [r for r in mat.index if str(r).strip().lower() not in remove]
        ].copy()

    row_order = [r for r in FEATURE_ORDER if r in mat.index]
    row_order += [r for r in mat.index if r not in row_order]

    mat = mat.loc[row_order].copy()

    if mat.empty:
        raise ValueError("Parsed heatmap matrix is empty.")

    return mat


def find_valid_summary_file(kind):
    if kind == "shap" and MANUAL_SHAP_SUMMARY is not None:
        return Path(MANUAL_SHAP_SUMMARY)

    if kind == "ablation" and MANUAL_ABLATION_SUMMARY is not None:
        return Path(MANUAL_ABLATION_SUMMARY)

    if kind == "shap":
        direct_candidates = [
            LAB / "OLD_NEW_MASTER_PLOTS_2009_2025_TEAL_SALMON" / "tables" / "shap_importance_adaboost_old_vs_new_no_geo.csv",
            LAB / "shap_adaboost_old_vs_new_all_leads_no_geo" / "shap_importance_adaboost_old_vs_new_all_leads_no_geo.csv",
        ]

        search_roots = [
            LAB / "OLD_NEW_MASTER_PLOTS_2009_2025_TEAL_SALMON",
            LAB / "shap_adaboost_old_vs_new_all_leads_no_geo",
            LAB / "mcs_output",
            LAB / "mcs_output_softtest",
        ]

        keyword_groups = [
            ["shap", "importance", "adaboost"],
            ["shap", "importance"],
        ]

    else:
        direct_candidates = [
            LAB / "OLD_NEW_MASTER_PLOTS_2009_2025_TEAL_SALMON" / "tables" / "adaboost_feature_ablation_old_vs_new_no_geo.csv",
            LAB / "feature_ablation_adaboost_old_vs_new_all_leads_no_geo" / "adaboost_feature_ablation_old_vs_new_no_geo.csv",
        ]

        search_roots = [
            LAB / "OLD_NEW_MASTER_PLOTS_2009_2025_TEAL_SALMON",
            LAB / "feature_ablation_adaboost_old_vs_new_all_leads_no_geo",
            LAB / "mcs_output",
            LAB / "mcs_output_softtest",
        ]

        keyword_groups = [
            ["ablation", "adaboost"],
            ["feature", "ablation"],
        ]

    candidates = []

    for fp in direct_candidates:
        if fp.exists():
            candidates.append(fp)

    for root in search_roots:
        for keys in keyword_groups:
            candidates.extend(recursive_csv_search(root, keys))

    seen = set()
    candidates = [fp for fp in candidates if not (str(fp) in seen or seen.add(str(fp)))]

    valid = []

    for fp in candidates:
        try:
            df = pd.read_csv(fp, low_memory=False)

            old_mat = matrix_from_summary(df, dataset_key="old", kind=kind)
            new_mat = matrix_from_summary(df, dataset_key="new", kind=kind)

            if old_mat.shape[0] > 0 and new_mat.shape[0] > 0:
                valid.append(fp)

        except Exception:
            continue

    if not valid:
        raise FileNotFoundError(
            f"No valid {kind} summary table found.\n"
            f"Set MANUAL_{kind.upper()}_SUMMARY at the top if needed."
        )

    print(f"Using {kind} summary file:")
    print(valid[0])

    return valid[0]


def plot_shap_pair(old_mat, new_mat):
    old_vals = old_mat.values.astype(float)
    new_vals = new_mat.values.astype(float)

    vmin = 0
    vmax = max(np.nanmax(old_vals), np.nanmax(new_vals))

    fig, axes = plt.subplots(1, 2, figsize=(14.5, 7.2))

    plt.subplots_adjust(
        left=0.08,
        right=0.88,
        bottom=0.12,
        top=0.80,
        wspace=0.38
    )

    for ax, mat, dataset_title, show_ylabel in [
        (axes[0], old_mat, OLD_NAME, True),
        (axes[1], new_mat, NEW_NAME, False),
    ]:
        vals = mat.values.astype(float)

        im = ax.imshow(
            vals,
            aspect="auto",
            cmap="viridis",
            vmin=vmin,
            vmax=vmax
        )

        ax.set_title(
            f"{dataset_title} {LABEL_NAME} | {MODEL_NAME}\n"
            f"SHAP heatmap, mean |SHAP value|",
            fontsize=12,
            pad=8
        )

        ax.set_xlabel("Lead time", fontsize=12)

        if show_ylabel:
            ax.set_ylabel("Predictor", fontsize=12, labelpad=12)
        else:
            ax.set_ylabel("")

        ax.set_xticks(np.arange(mat.shape[1]))
        ax.set_xticklabels(mat.columns, fontsize=11)

        ax.set_yticks(np.arange(mat.shape[0]))
        ax.set_yticklabels(mat.index, fontsize=11)

        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                val = vals[i, j]
                text_color = "white" if val > 0.55 * vmax else "black"

                ax.text(
                    j,
                    i,
                    f"{val:.3f}",
                    ha="center",
                    va="center",
                    fontsize=7.5,
                    color=text_color
                )

    cbar_ax = fig.add_axes([0.90, 0.18, 0.018, 0.56])
    cbar = fig.colorbar(im, cax=cbar_ax)
    cbar.set_label("Mean absolute SHAP value", fontsize=11)

    fig.suptitle(
        f"{OLD_NAME} vs {NEW_NAME} {MODEL_NAME} SHAP heatmap\n"
        f"No lon/lat/month",
        fontsize=16,
        y=0.95
    )

    out_png = SHAP_DIR / "conventional_vs_MR_adaboost_shap_heatmap_no_geo.png"
    plt.savefig(out_png, dpi=250, bbox_inches="tight")
    plt.show()

    print("Saved:", out_png)


def plot_ablation_pair(old_mat, new_mat):
    old_vals = old_mat.values.astype(float)
    new_vals = new_mat.values.astype(float)

    vmax = max(np.nanmax(np.abs(old_vals)), np.nanmax(np.abs(new_vals)))
    vmin = -vmax

    fig, axes = plt.subplots(1, 2, figsize=(14.5, 7.2))

    plt.subplots_adjust(
        left=0.08,
        right=0.88,
        bottom=0.12,
        top=0.80,
        wspace=0.38
    )

    for ax, mat, dataset_title, show_ylabel in [
        (axes[0], old_mat, OLD_NAME, True),
        (axes[1], new_mat, NEW_NAME, False),
    ]:
        vals = mat.values.astype(float)

        im = ax.imshow(
            vals,
            aspect="auto",
            cmap="RdBu_r",
            vmin=vmin,
            vmax=vmax
        )

        ax.set_title(
            f"{dataset_title} {LABEL_NAME} | {MODEL_NAME}\n"
            f"Feature ablation (default threshold)",
            fontsize=12,
            pad=8
        )

        ax.set_xlabel("Lead time", fontsize=12)

        if show_ylabel:
            ax.set_ylabel("Removed predictor", fontsize=12, labelpad=12)
        else:
            ax.set_ylabel("")

        ax.set_xticks(np.arange(mat.shape[1]))
        ax.set_xticklabels(mat.columns, fontsize=11)

        ax.set_yticks(np.arange(mat.shape[0]))
        ax.set_yticklabels(mat.index, fontsize=11)

        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                val = vals[i, j]
                text_color = "white" if abs(val) > 0.55 * vmax else "black"

                ax.text(
                    j,
                    i,
                    f"{val:.3f}",
                    ha="center",
                    va="center",
                    fontsize=7.5,
                    color=text_color
                )

    cbar_ax = fig.add_axes([0.90, 0.18, 0.018, 0.56])
    cbar = fig.colorbar(im, cax=cbar_ax)
    cbar.set_label("F1 drop", fontsize=11)

    fig.suptitle(
        f"{OLD_NAME} vs {NEW_NAME} {MODEL_NAME} feature ablation heatmap\n"
        f"No lon/lat/month | value = baseline F1 − ablated F1",
        fontsize=16,
        y=0.95
    )

    out_png = ABL_DIR / "conventional_vs_MR_adaboost_feature_ablation_no_geo.png"
    plt.savefig(out_png, dpi=250, bbox_inches="tight")
    plt.show()

    print("Saved:", out_png)

# ============================================================
# 1. CONFUSION MATRICES
# ============================================================

print("\n" + "=" * 90)
print("1. CONFUSION MATRICES")
print("=" * 90)

old_oof_fp = first_existing(OLD_OOF_CANDIDATES, "Conventional OOF")
new_oof_fp = first_existing(NEW_OOF_CANDIDATES, "MR OOF")

old_oof = clean_df(pd.read_csv(old_oof_fp, low_memory=False))
new_oof = clean_df(pd.read_csv(new_oof_fp, low_memory=False))

old_oof["lead_h"] = old_oof["lead_h"].astype(int)
new_oof["lead_h"] = new_oof["lead_h"].astype(int)

old_models = sorted(old_oof["model"].dropna().unique().tolist())
new_models = sorted(new_oof["model"].dropna().unique().tolist())

print("Loaded Conventional OOF:", old_oof_fp)
print("Loaded MR OOF:", new_oof_fp)

fig, axes = plt.subplots(2, 4, figsize=(18, 8.7))

plt.subplots_adjust(
    left=0.055,
    right=0.985,
    bottom=0.075,
    top=0.79,
    wspace=0.32,
    hspace=0.68
)

cm_rows = []

for col, lead in enumerate(LEADS):
    # ---------------- Conventional ----------------
    model_display = CONVENTIONAL_CM_MODELS[lead]
    model_actual = find_model_name(old_models, model_display)

    sub = old_oof[
        (old_oof["lead_h"] == lead)
        & (old_oof["model"] == model_actual)
    ].copy()

    y_true = sub["y_true"].astype(int).values
    y_pred = choose_default_prediction(sub)

    cm = pd.crosstab(
        pd.Series(y_true, name="True"),
        pd.Series(y_pred, name="Pred"),
        dropna=False
    ).reindex(index=[0, 1], columns=[0, 1], fill_value=0).values

    cm_norm = row_normalize_cm(cm)

    draw_normalized_cm(
        axes[0, col],
        cm_norm,
        f"{OLD_NAME} | {lead} h\n{model_display}"
    )

    cm_rows.append({
        "dataset": OLD_NAME,
        "lead_h": lead,
        "model": model_display,
        "tn": int(cm[0, 0]),
        "fp": int(cm[0, 1]),
        "fn": int(cm[1, 0]),
        "tp": int(cm[1, 1]),
        "tn_norm": cm_norm[0, 0],
        "fp_norm": cm_norm[0, 1],
        "fn_norm": cm_norm[1, 0],
        "tp_norm": cm_norm[1, 1],
    })

    # ---------------- MR ----------------
    model_display = MR_CM_MODELS[lead]
    model_actual = find_model_name(new_models, model_display)

    sub = new_oof[
        (new_oof["lead_h"] == lead)
        & (new_oof["model"] == model_actual)
    ].copy()

    y_true = sub["y_true"].astype(int).values
    y_pred = choose_default_prediction(sub)

    cm = pd.crosstab(
        pd.Series(y_true, name="True"),
        pd.Series(y_pred, name="Pred"),
        dropna=False
    ).reindex(index=[0, 1], columns=[0, 1], fill_value=0).values

    cm_norm = row_normalize_cm(cm)

    draw_normalized_cm(
        axes[1, col],
        cm_norm,
        f"{NEW_NAME} | {lead} h\n{model_display}"
    )

    cm_rows.append({
        "dataset": NEW_NAME,
        "lead_h": lead,
        "model": model_display,
        "tn": int(cm[0, 0]),
        "fp": int(cm[0, 1]),
        "fn": int(cm[1, 0]),
        "tp": int(cm[1, 1]),
        "tn_norm": cm_norm[0, 0],
        "fp_norm": cm_norm[0, 1],
        "fn_norm": cm_norm[1, 0],
        "tp_norm": cm_norm[1, 1],
    })

fig.suptitle(
    "Selected confusion matrices — row-normalized\n"
    "Conventional vs MR segmentation | default threshold",
    fontsize=18,
    fontweight="bold",
    y=0.965
)

cm_out = CM_DIR / "selected_confusion_matrices_conventional_vs_MR_default.png"
plt.savefig(cm_out, dpi=250, bbox_inches="tight")
plt.show()

cm_table = pd.DataFrame(cm_rows)
cm_table.to_csv(CM_DIR / "selected_confusion_matrices_conventional_vs_MR_default_values.csv", index=False)

print("Saved:", cm_out)
display(cm_table)

# ============================================================
# 2. SHAP HEATMAP
# ============================================================

print("\n" + "=" * 90)
print("2. SHAP HEATMAP")
print("=" * 90)

shap_fp = find_valid_summary_file("shap")
shap_df = pd.read_csv(shap_fp, low_memory=False)

old_shap = matrix_from_summary(shap_df, dataset_key="old", kind="shap")
new_shap = matrix_from_summary(shap_df, dataset_key="new", kind="shap")

plot_shap_pair(old_shap, new_shap)

print("\nSHAP matrix — Conventional:")
display(old_shap)

print("\nSHAP matrix — MR:")
display(new_shap)

# ============================================================
# 3. FEATURE ABLATION HEATMAP
# ============================================================

print("\n" + "=" * 90)
print("3. FEATURE ABLATION HEATMAP")
print("=" * 90)

abl_fp = find_valid_summary_file("ablation")
abl_df = pd.read_csv(abl_fp, low_memory=False)

old_abl = matrix_from_summary(abl_df, dataset_key="old", kind="ablation")
new_abl = matrix_from_summary(abl_df, dataset_key="new", kind="ablation")

plot_ablation_pair(old_abl, new_abl)

print("\nFeature ablation matrix — Conventional:")
display(old_abl)

print("\nFeature ablation matrix — MR:")
display(new_abl)

print("\nCELL 1 DONE.")
print("Output directory:")
print(OUT_DIR)

In [ ]:
# ============================================================
# CELL 1 — SETUP + HELPERS
# Conventional vs MR
# MR = morphology-refined segmentation
# ============================================================

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAS_CARTOPY = True
except Exception:
    HAS_CARTOPY = False
    print("[WARNING] Cartopy not available. Maps will use plain matplotlib axes.")

try:
    from IPython.display import display
except Exception:
    display = print

LAB = Path(r"C:\Users\_s2218026\Documents\lab")

OLD_NAME = "Conventional"
NEW_NAME = "MR"

COLOR_OLD = "#2A9D8F"   # teal
COLOR_NEW = "#E76F51"   # salmon

LABEL_NAME = "box2deg"
MODEL_NAME = "AdaBoost"

LEADS = [6, 12, 24, 48]
LEAD_LABELS = ["6 h", "12 h", "24 h", "48 h"]

START_YEAR = 2009
END_YEAR = 2025

LON_MIN, LON_MAX = 100, 180
LAT_MIN, LAT_MAX = 0, 30

CONVENTIONAL_CM_MODELS = {
    6:  "AdaBoost",
    12: "Naive Bayes",
    24: "AdaBoost",
    48: "MLP",
}

MR_CM_MODELS = {
    6:  "AdaBoost",
    12: "Naive Bayes",
    24: "AdaBoost",
    48: "Naive Bayes",
}

OLD_CASE_DIR = LAB / "mcs_output" / "exactlead_box2deg_case_tables_old_2009_2025_UPDATED"
NEW_CASE_DIR = LAB / "mcs_output_softtest" / "exactlead_box2deg_case_tables_new_2009_2025_UPDATED"

OLD_OOF_CANDIDATES = [
    LAB / "mcs_output" / "ml_old_box2deg_2009_2025_final" / "oof_predictions_all_models.csv",
    LAB / "mcs_output" / "ml_old_box2deg_2009_2025_final" / "oof_predictions_allmodels_old.csv",
    LAB / "mcs_output" / "ml_old_box2deg_2009_2025_UPDATED" / "oof_predictions_all_models.csv",
    LAB / "mcs_output" / "ml_old_box2deg_2009_2025_UPDATED" / "oof_predictions_allmodels_old.csv",
]

NEW_OOF_CANDIDATES = [
    LAB / "mcs_output_softtest" / "ml_new_box2deg_2009_2025_UPDATED" / "oof_predictions_allmodels_new.csv",
    LAB / "mcs_output_softtest" / "ml_new_box2deg_2009_2025_UPDATED" / "oof_predictions_all_models.csv",
    LAB / "mcs_output_softtest" / "ml_new_box2deg_2009_2025_UPDATED" / "oof_predictions_allmodels_soft.csv",
]

MANUAL_SHAP_SUMMARY = None
MANUAL_ABLATION_SUMMARY = None

REMOVE_SPATIOTEMPORAL = True

FEATURE_ORDER = [
    "eta850_abs",
    "rh600",
    "wshear_200_850",
    "pi_vmax_ms",
    "gpi",
    "area_km2",
    "tb_mean",
    "tb_min",
    "speed_kmh",
    "direction_deg",
]

OUT_DIR = LAB / "FINAL_PLOTS_CONVENTIONAL_vs_MR"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CM_DIR = OUT_DIR / "confusion_matrices"
SHAP_DIR = OUT_DIR / "shap_heatmap"
ABL_DIR = OUT_DIR / "feature_ablation"
BAR_DIR = OUT_DIR / "bar_plots"
MAP_DIR = OUT_DIR / "positive_maps"
TABLE_DIR = OUT_DIR / "tables"

for d in [CM_DIR, SHAP_DIR, ABL_DIR, BAR_DIR, MAP_DIR, TABLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def clean_df(df):
    return df.loc[:, ~df.columns.astype(str).str.startswith("Unnamed")].copy()

def first_existing(paths, label):
    for fp in paths:
        if fp.exists():
            return fp
    raise FileNotFoundError(
        f"No file found for {label}. Checked:\n" +
        "\n".join(str(fp) for fp in paths)
    )

def recursive_csv_search(base_dir, keywords):
    base_dir = Path(base_dir)
    hits = []
    if not base_dir.exists():
        return hits
    for fp in base_dir.rglob("*.csv"):
        name = fp.name.lower()
        if all(k.lower() in name for k in keywords):
            hits.append(fp)
    return sorted(hits)

def normalize_lead(x):
    try:
        return f"{int(float(x))} h"
    except Exception:
        s = str(x).strip().lower()
        s = s.replace("hours", "h").replace("hour", "h").replace(" ", "")
        if s in ["6", "6h"]:
            return "6 h"
        if s in ["12", "12h"]:
            return "12 h"
        if s in ["24", "24h"]:
            return "24 h"
        if s in ["48", "48h"]:
            return "48 h"
        return str(x)

def find_model_name(models_available, target_name):
    models_available = list(models_available)

    if target_name in models_available:
        return target_name

    def canon(s):
        return str(s).lower().replace(" ", "").replace("_", "").replace("-", "")

    target = canon(target_name)
    model_map = {canon(m): m for m in models_available}

    if target in model_map:
        return model_map[target]

    aliases = {
        "naivebayes": ["naivebayes", "nb", "gaussiannb", "gaussiannaivebayes"],
        "adaboost": ["adaboost", "ada", "adaboostclassifier"],
        "mlp": ["mlp", "multilayerperceptron", "neuralnetwork"],
        "randomforest": ["randomforest", "rf"],
        "svm": ["svm"],
    }

    for key, alias_list in aliases.items():
        if target == key or target in alias_list:
            for c, original in model_map.items():
                if c == key or c in alias_list:
                    return original

    raise ValueError(
        f"Could not find model '{target_name}'. Available models:\n"
        f"{sorted(models_available)}"
    )

def choose_default_prediction(sub):
    if "y_pred_default" in sub.columns:
        return sub["y_pred_default"].astype(int).values
    if "y_pred" in sub.columns:
        return sub["y_pred"].astype(int).values
    if "y_score" in sub.columns:
        return (sub["y_score"].astype(float).values >= 0.5).astype(int)
    if "y_prob" in sub.columns:
        return (sub["y_prob"].astype(float).values >= 0.5).astype(int)

    raise ValueError(
        "No prediction column found. Need y_pred_default, y_pred, y_score, or y_prob."
    )

def row_normalize_cm(cm):
    cm = np.asarray(cm, dtype=float)
    row_sum = cm.sum(axis=1, keepdims=True)
    return np.divide(cm, row_sum, out=np.zeros_like(cm), where=row_sum != 0)

def draw_normalized_cm(ax, cm_norm, title):
    ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])

    ax.set_xticklabels(["Non-dev", "Dev"], fontsize=11)
    ax.set_yticklabels(["Non-dev", "Dev"], fontsize=11)

    ax.set_xlabel("Predicted outcome", fontsize=11)
    ax.set_ylabel("True outcome", fontsize=11)
    ax.set_title(title, fontsize=13, fontweight="bold", pad=8)

    for i in range(2):
        for j in range(2):
            val = cm_norm[i, j]
            ax.text(
                j,
                i,
                f"{val:.2f}",
                ha="center",
                va="center",
                fontsize=15,
                fontweight="bold",
                color="white" if val > 0.55 else "black"
            )

def is_raw_prediction_table(df):
    cols = set(df.columns.astype(str))

    raw_cols = {
        "y_true",
        "y_prob",
        "y_score",
        "y_pred",
        "y_pred_default",
        "y_pred_tuned",
    }

    predictor_cols = {
        "lon",
        "lat",
        "month",
        "area_km2",
        "tb_mean",
        "tb_min",
        "speed_kmh",
        "direction_deg",
        "eta850_abs",
        "rh600",
        "wshear_200_850",
        "pi_vmax_ms",
        "gpi",
    }

    return bool(cols.intersection(raw_cols)) and len(cols.intersection(predictor_cols)) >= 5

def choose_existing_col(df, candidates):
    lower = {str(c).lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lower:
            return lower[c.lower()]
    return None

def matrix_from_summary(df, dataset_key, kind):
    df = clean_df(df).copy()
    df.columns = [str(c) for c in df.columns]

    if is_raw_prediction_table(df):
        raise ValueError("This is a raw prediction table, not a SHAP/ablation summary table.")

    if "dataset" in df.columns:
        ds = df["dataset"].astype(str).str.lower()

        if dataset_key == "old":
            valid_names = ["old", "conventional"]
        else:
            valid_names = ["new", "soft", "mr", "morphology-refined", "morphology refined"]

        df = df[ds.isin(valid_names)].copy()

    pred_col = choose_existing_col(
        df,
        ["feature", "predictor", "removed_feature", "removed predictor", "variable", "name"]
    )

    lead_col = choose_existing_col(
        df,
        ["lead_h", "lead", "lead_time", "lead time"]
    )

    if kind == "shap":
        value_col = choose_existing_col(
            df,
            ["mean_abs_shap", "mean(|SHAP value|)", "mean_shap", "shap_value", "shap", "importance", "value"]
        )
    else:
        value_col = choose_existing_col(
            df,
            ["delta_default_f1", "f1_drop", "delta_f1", "drop_f1", "delta_tuned_f1", "value"]
        )

    if pred_col is not None and lead_col is not None and value_col is not None:
        tmp = df[[pred_col, lead_col, value_col]].copy()

        tmp[pred_col] = tmp[pred_col].astype(str)
        tmp[lead_col] = tmp[lead_col].apply(normalize_lead)
        tmp[value_col] = pd.to_numeric(tmp[value_col], errors="coerce")

        mat = tmp.pivot_table(
            index=pred_col,
            columns=lead_col,
            values=value_col,
            aggfunc="mean"
        )

    else:
        lead_cols = []
        for c in df.columns:
            if normalize_lead(c) in LEAD_LABELS:
                lead_cols.append(c)

        if pred_col is None or len(lead_cols) == 0:
            raise ValueError(
                "Could not interpret table as heatmap summary.\n"
                f"Columns found: {df.columns.tolist()}"
            )

        mat = df.set_index(pred_col)[lead_cols].copy()
        mat.columns = [normalize_lead(c) for c in mat.columns]

    mat = mat[[c for c in LEAD_LABELS if c in mat.columns]].copy()

    if REMOVE_SPATIOTEMPORAL:
        remove = {"lon", "lat", "month", "mon"}
        mat = mat.loc[
            [r for r in mat.index if str(r).strip().lower() not in remove]
        ].copy()

    row_order = [r for r in FEATURE_ORDER if r in mat.index]
    row_order += [r for r in mat.index if r not in row_order]

    mat = mat.loc[row_order].copy()

    if mat.empty:
        raise ValueError("Parsed heatmap matrix is empty.")

    return mat

def find_valid_summary_file(kind):
    if kind == "shap" and MANUAL_SHAP_SUMMARY is not None:
        return Path(MANUAL_SHAP_SUMMARY)

    if kind == "ablation" and MANUAL_ABLATION_SUMMARY is not None:
        return Path(MANUAL_ABLATION_SUMMARY)

    if kind == "shap":
        direct_candidates = [
            LAB / "OLD_NEW_MASTER_PLOTS_2009_2025_TEAL_SALMON" / "tables" / "shap_importance_adaboost_old_vs_new_no_geo.csv",
            LAB / "shap_adaboost_old_vs_new_all_leads_no_geo" / "shap_importance_adaboost_old_vs_new_all_leads_no_geo.csv",
        ]
        search_roots = [
            LAB / "OLD_NEW_MASTER_PLOTS_2009_2025_TEAL_SALMON",
            LAB / "shap_adaboost_old_vs_new_all_leads_no_geo",
            LAB / "mcs_output",
            LAB / "mcs_output_softtest",
        ]
        keyword_groups = [
            ["shap", "importance", "adaboost"],
            ["shap", "importance"],
        ]
    else:
        direct_candidates = [
            LAB / "OLD_NEW_MASTER_PLOTS_2009_2025_TEAL_SALMON" / "tables" / "adaboost_feature_ablation_old_vs_new_no_geo.csv",
            LAB / "feature_ablation_adaboost_old_vs_new_all_leads_no_geo" / "adaboost_feature_ablation_old_vs_new_no_geo.csv",
        ]
        search_roots = [
            LAB / "OLD_NEW_MASTER_PLOTS_2009_2025_TEAL_SALMON",
            LAB / "feature_ablation_adaboost_old_vs_new_all_leads_no_geo",
            LAB / "mcs_output",
            LAB / "mcs_output_softtest",
        ]
        keyword_groups = [
            ["ablation", "adaboost"],
            ["feature", "ablation"],
        ]

    candidates = []

    for fp in direct_candidates:
        if fp.exists():
            candidates.append(fp)

    for root in search_roots:
        for keys in keyword_groups:
            candidates.extend(recursive_csv_search(root, keys))

    seen = set()
    candidates = [fp for fp in candidates if not (str(fp) in seen or seen.add(str(fp)))]

    valid = []

    for fp in candidates:
        try:
            df = pd.read_csv(fp, low_memory=False)
            old_mat = matrix_from_summary(df, dataset_key="old", kind=kind)
            new_mat = matrix_from_summary(df, dataset_key="new", kind=kind)

            if old_mat.shape[0] > 0 and new_mat.shape[0] > 0:
                valid.append(fp)

        except Exception:
            continue

    if not valid:
        raise FileNotFoundError(
            f"No valid {kind} summary table found.\n"
            f"Set MANUAL_{kind.upper()}_SUMMARY at the top if needed."
        )

    print(f"Using {kind} summary file:")
    print(valid[0])

    return valid[0]

def plot_shap_pair(old_mat, new_mat):
    old_vals = old_mat.values.astype(float)
    new_vals = new_mat.values.astype(float)

    vmin = 0
    vmax = max(np.nanmax(old_vals), np.nanmax(new_vals))

    fig, axes = plt.subplots(1, 2, figsize=(14.5, 7.2))

    plt.subplots_adjust(
        left=0.08,
        right=0.88,
        bottom=0.12,
        top=0.80,
        wspace=0.38
    )

    for ax, mat, dataset_title, show_ylabel in [
        (axes[0], old_mat, OLD_NAME, True),
        (axes[1], new_mat, NEW_NAME, False),
    ]:
        vals = mat.values.astype(float)

        im = ax.imshow(
            vals,
            aspect="auto",
            cmap="viridis",
            vmin=vmin,
            vmax=vmax
        )

        ax.set_title(
            f"{dataset_title} {LABEL_NAME} | {MODEL_NAME}\n"
            f"SHAP heatmap, mean |SHAP value|",
            fontsize=12,
            pad=8
        )

        ax.set_xlabel("Lead time", fontsize=12)

        if show_ylabel:
            ax.set_ylabel("Predictor", fontsize=12, labelpad=12)
        else:
            ax.set_ylabel("")

        ax.set_xticks(np.arange(mat.shape[1]))
        ax.set_xticklabels(mat.columns, fontsize=11)

        ax.set_yticks(np.arange(mat.shape[0]))
        ax.set_yticklabels(mat.index, fontsize=11)

        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                val = vals[i, j]
                text_color = "white" if val > 0.55 * vmax else "black"

                ax.text(
                    j,
                    i,
                    f"{val:.3f}",
                    ha="center",
                    va="center",
                    fontsize=7.5,
                    color=text_color
                )

    cbar_ax = fig.add_axes([0.90, 0.18, 0.018, 0.56])
    cbar = fig.colorbar(im, cax=cbar_ax)
    cbar.set_label("Mean absolute SHAP value", fontsize=11)

    fig.suptitle(
        f"{OLD_NAME} vs {NEW_NAME} {MODEL_NAME} SHAP heatmap\n"
        f"No lon/lat/month",
        fontsize=16,
        y=0.95
    )

    out_png = SHAP_DIR / "conventional_vs_MR_adaboost_shap_heatmap_no_geo.png"
    plt.savefig(out_png, dpi=250, bbox_inches="tight")
    plt.show()

    print("Saved:", out_png)

def plot_ablation_pair(old_mat, new_mat):
    old_vals = old_mat.values.astype(float)
    new_vals = new_mat.values.astype(float)

    vmax = max(np.nanmax(np.abs(old_vals)), np.nanmax(np.abs(new_vals)))
    vmin = -vmax

    fig, axes = plt.subplots(1, 2, figsize=(14.5, 7.2))

    plt.subplots_adjust(
        left=0.08,
        right=0.88,
        bottom=0.12,
        top=0.80,
        wspace=0.38
    )

    for ax, mat, dataset_title, show_ylabel in [
        (axes[0], old_mat, OLD_NAME, True),
        (axes[1], new_mat, NEW_NAME, False),
    ]:
        vals = mat.values.astype(float)

        im = ax.imshow(
            vals,
            aspect="auto",
            cmap="RdBu_r",
            vmin=vmin,
            vmax=vmax
        )

        ax.set_title(
            f"{dataset_title} {LABEL_NAME} | {MODEL_NAME}\n"
            f"Feature ablation (default threshold)",
            fontsize=12,
            pad=8
        )

        ax.set_xlabel("Lead time", fontsize=12)

        if show_ylabel:
            ax.set_ylabel("Removed predictor", fontsize=12, labelpad=12)
        else:
            ax.set_ylabel("")

        ax.set_xticks(np.arange(mat.shape[1]))
        ax.set_xticklabels(mat.columns, fontsize=11)

        ax.set_yticks(np.arange(mat.shape[0]))
        ax.set_yticklabels(mat.index, fontsize=11)

        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                val = vals[i, j]
                text_color = "white" if abs(val) > 0.55 * vmax else "black"

                ax.text(
                    j,
                    i,
                    f"{val:.3f}",
                    ha="center",
                    va="center",
                    fontsize=7.5,
                    color=text_color
                )

    cbar_ax = fig.add_axes([0.90, 0.18, 0.018, 0.56])
    cbar = fig.colorbar(im, cax=cbar_ax)
    cbar.set_label("F1 drop", fontsize=11)

    fig.suptitle(
        f"{OLD_NAME} vs {NEW_NAME} {MODEL_NAME} feature ablation heatmap\n"
        f"No lon/lat/month | value = baseline F1 − ablated F1",
        fontsize=16,
        y=0.95
    )

    out_png = ABL_DIR / "conventional_vs_MR_adaboost_feature_ablation_no_geo.png"
    plt.savefig(out_png, dpi=250, bbox_inches="tight")
    plt.show()

    print("Saved:", out_png)

def find_case_file(case_dir, dataset_name, lead):
    candidates = [
        case_dir / f"{LABEL_NAME}_case_table_{dataset_name}_{lead}h.csv",
        case_dir / f"zhang_case_table_{dataset_name}_{lead}h.csv",
        case_dir / f"zhang_case_table_{lead}h.csv",
        case_dir / f"case_table_{lead}h.csv",
    ]

    for fp in candidates:
        if fp.exists():
            return fp

    raise FileNotFoundError(
        f"No case-table file found for {dataset_name}, lead={lead}h.\n"
        + "\n".join(str(fp) for fp in candidates)
    )

def detect_lon_lat_cols(df):
    lon_candidates = [
        "centroid_lon",
        "lon",
        "longitude",
        "mcs_lon",
        "genesis_lon",
    ]

    lat_candidates = [
        "centroid_lat",
        "lat",
        "latitude",
        "mcs_lat",
        "genesis_lat",
    ]

    lon_col = next((c for c in lon_candidates if c in df.columns), None)
    lat_col = next((c for c in lat_candidates if c in df.columns), None)

    if lon_col is None or lat_col is None:
        raise ValueError(
            "Could not find longitude/latitude columns.\n"
            f"Available columns:\n{df.columns.tolist()}"
        )

    return lon_col, lat_col

def parse_and_filter_time(df):
    if "time" in df.columns:
        df["time"] = pd.to_datetime(
            df["time"],
            errors="coerce",
            utc=True
        ).dt.tz_convert(None)

        df = df.dropna(subset=["time"]).copy()

        df = df[
            (df["time"].dt.year >= START_YEAR)
            & (df["time"].dt.year <= END_YEAR)
        ].copy()

    return df

def load_case_positive_points(case_dir, dataset_name, display_name):
    all_points = []
    summary_rows = []

    for lead in LEADS:
        fp = find_case_file(case_dir, dataset_name, lead)

        df = pd.read_csv(fp, low_memory=False)
        df = clean_df(df)
        df = parse_and_filter_time(df)

        if "y" not in df.columns:
            raise ValueError(f"{fp} has no 'y' column.")

        df["y"] = df["y"].astype(int)

        total_n = len(df)
        positive_n = int(df["y"].sum())
        negative_n = int((df["y"] == 0).sum())
        positive_rate = positive_n / total_n if total_n > 0 else np.nan

        lon_col, lat_col = detect_lon_lat_cols(df)

        pos = df[df["y"] == 1].copy()

        points = pd.DataFrame({
            "dataset": dataset_name,
            "display_name": display_name,
            "lead_h": lead,
            "lon": pd.to_numeric(pos[lon_col], errors="coerce"),
            "lat": pd.to_numeric(pos[lat_col], errors="coerce"),
        }).dropna(subset=["lon", "lat"])

        if "time" in pos.columns:
            points["time"] = pos["time"].values

        if "track_id" in pos.columns:
            points["track_id"] = pos["track_id"].astype(str).values

        all_points.append(points)

        summary_rows.append({
            "dataset": dataset_name,
            "display_name": display_name,
            "lead_h": lead,
            "total_n": total_n,
            "positive_n": positive_n,
            "negative_n": negative_n,
            "positive_rate": positive_rate,
            "file": str(fp),
        })

        print(
            f"{display_name} | {lead}h | "
            f"total={total_n}, positive={positive_n}, "
            f"rate={positive_rate:.4f} | {fp.name}"
        )

    points_all = pd.concat(all_points, ignore_index=True)
    summary = pd.DataFrame(summary_rows)

    return points_all, summary

def setup_map(ax):
    if HAS_CARTOPY:
        ax.set_extent(
            [LON_MIN, LON_MAX, LAT_MIN, LAT_MAX],
            crs=ccrs.PlateCarree()
        )

        ax.add_feature(cfeature.OCEAN, facecolor="white", zorder=0)
        ax.add_feature(cfeature.LAND, facecolor="whitesmoke", zorder=0)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.7, zorder=1)
        ax.add_feature(cfeature.BORDERS, linewidth=0.35, alpha=0.7, zorder=1)

        gl = ax.gridlines(
            draw_labels=True,
            linewidth=0.35,
            color="gray",
            alpha=0.45,
            linestyle="--"
        )

        gl.top_labels = False
        gl.right_labels = False

    else:
        ax.set_xlim(LON_MIN, LON_MAX)
        ax.set_ylim(LAT_MIN, LAT_MAX)
        ax.grid(True, linestyle="--", alpha=0.35)

    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")

print("CELL 1 loaded.")
print("Output directory:", OUT_DIR)

In [ ]:
# ============================================================
# CELL 3 — POSITIVE BAR PLOTS + POSITIVE CENTROID MAP
# Run Cell 1 first.
# ============================================================

print("\n" + "=" * 90)
print("POSITIVE BARS + MAPS")
print("=" * 90)

old_points, old_summary = load_case_positive_points(
    case_dir=OLD_CASE_DIR,
    dataset_name="old",
    display_name=OLD_NAME
)

new_points, new_summary = load_case_positive_points(
    case_dir=NEW_CASE_DIR,
    dataset_name="new",
    display_name=NEW_NAME
)

points_all = pd.concat([old_points, new_points], ignore_index=True)
summary_all = pd.concat([old_summary, new_summary], ignore_index=True)

summary_all.to_csv(
    TABLE_DIR / "positive_summary_conventional_vs_MR_long.csv",
    index=False
)

points_all.to_csv(
    TABLE_DIR / "positive_points_conventional_vs_MR.csv",
    index=False
)

print("\nPositive summary:")
display(summary_all)

plot_df = summary_all.copy()

old_bar = plot_df[plot_df["display_name"] == OLD_NAME].sort_values("lead_h")
new_bar = plot_df[plot_df["display_name"] == NEW_NAME].sort_values("lead_h")

x = np.arange(len(LEADS))
width = 0.36

fig, ax = plt.subplots(figsize=(9.5, 5.4))

ax.bar(
    x - width / 2,
    old_bar["positive_n"],
    width=width,
    label=OLD_NAME,
    color=COLOR_OLD,
    edgecolor="black",
    linewidth=0.6
)

ax.bar(
    x + width / 2,
    new_bar["positive_n"],
    width=width,
    label=NEW_NAME,
    color=COLOR_NEW,
    edgecolor="black",
    linewidth=0.6
)

ax.set_xticks(x)
ax.set_xticklabels([f"{L} h" for L in LEADS])
ax.set_xlabel("Lead time")
ax.set_ylabel("Number of positive MCS snapshots")
ax.set_title(
    "Positive MCS label counts, 2009–2025\n"
    "Conventional vs MR segmentation"
)
ax.legend(title="Dataset")
ax.grid(True, axis="y", alpha=0.3)

for i, (old_v, new_v) in enumerate(zip(old_bar["positive_n"], new_bar["positive_n"])):
    ax.text(i - width / 2, old_v, f"{int(old_v)}", ha="center", va="bottom", fontsize=10)
    ax.text(i + width / 2, new_v, f"{int(new_v)}", ha="center", va="bottom", fontsize=10)

plt.tight_layout()

out_bar_counts = BAR_DIR / "positive_counts_conventional_vs_MR_teal_salmon.png"
plt.savefig(out_bar_counts, dpi=250, bbox_inches="tight")
plt.show()

print("Saved:", out_bar_counts)

fig, ax = plt.subplots(figsize=(9.5, 5.4))

ax.bar(
    x - width / 2,
    old_bar["positive_rate"],
    width=width,
    label=OLD_NAME,
    color=COLOR_OLD,
    edgecolor="black",
    linewidth=0.6
)

ax.bar(
    x + width / 2,
    new_bar["positive_rate"],
    width=width,
    label=NEW_NAME,
    color=COLOR_NEW,
    edgecolor="black",
    linewidth=0.6
)

ax.set_xticks(x)
ax.set_xticklabels([f"{L} h" for L in LEADS])
ax.set_xlabel("Lead time")
ax.set_ylabel("Positive label rate")
ax.set_title(
    "Positive MCS label rate, 2009–2025\n"
    "Conventional vs MR segmentation"
)
ax.legend(title="Dataset")
ax.grid(True, axis="y", alpha=0.3)

for i, (old_v, new_v) in enumerate(zip(old_bar["positive_rate"], new_bar["positive_rate"])):
    ax.text(i - width / 2, old_v, f"{old_v:.3f}", ha="center", va="bottom", fontsize=10)
    ax.text(i + width / 2, new_v, f"{new_v:.3f}", ha="center", va="bottom", fontsize=10)

plt.tight_layout()

out_bar_rate = BAR_DIR / "positive_rate_conventional_vs_MR_teal_salmon.png"
plt.savefig(out_bar_rate, dpi=250, bbox_inches="tight")
plt.show()

print("Saved:", out_bar_rate)

if HAS_CARTOPY:
    fig, axes = plt.subplots(
        2,
        2,
        figsize=(15.5, 8.8),
        subplot_kw={"projection": ccrs.PlateCarree()}
    )
else:
    fig, axes = plt.subplots(
        2,
        2,
        figsize=(15.5, 8.8)
    )

plt.subplots_adjust(
    left=0.05,
    right=0.98,
    bottom=0.08,
    top=0.84,
    wspace=0.20,
    hspace=0.32
)

axes = axes.flatten()

for ax, lead in zip(axes, LEADS):
    old_sub = points_all[
        (points_all["display_name"] == OLD_NAME)
        & (points_all["lead_h"] == lead)
    ].copy()

    new_sub = points_all[
        (points_all["display_name"] == NEW_NAME)
        & (points_all["lead_h"] == lead)
    ].copy()

    setup_map(ax)

    old_kwargs = dict(
        s=32,
        c=COLOR_OLD,
        marker="o",
        alpha=0.76,
        label=f"{OLD_NAME} (n={len(old_sub)})",
        zorder=3
    )

    new_kwargs = dict(
        s=42,
        c=COLOR_NEW,
        marker="x",
        alpha=0.92,
        linewidths=1.25,
        label=f"{NEW_NAME} (n={len(new_sub)})",
        zorder=4
    )

    if HAS_CARTOPY:
        old_kwargs["transform"] = ccrs.PlateCarree()
        new_kwargs["transform"] = ccrs.PlateCarree()

    ax.scatter(old_sub["lon"], old_sub["lat"], **old_kwargs)
    ax.scatter(new_sub["lon"], new_sub["lat"], **new_kwargs)

    ax.set_title(f"{lead} h lead", fontsize=13, fontweight="bold")
    ax.legend(loc="lower left", fontsize=9, frameon=True)

fig.suptitle(
    "Positive MCS centroid locations, 2009–2025\n"
    "Conventional vs MR segmentation | box2deg exact-lead labels",
    fontsize=17,
    fontweight="bold",
    y=0.965
)

out_map = MAP_DIR / "positive_centroid_maps_conventional_vs_MR_teal_salmon.png"
plt.savefig(out_map, dpi=250, bbox_inches="tight")
plt.show()

print("Saved:", out_map)

print("\nCELL 3 DONE.")
print("Output directory:", OUT_DIR)